In [ ]:

# ============================================================
# CHECKPOINT 4 — D1 / C3-5
# CELL 1: NAB MAIN 23-FEATURE ARTIFACT DISCOVERY
# MODE: STRICTLY READ-ONLY
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

BASE = Path('/content/drive/MyDrive/Master/Thesis/Code')

assert BASE.is_dir(), (
    'STOP: Thesis Code directory not found.'
)

print('Google Drive mounted:', BASE)

import pandas as pd

# ------------------------------------------------------------
# 1. Exact candidate directory (not independently verified)
# ------------------------------------------------------------

NAB_DIR = (
    BASE
    / 'processed_data'
    / 'PH4'
    / 'EXP_NAB_60min_stride30'
)

FILES = {
    'PRE_PURGE': NAB_DIR / 'nab_windows_60min_stride30.csv',
    'POST_PURGE': NAB_DIR / 'nab_windows_60min_stride30_purged.csv'
}

EXPECTED_ROWS = {
    'PRE_PURGE': 103526,
    'POST_PURGE': 103412
}

EXPECTED_FEATURES = [
    'energy', 'first_last_diff', 'iqr', 'kurtosis',
    'mad', 'max', 'max_abs_diff', 'mean',
    'mean_abs_diff', 'median', 'min', 'n_points',
    'peak_count', 'q25', 'q75', 'range',
    'rms', 'skew', 'slope', 'std',
    'std_diff', 'sum', 'var'
]

print('\n' + '=' * 75)
print('C3-5 | NAB MAIN 23-FEATURE ARTIFACT CHECK')
print('=' * 75)

print('\nCandidate directory:')
print(NAB_DIR)

# ------------------------------------------------------------
# 2. Verify exact paths BEFORE reading any CSV
# ------------------------------------------------------------

missing = []

for label, path in FILES.items():
    exists = path.is_file()

    print(f'\n[{label}]')
    print('Path:', path)
    print('Status:', 'FOUND' if exists else 'NOT FOUND')

    if exists:
        print('Size (bytes):', path.stat().st_size)
        print('Expected historical rows:', EXPECTED_ROWS[label])
    else:
        missing.append(label)

# ------------------------------------------------------------
# 3. STOP if either candidate file is absent
#    Inspect only ONE immediate known parent directory.
#    No recursive search.
# ------------------------------------------------------------

if missing:

    print('\n' + '!' * 75)
    print('STOP: ONE OR MORE CANDIDATE NAB FILES NOT FOUND')
    print('Missing:', missing)

    inspect_dir = (
        NAB_DIR if NAB_DIR.is_dir()
        else NAB_DIR.parent
    )

    print('\nImmediate directory to inspect:')
    print(inspect_dir)

    if inspect_dir.is_dir():
        print('\nDirect children (non-recursive):')

        children = sorted(
            inspect_dir.iterdir(),
            key=lambda p: p.name.lower()
        )

        for item in children:
            kind = 'DIR ' if item.is_dir() else 'FILE'
            print(f'  [{kind}] {item.name}')

        print('\nTotal direct children:', len(children))
    else:
        print('Parent directory is also unavailable.')

    print('\nNo CSV data was loaded.')
    print('Please share this output before proceeding.')

    raise SystemExit('STOP_C3_5_PATH_VERIFICATION')

# ------------------------------------------------------------
# 4. Both files exist: inspect headers only
#    nrows=0 does NOT load full datasets.
# ------------------------------------------------------------

print('\n' + '=' * 75)
print('BOTH CANDIDATE FILES FOUND — HEADER VERIFICATION')
print('=' * 75)

schema_ok = True

for label, path in FILES.items():

    headers = pd.read_csv(path, nrows=0).columns.tolist()

    print(f'\n[{label}]')
    print('Total columns:', len(headers))
    print('Column names:')

    for i, name in enumerate(headers, start=1):
        print(f'  {i:02d}. {name}')

    present = [f for f in EXPECTED_FEATURES if f in headers]
    absent = [f for f in EXPECTED_FEATURES if f not in headers]

    print('\nExpected 23-feature names found:', len(present))
    print('Missing expected feature names:', absent)
    print('n_points present:', 'n_points' in headers)

    if absent:
        schema_ok = False
        print('SCHEMA_STATUS: REVIEW_REQUIRED')
    else:
        print('SCHEMA_STATUS: PASS_EXPECTED_FEATURE_NAMES')

# ------------------------------------------------------------
# 5. Final status
# ------------------------------------------------------------

print('\n' + '=' * 75)

if schema_ok:
    print('PASS_C3_5_CANDIDATE_PATHS_AND_FEATURE_HEADERS')
    print('Ready for the next verification cell.')
else:
    print('REVIEW_C3_5_FEATURE_SCHEMA')
    print('Verify artifact identity before numerical analysis.')

print('\nNo full CSV loaded.')
print('No rows filtered or modified.')
print('No files created or overwritten.')
print('No model trained or evaluated.')
print('C3-5 revision status: NOT DONE')

print('=' * 75)


In [ ]:

# CHECKPOINT 4 — D1 / C3-5 — CELL 2
# Main 23-feature NAB verification (READ ONLY)

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path('/content/drive/MyDrive/Master/Thesis/Code')
assert BASE.is_dir(), 'STOP: Thesis Code directory not found.'

PRE = (BASE / 'processed_data/PH4'
       / 'EXP_NAB_60min_stride30'
       / 'nab_windows_60min_stride30.csv')

POST = (BASE / 'experiments'
        / 'EXP_SPLIT_BOUNDARY_PURGE_CORRECTION'
        / 'NAB/60_30/all_corrected_partitions.csv')

FEATURES = [
    'energy', 'first_last_diff', 'iqr', 'kurtosis',
    'mad', 'max', 'max_abs_diff', 'mean',
    'mean_abs_diff', 'median', 'min', 'n_points',
    'peak_count', 'q25', 'q75', 'range', 'rms',
    'skew', 'slope', 'std', 'std_diff', 'sum', 'var'
]

DEGENERATE_ZERO_FEATURES = [
    'std', 'var', 'mad', 'iqr', 'range',
    'skew', 'kurtosis', 'slope', 'peak_count',
    'first_last_diff', 'mean_abs_diff',
    'max_abs_diff', 'std_diff'
]

def verify_nab():
    print('=' * 75)
    print('C3-5 — ORIGINAL MAIN NAB 23-FEATURE VERIFICATION')
    print('=' * 75)

    files = {'PRE_PURGE': PRE, 'POST_PURGE': POST}

    for name, path in files.items():
        print(f'\n{name}: {path}')
        print('FOUND' if path.is_file() else 'NOT FOUND')

    if not all(p.is_file() for p in files.values()):
        print('\nSTOP: Required artifact unavailable.')

        if POST.parent.is_dir():
            print('\nCorrected 60_30 directory (direct children):')
            for p in sorted(POST.parent.iterdir()):
                print(' ', p.name)

        return

    # Header verification precedes full data loading.
    for name, path in files.items():
        cols = pd.read_csv(path, nrows=0).columns.tolist()

        required = FEATURES + [
            'series', 'window_start', 'window_end', 'label'
        ]
        if name == 'POST_PURGE':
            required += ['split']

        missing = [c for c in required if c not in cols]

        print(f'\n{name} COLUMN COUNT:', len(cols))
        print('MISSING REQUIRED COLUMNS:', missing)

        if missing:
            print('STOP: Schema requires investigation.')
            return

    original = pd.read_csv(PRE, low_memory=False)
    corrected = pd.read_csv(POST, low_memory=False)

    expected = {
        'PRE_PURGE': 103526,
        'POST_PURGE': 103412
    }

    for name, df in [
        ('PRE_PURGE', original),
        ('POST_PURGE', corrected)
    ]:
        print('\n' + '-' * 75)
        print(name)
        print('-' * 75)

        print('Rows:', len(df))
        print('Expected:', expected[name])
        print('ROW_COUNT:', (
            'PASS' if len(df) == expected[name]
            else 'REVIEW'
        ))
        print('Unique series:', df['series'].nunique())

        points = pd.to_numeric(
            df['n_points'], errors='coerce'
        )
        single = points.eq(1)

        print('Single-observation windows:', int(single.sum()))
        print('Single-observation percentage:',
              round(100 * single.mean(), 6))
        print('Invalid n_points:', int(points.isna().sum()))

        if name == 'PRE_PURGE':
            print('MAIN_BRANCH_34588_CHECK:',
                  'PASS' if int(single.sum()) == 34588
                  else 'REVIEW')

        if name == 'POST_PURGE':
            split_table = pd.DataFrame({
                'split': df['split'],
                'single': single
            }).groupby('split', dropna=False).agg(
                windows=('single', 'size'),
                single_windows=('single', 'sum')
            )

            split_table['single_pct'] = (
                100 * split_table['single_windows']
                / split_table['windows']
            ).round(6)

            print('\nPOST-PURGE SPLIT COVERAGE:')
            print(split_table.to_string())

            print('\nEXPECTED SPLIT COUNTS:')
            print('train=62096, validation=20648, test=20668')

    # Verify the actual original main-feature values.
    single_rows = original.loc[
        pd.to_numeric(
            original['n_points'], errors='coerce'
        ).eq(1)
    ]

    print('\n' + '=' * 75)
    print('SINGLE-OBSERVATION DEGENERATE FEATURE AUDIT')
    print('=' * 75)

    for feature in DEGENERATE_ZERO_FEATURES:
        values = pd.to_numeric(
            single_rows[feature], errors='coerce'
        ).to_numpy(dtype=float)

        nonfinite = int((~np.isfinite(values)).sum())
        nonzero = int(np.sum(
            np.isfinite(values)
            & ~np.isclose(values, 0, atol=1e-12, rtol=0)
        ))

        status = (
            'PASS' if nonfinite == 0 and nonzero == 0
            else 'REVIEW'
        )

        print(
            f'{feature:20s} {status:7s} '
            f'nonzero={nonzero} nonfinite={nonfinite}'
        )

    print('\nNONFINITE COUNTS: ALL 23 ORIGINAL FEATURES')
    for feature in FEATURES:
        numeric = pd.to_numeric(
            single_rows[feature], errors='coerce'
        ).to_numpy(dtype=float)
        bad = int((~np.isfinite(numeric)).sum())
        print(f'{feature:20s}: {bad}')

    print('\nSAMPLE: FIRST 3 SINGLE-OBSERVATION WINDOWS')
    columns = [
        'series', 'window_start', 'n_points',
        'mean', 'std', 'var', 'skew',
        'kurtosis', 'slope', 'energy', 'rms'
    ]
    print(single_rows[columns].head(3).to_string(index=False))

    print('\nCELL 2 COMPLETE — READ ONLY')
    print('No source files modified.')
    print('No models trained or thresholds changed.')
    print('C3-5 remains IN PROGRESS pending review.')

verify_nab()


In [ ]:

# CHECKPOINT 4 — C3-5 — CELL 3
# Final descriptive audit of corrected NAB windows
# STRICTLY READ-ONLY

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path('/content/drive/MyDrive/Master/Thesis/Code')

assert BASE.is_dir(), (
    'STOP: Thesis Code directory not found.'
)

PATH = (
    BASE / 'experiments'
    / 'EXP_SPLIT_BOUNDARY_PURGE_CORRECTION'
    / 'NAB' / '60_30'
    / 'all_corrected_partitions.csv'
)

print("=" * 75)
print("C3-5 — FINAL NAB SINGLE-OBSERVATION DISTRIBUTION AUDIT")
print("=" * 75)

if not PATH.is_file():
    raise SystemExit(
        f"STOP: Authoritative corrected NAB artifact unavailable: {PATH}"
    )

# Read only the columns needed for this audit.
COLUMNS = ['series', 'split', 'n_points', 'label']

headers = pd.read_csv(PATH, nrows=0).columns.tolist()
missing = [c for c in COLUMNS if c not in headers]

if missing:
    raise SystemExit(
        f"STOP: Missing required columns: {missing}"
    )

df = pd.read_csv(PATH, usecols=COLUMNS, low_memory=False)

df['n_points'] = pd.to_numeric(
    df['n_points'], errors='coerce'
)

single = df['n_points'].eq(1)

print("\nDATASET SUMMARY")
print("Total windows:", len(df))
print("Unique series:", df['series'].nunique())
print("Single-observation windows:", int(single.sum()))
print("Multi-observation windows:", int((df['n_points'] > 1).sum()))
print("Zero-observation windows:", int(df['n_points'].eq(0).sum()))
print("Invalid n_points:", int(df['n_points'].isna().sum()))

print("\nN_POINTS DISTRIBUTION — TOP 15 VALUES")
print(
    df['n_points']
      .value_counts(dropna=False)
      .head(15)
      .to_string()
)

# Series-level coverage, with no series excluded.
series_audit = (
    df.assign(single=single)
      .groupby('series', dropna=False)
      .agg(
          total_windows=('single', 'size'),
          single_windows=('single', 'sum')
      )
)

series_audit['single_pct'] = (
    100 * series_audit['single_windows']
    / series_audit['total_windows']
)

print("\nSERIES WITH SINGLE-OBSERVATION WINDOWS:",
      int((series_audit['single_windows'] > 0).sum()))

print("\nTOP 12 SERIES BY SINGLE-OBSERVATION COUNT")
print(
    series_audit
      .sort_values('single_windows', ascending=False)
      .head(12)
      .round(3)
      .to_string()
)

print("\nTOP 12 SERIES BY SINGLE-OBSERVATION PERCENTAGE")
print(
    series_audit
      .sort_values('single_pct', ascending=False)
      .head(12)
      .round(3)
      .to_string()
)

# Labels are examined only as existing NAB annotations.
# They are NOT used for training or threshold selection.
print("\nEXISTING LABEL VALUES")
print(
    df['label']
      .value_counts(dropna=False)
      .to_string()
)

label_audit = pd.crosstab(
    index=[
        df['split'],
        pd.Series(
            np.where(single, 'single', 'other'),
            index=df.index,
            name='window_type'
        )
    ],
    columns=df['label'],
    dropna=False
)

print("\nLABEL DISTRIBUTION BY SPLIT AND WINDOW TYPE")
print(label_audit.to_string())

print("\nFINAL VERIFICATION")
print("ROW_COUNT:",
      "PASS" if len(df) == 103412 else "REVIEW")

print("SINGLE_COUNT:",
      "PASS" if int(single.sum()) == 34574 else "REVIEW")

print("SPLIT_COUNTS:")
print(df['split'].value_counts().to_string())

print("\nCELL 3 COMPLETE — READ ONLY")
print("No source files modified.")
print("No windows excluded.")
print("No labels or thresholds changed.")
print("No models trained or evaluated.")
print("C3-5 remains IN PROGRESS until thesis revision.")


In [ ]:

# ============================================================
# D3 / C3-9 — CELL 1
# Verify historical residential scoring artifacts and schemas
# READ-ONLY: No fitting, recalibration, modification, or saving
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd

BASE = Path('/content/drive/MyDrive/Master/Thesis/Code')
assert BASE.is_dir(), 'STOP: Thesis Code directory not found.'

PATHS = {
    'UK-DALE': BASE / 'experiments/EXP_SPLIT_BOUNDARY_PURGE_CORRECTION/UKDALE/ukdale_if_scored_boundary_corrected.csv',
    'AMPds2': BASE / 'experiments/EXP_SPLIT_BOUNDARY_PURGE_CORRECTION/AMPDS2/ampds2_if_scored_boundary_corrected.csv'
}

print('=' * 78)
print('D3 / C3-9 — RESIDENTIAL HISTORICAL SCORE ARTIFACT VERIFICATION')
print('=' * 78)
print('Drive root:', BASE)

missing = []

for dataset, path in PATHS.items():
    print(f'\n[{dataset}]')
    print('Exact path:', path)
    if path.is_file():
        print('Status: FOUND')
        print('File size (bytes):', path.stat().st_size)
    else:
        print('Status: NOT FOUND')
        missing.append(dataset)

if missing:
    raise RuntimeError(f'STOP: Required scoring file(s) unavailable: {missing}. No alternative paths searched.')

for dataset, path in PATHS.items():
    print('\n' + '-' * 78)
    print(f'{dataset} — READ-ONLY SCHEMA INSPECTION')
    print('-' * 78)

    sample = pd.read_csv(path, nrows=3)
    columns = sample.columns.tolist()

    print('Column count:', len(columns))
    print('All columns:')
    for column in columns:
        print('  ', repr(column))

    keywords = ('score', 'split', 'channel', 'meter', 'appliance', 'flag', 'anomal', 'series', 'row', 'source', 'partition')
    relevant_columns = [col for col in columns if any(key in col.lower() for key in keywords)]

    print('\nPotential score, partition, and channel metadata columns:')
    print(relevant_columns)

    if relevant_columns:
        print('\nFirst three rows — relevant columns only:')
        print(sample[relevant_columns].to_string(index=False))
    else:
        print('No matching metadata columns identified. Manual schema review required.')

print('\n' + '=' * 78)
print('CELL 1 COMPLETE — READ ONLY')
print('No full datasets loaded.')
print('No thresholds calculated or changed.')
print('No models trained or evaluated.')
print('No files modified or created.')
print('D3 / C3-9 remains IN PROGRESS.')
print('=' * 78)


In [ ]:

# ============================================================
# D3 / C3-9 — CELL 2 (CORRECTED)
# Historical pooled-threshold verification
# READ ONLY — No training, recalibration, or file changes
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path('/content/drive/MyDrive/Master/Thesis/Code')
assert BASE.is_dir(), 'STOP: Thesis Code directory not found.'

CONFIG = {
    'UK-DALE': {
        'path': BASE / 'experiments/EXP_SPLIT_BOUNDARY_PURGE_CORRECTION/UKDALE/ukdale_if_scored_boundary_corrected.csv',
        'threshold': 0.058252700304033553,
        'train': 218358,
        'validation': 72771,
        'test': 72780
    },
    'AMPds2': {
        'path': BASE / 'experiments/EXP_SPLIT_BOUNDARY_PURGE_CORRECTION/AMPDS2/ampds2_if_scored_boundary_corrected.csv',
        'threshold': 0.029990580900317053,
        'train': 441483,
        'validation': 147147,
        'test': 147147
    }
}

COLUMNS = [
    'split',
    'anomaly_score',
    'if_decision_function',
    'if_threshold_corrected',
    'if_threshold_percentile_corrected',
    'predicted_anomaly_corrected'
]

TOL = 1e-10

def audit_residential_thresholds():

    print('=' * 85)
    print('D3 / C3-9 — FROZEN POOLED-THRESHOLD VERIFICATION')
    print('=' * 85)

    for dataset, config in CONFIG.items():

        print('\n' + '=' * 85)
        print('DATASET:', dataset)
        print('=' * 85)

        path = config['path']

        if not path.is_file():
            print('STOP: Historical file not found:', path)
            return

        headers = pd.read_csv(path, nrows=0).columns.tolist()
        missing = [c for c in COLUMNS if c not in headers]

        if missing:
            print('STOP: Missing columns:', missing)
            return

        df = pd.read_csv(
            path,
            usecols=COLUMNS,
            low_memory=False,
            float_precision='round_trip'
        )

        # ----------------------------------------------------
        # 1. Verify historical split identifiers and counts
        # ----------------------------------------------------

        print('\n1. HISTORICAL POPULATION')

        counts = df['split'].value_counts(dropna=False)

        print('Saved split identifiers and counts:')
        print(counts.to_string())

        present = set(df['split'].dropna().unique())

        validation_labels = [
            s for s in ('val', 'validation') if s in present
        ]

        if len(validation_labels) != 1:
            print('STOP: Validation split identifier is ambiguous.')
            return

        validation_label = validation_labels[0]

        expected = {
            'train': config['train'],
            validation_label: config['validation'],
            'test': config['test']
        }

        population_pass = (
            len(df) == sum(expected.values())
            and present == set(expected)
            and counts.to_dict() == expected
        )

        print('\nHistorical validation label:', repr(validation_label))
        print('Total windows:', len(df))
        print('Expected total:', sum(expected.values()))
        print('POPULATION:', 'PASS' if population_pass else 'REVIEW')

        if not population_pass:
            print('STOP: Population mismatch.')
            return

        # ----------------------------------------------------
        # 2. Verify original continuous anomaly-score target
        # ----------------------------------------------------

        print('\n2. ORIGINAL SCORE IDENTITY')

        scores = pd.to_numeric(
            df['anomaly_score'], errors='coerce'
        ).to_numpy(dtype=float)

        decisions = pd.to_numeric(
            df['if_decision_function'], errors='coerce'
        ).to_numpy(dtype=float)

        if not (
            np.isfinite(scores).all()
            and np.isfinite(decisions).all()
        ):
            print('STOP: Nonfinite model scores.')
            return

        max_difference = float(
            np.max(np.abs(scores + decisions))
        )

        score_pass = max_difference <= TOL

        print('Expected: anomaly_score = -if_decision_function')
        print('Maximum discrepancy:', f'{max_difference:.15g}')
        print('SCORE IDENTITY:', 'PASS' if score_pass else 'REVIEW')

        if not score_pass:
            print('STOP: Historical scoring identity mismatch.')
            return

        # ----------------------------------------------------
        # 3. Verify original validation-pooled threshold
        # ----------------------------------------------------

        print('\n3. FROZEN POOLED 95TH-PERCENTILE THRESHOLD')

        val_mask = df['split'].eq(validation_label).to_numpy()
        eval_mask = df['split'].isin(
            [validation_label, 'test']
        ).to_numpy()

        frozen_threshold = config['threshold']

        reproduced_threshold = float(
            np.quantile(scores[val_mask], 0.95)
        )

        threshold_error = abs(
            reproduced_threshold - frozen_threshold
        )

        stored_thresholds = pd.to_numeric(
            df.loc[eval_mask, 'if_threshold_corrected'],
            errors='coerce'
        ).to_numpy(dtype=float)

        if not np.isfinite(stored_thresholds).all():
            print('STOP: Invalid saved corrected thresholds.')
            return

        saved_threshold_error = float(
            np.max(np.abs(stored_thresholds - frozen_threshold))
        )

        percentiles = pd.to_numeric(
            df.loc[eval_mask, 'if_threshold_percentile_corrected'],
            errors='coerce'
        ).to_numpy(dtype=float)

        percentile_pass = (
            np.isfinite(percentiles).all()
            and np.all(
                np.isclose(percentiles, 95.0, atol=1e-12, rtol=0)
                | np.isclose(percentiles, 0.95, atol=1e-12, rtol=0)
            )
        )

        print('Frozen threshold:', repr(frozen_threshold))
        print('Validation-derived threshold:', repr(reproduced_threshold))
        print('Reproduction discrepancy:', f'{threshold_error:.15g}')
        print('Saved-threshold discrepancy:', f'{saved_threshold_error:.15g}')
        print('Saved percentile values:', np.unique(percentiles))
        print('PERCENTILE METADATA:', 'PASS' if percentile_pass else 'REVIEW')

        threshold_pass = (
            threshold_error <= TOL
            and saved_threshold_error <= TOL
            and percentile_pass
        )

        print('FROZEN THRESHOLD:', 'PASS' if threshold_pass else 'REVIEW')

        if not threshold_pass:
            print('STOP: Threshold verification needs investigation.')
            return

        # ----------------------------------------------------
        # 4. Verify the original corrected binary decisions
        # ----------------------------------------------------

        print('\n4. HISTORICAL CORRECTED FLAG VERIFICATION')

        saved_flags = pd.to_numeric(
            df.loc[eval_mask, 'predicted_anomaly_corrected'],
            errors='coerce'
        ).to_numpy(dtype=float)

        if not (
            np.isfinite(saved_flags).all()
            and np.isin(saved_flags, [0, 1]).all()
        ):
            print('STOP: Invalid corrected flags.')
            return

        recomputed_flags = (
            scores[eval_mask] > frozen_threshold
        ).astype(int)

        mismatches = int(
            np.count_nonzero(saved_flags != recomputed_flags)
        )

        print('Decision rule: STRICT anomaly_score > frozen threshold')
        print('Flag mismatches:', mismatches)
        print('CORRECTED FLAGS:', 'PASS' if mismatches == 0 else 'REVIEW')

        if mismatches != 0:
            print('STOP: Historical prediction mismatch.')
            return

        # ----------------------------------------------------
        # 5. Report frozen pooled validation/test flagging
        # ----------------------------------------------------

        print('\n5. HISTORICAL POOLED FLAGGING')

        for split in [validation_label, 'test']:

            mask = df['split'].eq(split).to_numpy()

            total = int(mask.sum())
            flagged = int(
                np.count_nonzero(scores[mask] > frozen_threshold)
            )

            rate = 100.0 * flagged / total

            print(
                f'{split:12s} '
                f'flagged={flagged:7d} / {total:7d} '
                f'rate={rate:.6f}%'
            )

        print('\nDATASET VERIFICATION: PASS')

        del df, scores, decisions

    print('\n' + '=' * 85)
    print('CELL 2 OVERALL STATUS: PASS')
    print('Both historical pooled-threshold protocols verified.')
    print('No data, model, threshold, or prediction modified.')
    print('No files written.')
    print('D3 / C3-9 remains IN PROGRESS.')
    print('=' * 85)

audit_residential_thresholds()


In [ ]:

# ============================================================
# D3 / C3-9 — CELL 3
# Channel-level behavior under the ORIGINAL pooled thresholds
# READ ONLY — no fitting, no recalibration, no file writes
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path('/content/drive/MyDrive/Master/Thesis/Code')
assert BASE.is_dir(), 'STOP: Thesis Code directory not found.'

CONFIG = {
    'UK-DALE': {
        'path': BASE / 'experiments/EXP_SPLIT_BOUNDARY_PURGE_CORRECTION/UKDALE/ukdale_if_scored_boundary_corrected.csv',
        'threshold': 0.058252700304033553,
        'identifier': 'series_key',
        'metadata': ['building', 'meter', 'primary_category'],
        'expected_channels': 16,
        'expected_splits': {'train': 218358, 'val': 72771, 'test': 72780},
        'expected_flags': {'val': 3639, 'test': 3551}
    },
    'AMPds2': {
        'path': BASE / 'experiments/EXP_SPLIT_BOUNDARY_PURGE_CORRECTION/AMPDS2/ampds2_if_scored_boundary_corrected.csv',
        'threshold': 0.029990580900317053,
        'identifier': 'channel_code',
        'metadata': ['channel_name', 'primary_category'],
        'expected_channels': 21,
        'expected_splits': {'train': 441483, 'val': 147147, 'test': 147147},
        'expected_flags': {'val': 7358, 'test': 14560}
    }
}

def run_channel_audit():

    print('=' * 90)
    print('D3 / C3-9 — POOLED-THRESHOLD CHANNEL-LEVEL AUDIT')
    print('=' * 90)

    for dataset, cfg in CONFIG.items():

        print('\n' + '=' * 90)
        print('DATASET:', dataset)
        print('=' * 90)

        path = cfg['path']
        key = cfg['identifier']
        metadata_cols = cfg['metadata']

        if not path.is_file():
            print('STOP: Historical scored file unavailable:', path)
            return

        required = [
            'split',
            'anomaly_score',
            'if_threshold_corrected',
            'predicted_anomaly_corrected',
            key
        ] + metadata_cols

        actual_columns = pd.read_csv(path, nrows=0).columns.tolist()
        missing = [col for col in required if col not in actual_columns]

        if missing:
            print('STOP: Missing columns:', missing)
            return

        df = pd.read_csv(
            path,
            usecols=required,
            low_memory=False,
            float_precision='round_trip'
        )

        split_counts = df['split'].value_counts(dropna=False).to_dict()

        if split_counts != cfg['expected_splits']:
            print('STOP: Historical split population mismatch.')
            print('Actual:', split_counts)
            print('Expected:', cfg['expected_splits'])
            return

        if df[key].isna().any():
            print('STOP: Missing channel identifiers.')
            return

        unique_channels = df[key].nunique()

        print('Historical rows:', len(df))
        print('Distinct channels/meters:', unique_channels)
        print('Frozen pooled threshold:', repr(cfg['threshold']))

        if unique_channels != cfg['expected_channels']:
            print('STOP: Unexpected number of channels/meters.')
            return

        # Restrict this audit to the original corrected
        # validation and test partitions.
        evaluated = df.loc[
            df['split'].isin(['val', 'test'])
        ].copy()

        evaluated['anomaly_score'] = pd.to_numeric(
            evaluated['anomaly_score'], errors='coerce'
        )

        evaluated['if_threshold_corrected'] = pd.to_numeric(
            evaluated['if_threshold_corrected'], errors='coerce'
        )

        saved_flags = pd.to_numeric(
            evaluated['predicted_anomaly_corrected'],
            errors='coerce'
        )

        if not np.isfinite(evaluated['anomaly_score'].to_numpy()).all():
            print('STOP: Nonfinite anomaly scores.')
            return

        if not np.isfinite(evaluated['if_threshold_corrected'].to_numpy()).all():
            print('STOP: Invalid saved thresholds.')
            return

        if not np.isfinite(saved_flags.to_numpy()).all():
            print('STOP: Nonfinite corrected predictions.')
            return

        if not saved_flags.isin([0, 1]).all():
            print('STOP: Nonbinary corrected predictions.')
            return

        threshold_matches = np.isclose(
            evaluated['if_threshold_corrected'].to_numpy(),
            cfg['threshold'],
            atol=1e-12,
            rtol=0
        ).all()

        if not threshold_matches:
            print('STOP: Saved corrected thresholds do not match the frozen pooled threshold.')
            return

        # Apply ONLY the original historical decision rule.
        # This creates a temporary in-memory column.
        evaluated['frozen_flag'] = (
            evaluated['anomaly_score'] > cfg['threshold']
        ).astype(np.int8)

        mismatches = int(
            np.count_nonzero(
                evaluated['frozen_flag'].to_numpy() != saved_flags.to_numpy()
            )
        )

        print('Corrected decision mismatches:', mismatches)

        if mismatches != 0:
            print('STOP: Flag verification failed.')
            return

        # Ensure that descriptive metadata map uniquely
        # to the original physical meter/channel identifier.
        metadata = evaluated[[key] + metadata_cols].drop_duplicates()

        if metadata[key].duplicated().any():
            print('STOP: Inconsistent channel metadata.')
            print(metadata.loc[metadata[key].duplicated(keep=False)].to_string(index=False))
            return

        metadata = metadata.set_index(key)

        validation = (
            evaluated.loc[evaluated['split'].eq('val')]
            .groupby(key)
            .agg(
                val_windows=('frozen_flag', 'size'),
                val_flagged=('frozen_flag', 'sum')
            )
        )

        test = (
            evaluated.loc[evaluated['split'].eq('test')]
            .groupby(key)
            .agg(
                test_windows=('frozen_flag', 'size'),
                test_flagged=('frozen_flag', 'sum')
            )
        )

        result = metadata.join(validation, how='outer').join(test, how='outer')

        if result[['val_windows', 'val_flagged', 'test_windows', 'test_flagged']].isna().any().any():
            print('STOP: At least one channel is missing from validation or test.')
            return

        result['val_rate_pct'] = (
            100.0 * result['val_flagged'] / result['val_windows']
        )

        result['test_rate_pct'] = (
            100.0 * result['test_flagged'] / result['test_windows']
        )

        result['test_minus_val_pp'] = (
            result['test_rate_pct'] - result['val_rate_pct']
        )

        result = result.sort_values(
            ['val_rate_pct', 'test_rate_pct'],
            ascending=[False, False]
        )

        total_val = int(result['val_windows'].sum())
        total_test = int(result['test_windows'].sum())
        flagged_val = int(result['val_flagged'].sum())
        flagged_test = int(result['test_flagged'].sum())

        if (
            total_val != cfg['expected_splits']['val']
            or total_test != cfg['expected_splits']['test']
            or flagged_val != cfg['expected_flags']['val']
            or flagged_test != cfg['expected_flags']['test']
        ):
            print('STOP: Channel aggregation does not reproduce historical totals.')
            print('Totals:', total_val, total_test, flagged_val, flagged_test)
            return

        print('\nORIGINAL POOLED CALIBRATION — ALL CHANNELS')
        print('Rates are percentages; differences are percentage points.')
        print('Sorted by validation flagged-window rate, highest first.\n')

        print(
            result.reset_index().to_string(
                index=False,
                float_format=lambda value: f'{value:.4f}'
            )
        )

        print('\nAGGREGATE VERIFICATION')
        print(f'Validation: {flagged_val:,} / {total_val:,} = {100.0 * flagged_val / total_val:.6f}%')
        print(f'Test:       {flagged_test:,} / {total_test:,} = {100.0 * flagged_test / total_test:.6f}%')
        print('CHANNEL AGGREGATION: PASS')

        print('\nCHANNEL-LEVEL VALIDATION DIAGNOSTICS')
        print('Minimum validation rate:', f"{result['val_rate_pct'].min():.6f}%")
        print('Median validation rate:', f"{result['val_rate_pct'].median():.6f}%")
        print('Maximum validation rate:', f"{result['val_rate_pct'].max():.6f}%")
        print('Channels above 5% validation flagging:', int((result['val_rate_pct'] > 5.0).sum()))
        print('Channels below 5% validation flagging:', int((result['val_rate_pct'] < 5.0).sum()))

        if dataset == 'AMPds2':
            if 'EQE' not in result.index:
                print('STOP: Expected EQE channel not found.')
                return

            eqe = result.loc['EQE']

            print('\nEQE SECURITY/NETWORK — ORIGINAL POOLED THRESHOLD')
            print('Validation windows:', int(eqe['val_windows']))
            print('Validation flagged:', int(eqe['val_flagged']))
            print('Validation rate:', f"{eqe['val_rate_pct']:.6f}%")
            print('Test windows:', int(eqe['test_windows']))
            print('Test flagged:', int(eqe['test_flagged']))
            print('Test rate:', f"{eqe['test_rate_pct']:.6f}%")
            print('Test minus validation:', f"{eqe['test_minus_val_pp']:+.6f} percentage points")

        print('\nDATASET AUDIT: PASS')

        del df, evaluated, result

    print('\n' + '=' * 90)
    print('CELL 3 COMPLETE — BOTH DATASETS PASSED')
    print('Original pooled thresholds preserved.')
    print('Only historical saved scores and corrected decisions used.')
    print('No per-channel thresholds calculated.')
    print('No model training or synthetic anomaly injection.')
    print('No historical files modified and no new files saved.')
    print('C3-9 remains IN PROGRESS pending sensitivity analysis and thesis integration.')
    print('=' * 90)

run_channel_audit()


In [ ]:

# ============================================================
# D3 / C3-9 — CELL 4
# PER-CHANNEL THRESHOLD-ONLY SENSITIVITY ANALYSIS
#
# PRIMARY METHOD: Frozen pooled 95th-percentile threshold
# SENSITIVITY ONLY: Per-channel validation 95th percentile
#
# READ ONLY — No model fitting, no historical modifications
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path('/content/drive/MyDrive/Master/Thesis/Code')
assert BASE.is_dir(), 'STOP: Thesis Code directory not found.'

CONFIG = {
    'UK-DALE': {
        'path': BASE / 'experiments/EXP_SPLIT_BOUNDARY_PURGE_CORRECTION/UKDALE/ukdale_if_scored_boundary_corrected.csv',
        'key': 'series_key',
        'pooled_threshold': 0.058252700304033553,
        'expected_channels': 16,
        'expected_splits': {'train': 218358, 'val': 72771, 'test': 72780},
        'expected_primary_flags': {'val': 3639, 'test': 3551}
    },
    'AMPds2': {
        'path': BASE / 'experiments/EXP_SPLIT_BOUNDARY_PURGE_CORRECTION/AMPDS2/ampds2_if_scored_boundary_corrected.csv',
        'key': 'channel_code',
        'pooled_threshold': 0.029990580900317053,
        'expected_channels': 21,
        'expected_splits': {'train': 441483, 'val': 147147, 'test': 147147},
        'expected_primary_flags': {'val': 7358, 'test': 14560}
    }
}

def run_d3_threshold_sensitivity():

    print('=' * 95)
    print('D3 / C3-9 — PER-CHANNEL THRESHOLD-ONLY SENSITIVITY')
    print('PRIMARY POOLED THRESHOLDS REMAIN UNCHANGED')
    print('=' * 95)

    for dataset, cfg in CONFIG.items():

        print('\n' + '=' * 95)
        print('DATASET:', dataset)
        print('=' * 95)

        path = cfg['path']
        key = cfg['key']
        pooled_threshold = cfg['pooled_threshold']

        if not path.is_file():
            raise SystemExit(f'STOP: Historical file unavailable: {path}')

        required = [
            key,
            'split',
            'anomaly_score',
            'if_threshold_corrected',
            'predicted_anomaly_corrected'
        ]

        columns = pd.read_csv(path, nrows=0).columns.tolist()
        missing = [c for c in required if c not in columns]

        if missing:
            raise SystemExit(f'STOP: Missing columns in {dataset}: {missing}')

        df = pd.read_csv(
            path,
            usecols=required,
            low_memory=False,
            float_precision='round_trip'
        )

        # 1. Preserve and verify the historical population.
        actual_splits = df['split'].value_counts(dropna=False).to_dict()

        if actual_splits != cfg['expected_splits']:
            raise SystemExit(
                f'STOP: Historical population mismatch in {dataset}: {actual_splits}'
            )

        if df[key].isna().any():
            raise SystemExit(f'STOP: Missing channel identifiers in {dataset}')

        if df[key].nunique() != cfg['expected_channels']:
            raise SystemExit(f'STOP: Unexpected channel count in {dataset}')

        df['anomaly_score'] = pd.to_numeric(
            df['anomaly_score'], errors='coerce'
        )

        if not np.isfinite(df['anomaly_score'].to_numpy()).all():
            raise SystemExit(f'STOP: Nonfinite historical scores in {dataset}')

        evaluated = df.loc[df['split'].isin(['val', 'test'])].copy()

        saved_thresholds = pd.to_numeric(
            evaluated['if_threshold_corrected'], errors='coerce'
        ).to_numpy(dtype=float)

        if not np.isfinite(saved_thresholds).all():
            raise SystemExit(f'STOP: Invalid historical thresholds in {dataset}')

        if not np.allclose(
            saved_thresholds, pooled_threshold,
            rtol=0, atol=1e-12
        ):
            raise SystemExit(f'STOP: Frozen threshold mismatch in {dataset}')

        saved_flags = pd.to_numeric(
            evaluated['predicted_anomaly_corrected'], errors='coerce'
        ).to_numpy(dtype=float)

        if not np.isfinite(saved_flags).all() or not np.isin(saved_flags, [0, 1]).all():
            raise SystemExit(f'STOP: Invalid historical flags in {dataset}')

        evaluated['primary_flag'] = (
            evaluated['anomaly_score'] > pooled_threshold
        ).astype(np.int8)

        mismatch_count = int(
            np.count_nonzero(
                evaluated['primary_flag'].to_numpy() != saved_flags
            )
        )

        if mismatch_count != 0:
            raise SystemExit(
                f'STOP: {mismatch_count} historical decision mismatches in {dataset}'
            )

        print('Historical population: PASS')
        print('Historical corrected decisions: PASS')
        print('Primary pooled threshold:', repr(pooled_threshold))
        print('Channels:', cfg['expected_channels'])

        # 2. Calculate ALTERNATIVE channel-specific thresholds.
        #    Validation scores only. No labels or test scores.
        validation = evaluated.loc[evaluated['split'].eq('val')]

        channel_thresholds = (
            validation
            .groupby(key)['anomaly_score']
            .quantile(0.95, interpolation='linear')
        )

        if len(channel_thresholds) != cfg['expected_channels']:
            raise SystemExit(f'STOP: Missing validation channels in {dataset}')

        if not np.isfinite(channel_thresholds.to_numpy()).all():
            raise SystemExit(f'STOP: Invalid sensitivity thresholds in {dataset}')

        # Temporary in-memory analysis columns only.
        evaluated['sensitivity_threshold'] = (
            evaluated[key].map(channel_thresholds)
        )

        if evaluated['sensitivity_threshold'].isna().any():
            raise SystemExit(f'STOP: Unmapped sensitivity thresholds in {dataset}')

        evaluated['sensitivity_flag'] = (
            evaluated['anomaly_score']
            > evaluated['sensitivity_threshold']
        ).astype(np.int8)

        # 3. Build a descriptive comparison for each channel.
        summary = pd.DataFrame(index=channel_thresholds.index)
        summary['channel_q95'] = channel_thresholds

        for split in ['val', 'test']:

            grouped = (
                evaluated.loc[evaluated['split'].eq(split)]
                .groupby(key)
                .agg(
                    windows=('anomaly_score', 'size'),
                    primary_flags=('primary_flag', 'sum'),
                    sensitivity_flags=('sensitivity_flag', 'sum')
                )
            )

            summary[f'{split}_windows'] = grouped['windows']
            summary[f'{split}_primary_flags'] = grouped['primary_flags']
            summary[f'{split}_sensitivity_flags'] = grouped['sensitivity_flags']

            summary[f'{split}_primary_pct'] = (
                100 * grouped['primary_flags'] / grouped['windows']
            )

            summary[f'{split}_sensitivity_pct'] = (
                100 * grouped['sensitivity_flags'] / grouped['windows']
            )

        if summary.isna().any().any():
            raise SystemExit(f'STOP: Incomplete channel comparison in {dataset}')

        summary['test_change_pp'] = (
            summary['test_sensitivity_pct']
            - summary['test_primary_pct']
        )

        # 4. Reconcile primary results before reporting sensitivity.
        for split in ['val', 'test']:

            actual_count = int(summary[f'{split}_primary_flags'].sum())
            expected_count = cfg['expected_primary_flags'][split]

            if actual_count != expected_count:
                raise SystemExit(
                    f'STOP: Primary {split} flag mismatch in {dataset}: '
                    f'{actual_count} versus {expected_count}'
                )

            total = int(summary[f'{split}_windows'].sum())

            if total != cfg['expected_splits'][split]:
                raise SystemExit(f'STOP: {split} population mismatch in {dataset}')

        # 5. Report channel-level threshold-only sensitivity.
        output_columns = [
            'channel_q95',
            'val_windows',
            'val_primary_pct',
            'val_sensitivity_pct',
            'test_windows',
            'test_primary_flags',
            'test_sensitivity_flags',
            'test_primary_pct',
            'test_sensitivity_pct',
            'test_change_pp'
        ]

        summary = summary.sort_values(
            'test_primary_pct', ascending=False
        )

        print('\nPER-CHANNEL RESULTS — SENSITIVITY ONLY')
        print('channel_q95 = alternative validation-only threshold')
        print('test_change_pp = sensitivity test rate minus PRIMARY test rate')
        print('Negative change indicates fewer model-generated test flags.')
        print('It does NOT establish improved anomaly-detection accuracy.\n')

        print(
            summary[output_columns].to_string(
                float_format=lambda value: f'{value:.6f}'
            )
        )

        # 6. Dataset-wide comparison of primary vs sensitivity.
        print('\nDATASET-WIDE FLAGGING COMPARISON')

        for split in ['val', 'test']:

            total = int(summary[f'{split}_windows'].sum())
            original = int(summary[f'{split}_primary_flags'].sum())
            sensitivity = int(summary[f'{split}_sensitivity_flags'].sum())

            original_pct = 100 * original / total
            sensitivity_pct = 100 * sensitivity / total

            print(
                f'{split.upper():5s} | '
                f'windows={total:,} | '
                f'PRIMARY={original:,} ({original_pct:.6f}%) | '
                f'SENSITIVITY={sensitivity:,} ({sensitivity_pct:.6f}%) | '
                f'change={sensitivity_pct - original_pct:+.6f} pp'
            )

        if dataset == 'AMPds2':

            if 'EQE' not in summary.index:
                raise SystemExit('STOP: EQE missing from AMPds2 analysis.')

            eqe = summary.loc['EQE']

            print('\nEQE SECURITY/NETWORK — THRESHOLD-ONLY SENSITIVITY')
            print('Original pooled threshold:', repr(pooled_threshold))
            print('Alternative EQE validation Q95:', repr(float(eqe['channel_q95'])))
            print('Original validation rate:', f"{eqe['val_primary_pct']:.6f}%")
            print('Sensitivity validation rate:', f"{eqe['val_sensitivity_pct']:.6f}%")
            print('Original test rate:', f"{eqe['test_primary_pct']:.6f}%")
            print('Sensitivity test rate:', f"{eqe['test_sensitivity_pct']:.6f}%")
            print('Test flagging difference:', f"{eqe['test_change_pp']:+.6f} percentage points")

        print('\nDATASET STATUS: PASS')
        print('Original pooled method retained as PRIMARY.')
        print('Channel-specific calibration examined as SENSITIVITY ONLY.')

        del df, evaluated, summary

    print('\n' + '=' * 95)
    print('CELL 4 COMPLETE — BOTH DATASETS PASSED')
    print('No models trained or retrained.')
    print('No test scores used to derive thresholds.')
    print('No labels used.')
    print('No datasets, historical predictions or primary thresholds modified.')
    print('No files written.')
    print('C3-9 remains IN PROGRESS pending interpretation and manuscript integration.')
    print('=' * 95)

run_d3_threshold_sensitivity()


In [ ]:

# ============================================================
# D4 / C3-11 — CELL 1
# TREE SHAP: HISTORICAL MODEL AND XAI ARTIFACT INVENTORY
#
# READ ONLY
# No model fitting, explanation computation, or file writes
# ============================================================

from pathlib import Path
from importlib import metadata
import hashlib
import json
import pandas as pd

from google.colab import drive

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/Master/Thesis/Code')
RECOVERY = BASE / 'SUPERVISOR_FINAL_EXPERIMENTS_2026/03_integrity_checks/IF_RECOVERY'
CORRECTED = BASE / 'experiments/EXP_SPLIT_BOUNDARY_PURGE_CORRECTION'

CONFIG = {
    'UK-DALE': {
        'model': RECOVERY / 'UKDALE/ukdale_full_train_IF_FUNCTIONALLY_VERIFIED_20260928T180619_923276Z.joblib',
        'report': RECOVERY / 'UKDALE/STAGE7_CELL6C_UKDALE_IF_RECOVERY_20260928T180619_923276Z.json',
        'scores': CORRECTED / 'UKDALE/ukdale_if_scored_boundary_corrected.csv',
        'training_rows': 218358,
        'xai_dir': BASE / 'experiments/EXP_UKDALE_Selected_Pipeline/xai'
    },
    'AMPds2': {
        'model': RECOVERY / 'AMPDS2/ampds2_full_train_IF_FUNCTIONALLY_VERIFIED_20260928T181300_250116Z.joblib',
        'report': RECOVERY / 'AMPDS2/STAGE7_CELL6D_AMPDS2_IF_RECOVERY_20260928T181300_250116Z.json',
        'scores': CORRECTED / 'AMPDS2/ampds2_if_scored_boundary_corrected.csv',
        'training_rows': 441483,
        'xai_dir': BASE / 'experiments/EXP_AMPDS2_Selected_Pipeline/xai'
    }
}

FEATURES = [
    'energy', 'first_last_diff', 'iqr', 'kurtosis', 'mad',
    'max', 'max_abs_diff', 'mean', 'mean_abs_diff',
    'median', 'min', 'n_points', 'peak_count',
    'q25', 'q75', 'range', 'rms', 'skew',
    'slope', 'std', 'std_diff', 'sum', 'var'
]

SCALED_FEATURES = [f'{feature}_z' for feature in FEATURES]

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def package_version(package):
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return 'NOT INSTALLED'

print('=' * 90)
print('D4 / C3-11 — TREE SHAP HISTORICAL ARTIFACT VERIFICATION')
print('=' * 90)

print('\nSOFTWARE ENVIRONMENT')
for package in ['scikit-learn', 'shap', 'numpy', 'pandas', 'joblib']:
    print(f'{package:16s}: {package_version(package)}')

if not BASE.is_dir():
    raise RuntimeError('STOP: Thesis Code directory is missing.')

for dataset, cfg in CONFIG.items():

    print('\n' + '=' * 90)
    print('DATASET:', dataset)
    print('=' * 90)

    # 1. Check exact paths, without searching for replacements.
    for name in ['model', 'report', 'scores']:
        path = cfg[name]
        print(f'\n{name.upper()}')
        print('Path:', path)
        print('Status:', 'FOUND' if path.is_file() else 'MISSING')

        if not path.is_file():
            raise RuntimeError(
                f'STOP: Required {dataset} {name} artifact is missing.'
            )

        print('Size (bytes):', path.stat().st_size)

    # 2. Read the original model-recovery provenance.
    with cfg['report'].open('r', encoding='utf-8') as handle:
        report = json.load(handle)

    print('\nHISTORICAL MODEL PROVENANCE')

    checks = {
        'Accepted reconstruction': report.get('accepted') is True,
        'Original serialized model absent': report.get('original_fitted_model_artifact_absent') is True,
        'Functional reconstruction identified': report.get('functional_reconstruction_not_original_binary_artifact') is True,
        'Model path matches report': report.get('recovered_estimator_path') == str(cfg['model']),
        'Scoring CSV matches report': report.get('source_corrected_csv') == str(cfg['scores']),
        'Training population matches': report.get('train_rows_only') == cfg['training_rows'],
        'Canonical feature order matches': report.get('feature_order_normalized') == SCALED_FEATURES,
        'Original scoring target matches': report.get('scoring_function') == '-IsolationForest.decision_function(X)',
        'Historical score reproduction passed': report.get('acceptance_checks', {}).get('full_historical_score_match') is True,
        'Corrected score reproduction passed': report.get('acceptance_checks', {}).get('full_corrected_score_match') is True,
        'Saved corrected decisions reproduced': report.get('acceptance_checks', {}).get('corrected_saved_decision_mismatches_zero') is True,
        'Frozen model not overwritten': report.get('original_artifacts_overwritten') is False
    }

    for description, passed in checks.items():
        print(f'{"PASS" if passed else "FAIL":5s} | {description}')

    if not all(checks.values()):
        raise RuntimeError(f'STOP: {dataset} provenance verification failed.')

    # 3. Verify exact serialized artifact identity.
    expected_sha = report.get('recovered_estimator_sha256')
    actual_sha = sha256_file(cfg['model'])

    print('\nSERIALIZED MODEL INTEGRITY')
    print('Recorded SHA256:', expected_sha)
    print('Current  SHA256:', actual_sha)

    if not expected_sha or actual_sha != expected_sha:
        raise RuntimeError(f'STOP: {dataset} model SHA256 mismatch.')

    print('MODEL SHA256: PASS')

    print('\nRECOVERED MODEL SPECIFICATION')
    print('Trees:', report.get('fitted_estimators'))
    print('Maximum samples per tree:', report.get('fitted_max_samples'))
    print('Original input dtype:', report.get('train_input_dtype'))
    print('Feature count:', len(report.get('feature_order_normalized', [])))
    print('Reconstruction result:', report.get('result'))

    # 4. Read the scored CSV header only.
    score_columns = pd.read_csv(cfg['scores'], nrows=0).columns.tolist()

    required_columns = [
        'split', 'anomaly_score', 'if_decision_function',
        'if_threshold_corrected', 'predicted_anomaly_corrected'
    ] + SCALED_FEATURES

    missing = [c for c in required_columns if c not in score_columns]

    print('\nSCORING INPUT SCHEMA')
    print('Total CSV columns:', len(score_columns))
    print('Required normalized features:', len(SCALED_FEATURES))
    print('Missing required columns:', missing)

    if missing:
        raise RuntimeError(f'STOP: {dataset} input schema mismatch.')

    print('SCORING SCHEMA: PASS')

    # 5. Inspect existing local SHAP/LIME evidence.
    print('\nEXISTING HISTORICAL XAI ARTIFACTS')

    xai_dir = cfg['xai_dir']
    print('Directory:', xai_dir)

    if not xai_dir.is_dir():
        print('Historical XAI directory not found.')
        print('This does not invalidate the recovered model.')
    else:
        xai_files = sorted(
            p for p in xai_dir.iterdir()
            if p.is_file()
            and ('shap' in p.name.lower() or 'lime' in p.name.lower())
        )

        print('SHAP/LIME artifact count:', len(xai_files))

        for path in xai_files[:30]:
            print(' ', path.name, '|', path.stat().st_size, 'bytes')

        if len(xai_files) > 30:
            print(' Additional artifacts omitted from display.')

    print('\nDATASET INVENTORY: PASS')

# 6. Separate corrected historical AMPds2 LIME artifacts.
corrected_lime_dir = (
    CORRECTED / 'AMPDS2/XAI_LIME_CORRECTION'
)

print('\n' + '=' * 90)
print('AMPds2 CORRECTED HISTORICAL LIME ARTIFACTS')
print('=' * 90)
print('Directory:', corrected_lime_dir)
print('Directory exists:', corrected_lime_dir.is_dir())

if corrected_lime_dir.is_dir():
    corrected_files = sorted(
        p for p in corrected_lime_dir.iterdir() if p.is_file()
    )
    for path in corrected_files[:30]:
        print(' ', path.name)

print('\n' + '=' * 90)
print('CELL 1 COMPLETE — HISTORICAL MODEL INVENTORY VERIFIED')
print('Both recovered production Isolation Forest artifacts verified.')
print('No models loaded, trained, or retrained.')
print('No SHAP or LIME explanations calculated.')
print('No files created, overwritten, or modified.')
print('D4 / C3-11 remains IN PROGRESS.')
print('=' * 90)


In [ ]:

# ============================================================
# D4 / C3-11 — CELL 2
# INDEPENDENT HISTORICAL ISOLATION FOREST SCORE VERIFICATION
#
# READ ONLY
# No training, no model modification, no SHAP computation
# No files written
# ============================================================

from pathlib import Path
import hashlib
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

BASE = Path('/content/drive/MyDrive/Master/Thesis/Code')

RECOVERY = BASE / 'SUPERVISOR_FINAL_EXPERIMENTS_2026/03_integrity_checks/IF_RECOVERY'
CORRECTED = BASE / 'experiments/EXP_SPLIT_BOUNDARY_PURGE_CORRECTION'

CONFIG = {
    'UK-DALE': {
        'model': RECOVERY / 'UKDALE/ukdale_full_train_IF_FUNCTIONALLY_VERIFIED_20260928T180619_923276Z.joblib',
        'scores': CORRECTED / 'UKDALE/ukdale_if_scored_boundary_corrected.csv',
        'sha256': 'c1695e8c4265f6e90994be88ea7ec645acba8e10ed6240697aa73703ce3c9c5b',
        'channel': 'series_key',
        'expected_channels': 16
    },
    'AMPds2': {
        'model': RECOVERY / 'AMPDS2/ampds2_full_train_IF_FUNCTIONALLY_VERIFIED_20260928T181300_250116Z.joblib',
        'scores': CORRECTED / 'AMPDS2/ampds2_if_scored_boundary_corrected.csv',
        'sha256': '890a47ce9a2b30902010c78c0084b643c2f7b6c41257c9655d6be6cb03e2d7ca',
        'channel': 'channel_code',
        'expected_channels': 21
    }
}

FEATURES = [
    'energy', 'first_last_diff', 'iqr', 'kurtosis', 'mad',
    'max', 'max_abs_diff', 'mean', 'mean_abs_diff',
    'median', 'min', 'n_points', 'peak_count',
    'q25', 'q75', 'range', 'rms', 'skew',
    'slope', 'std', 'std_diff', 'sum', 'var'
]

SCALED = [f'{name}_z' for name in FEATURES]
EXPECTED_SPLITS = {'train', 'val', 'test'}
TOLERANCE = 1e-10

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def build_deterministic_sample(csv_path, channel_col):
    """
    Stream the historical CSV without loading the complete file.
    For every channel/split combination, retain:
      - First two observed rows
      - Two highest anomaly-score rows
      - Two lowest anomaly-score rows

    No random selection, file modification, or test-driven tuning.
    """
    required = [channel_col, 'split', 'anomaly_score',
                'if_decision_function'] + SCALED

    header = pd.read_csv(csv_path, nrows=0).columns.tolist()
    missing = [c for c in required if c not in header]

    if missing:
        raise RuntimeError(f'Missing CSV columns: {missing}')

    first = {}
    highest = {}
    lowest = {}
    row_offset = 0

    reader = pd.read_csv(
        csv_path,
        usecols=required,
        chunksize=75000,
        float_precision='round_trip',
        low_memory=False
    )

    for chunk in reader:
        chunk['__source_row__'] = np.arange(
            row_offset, row_offset + len(chunk), dtype=np.int64
        )
        row_offset += len(chunk)

        for (split, channel), group in chunk.groupby(
            ['split', channel_col], sort=False
        ):
            key = (str(split), str(channel))

            if key not in first:
                first[key] = group.head(2).copy()
            elif len(first[key]) < 2:
                first[key] = pd.concat(
                    [first[key], group.head(2)], ignore_index=True
                ).head(2)

            new_high = group.nlargest(2, 'anomaly_score')

            if key in highest:
                new_high = pd.concat(
                    [highest[key], new_high], ignore_index=True
                ).nlargest(2, 'anomaly_score')

            highest[key] = new_high.copy()

            new_low = group.nsmallest(2, 'anomaly_score')

            if key in lowest:
                new_low = pd.concat(
                    [lowest[key], new_low], ignore_index=True
                ).nsmallest(2, 'anomaly_score')

            lowest[key] = new_low.copy()

    groups = sorted(set(first) | set(highest) | set(lowest))

    frames = []
    for key in groups:
        frames.extend([first[key], highest[key], lowest[key]])

    sample = pd.concat(frames, ignore_index=True)
    sample = sample.drop_duplicates('__source_row__')
    sample = sample.sort_values('__source_row__').reset_index(drop=True)

    return sample, groups, row_offset

print('=' * 90)
print('D4 / C3-11 — HISTORICAL PRODUCTION IF SCORE COMPATIBILITY')
print('=' * 90)

for dataset, cfg in CONFIG.items():

    print('\n' + '=' * 90)
    print('DATASET:', dataset)
    print('=' * 90)

    for name in ['model', 'scores']:
        if not cfg[name].is_file():
            raise RuntimeError(f'STOP: Missing {dataset} {name}')

    # Reconfirm the exact serialized artifact before loading.
    actual_hash = file_sha256(cfg['model'])

    if actual_hash != cfg['sha256']:
        raise RuntimeError(f'STOP: Model SHA256 mismatch for {dataset}')

    model = joblib.load(cfg['model'])

    if not isinstance(model, IsolationForest):
        raise RuntimeError(f'STOP: Unexpected model class for {dataset}')

    if (
        len(model.estimators_) != 300
        or model.n_features_in_ != 23
        or model.max_samples_ != 256
    ):
        raise RuntimeError(
            f'STOP: Recovered model specification mismatch for {dataset}'
        )

    print('MODEL INTEGRITY: PASS')
    print('Model class:', type(model).__name__)
    print('Trees:', len(model.estimators_))
    print('Features:', model.n_features_in_)
    print('max_samples_:', model.max_samples_)

    sample, groups, source_rows = build_deterministic_sample(
        cfg['scores'], cfg['channel']
    )

    expected_group_count = cfg['expected_channels'] * 3

    actual_splits = set(sample['split'].unique())
    actual_channels = sample[cfg['channel']].nunique()

    if actual_splits != EXPECTED_SPLITS:
        raise RuntimeError(
            f'STOP: Unexpected split labels for {dataset}: {actual_splits}'
        )

    if actual_channels != cfg['expected_channels']:
        raise RuntimeError(
            f'STOP: Channel coverage mismatch for {dataset}'
        )

    if len(groups) != expected_group_count:
        raise RuntimeError(
            f'STOP: Expected {expected_group_count} channel/split groups '
            f'but found {len(groups)} for {dataset}'
        )

    print('\nDETERMINISTIC SCORING SAMPLE')
    print('Historical CSV rows streamed:', f'{source_rows:,}')
    print('Channels:', actual_channels)
    print('Chronological splits:', sorted(actual_splits))
    print('Channel/split groups:', len(groups))
    print('Unique sampled rows:', len(sample))

    # Preserve the historical canonical feature ordering and
    # the float32 representation required by the recovered model.
    X = sample[SCALED].astype(np.float32)

    if not np.isfinite(X.to_numpy()).all():
        raise RuntimeError(f'STOP: Nonfinite model inputs in {dataset}')

    saved_score = sample['anomaly_score'].to_numpy(dtype=np.float64)
    saved_decision = sample['if_decision_function'].to_numpy(
        dtype=np.float64
    )

    if not (
        np.isfinite(saved_score).all()
        and np.isfinite(saved_decision).all()
    ):
        raise RuntimeError(f'STOP: Nonfinite historical scores in {dataset}')

    recovered_decision = np.asarray(
        model.decision_function(X), dtype=np.float64
    )

    recovered_score = -recovered_decision

    historical_identity_error = float(
        np.max(np.abs(saved_score + saved_decision))
    )

    decision_error = np.abs(
        recovered_decision - saved_decision
    )

    score_error = np.abs(
        recovered_score - saved_score
    )

    max_decision_error = float(np.max(decision_error))
    max_score_error = float(np.max(score_error))

    print('\nHISTORICAL SCORING IDENTITY')
    print('Target: anomaly_score = -decision_function(X)')
    print('Saved score identity error:',
          f'{historical_identity_error:.15g}')
    print('Recomputed decision-function maximum error:',
          f'{max_decision_error:.15g}')
    print('Recomputed anomaly-score maximum error:',
          f'{max_score_error:.15g}')

    print('\nSPLIT-LEVEL SCORE REPRODUCTION')

    for split in ['train', 'val', 'test']:
        mask = sample['split'].eq(split).to_numpy()

        print(
            f'{split:5s} | '
            f'sampled={int(mask.sum()):4d} | '
            f'max_abs_score_error={float(np.max(score_error[mask])):.15g}'
        )

    # The main historical score must be reproduced on
    # this independent, stratified verification sample.
    passed = (
        historical_identity_error <= TOLERANCE
        and max_decision_error <= TOLERANCE
        and max_score_error <= TOLERANCE
    )

    print('\nSCORING COMPATIBILITY:',
          'PASS' if passed else 'REVIEW')

    if not passed:
        worst = int(np.argmax(score_error))

        print('Largest-error source row:',
              int(sample.iloc[worst]['__source_row__']))
        print('Saved score:', repr(float(saved_score[worst])))
        print('Recovered score:', repr(float(recovered_score[worst])))

        raise RuntimeError(
            f'STOP: Historical scoring mismatch in {dataset}. '
            'Do not proceed to TreeSHAP.'
        )

    print('DATASET STATUS: PASS')

    del sample, X, model

print('\n' + '=' * 90)
print('CELL 2 COMPLETE — BOTH PRODUCTION MODEL SCORERS VERIFIED')
print('All channels and chronological splits represented.')
print('Canonical 23-feature float32 inputs preserved.')
print('No model fitting or model modification.')
print('No TreeSHAP or Kernel SHAP computation.')
print('No source files changed and no files written.')
print('D4 / C3-11 remains IN PROGRESS.')
print('=' * 90)


In [ ]:

# ============================================================
# D4 / C3-11 — CELL 3
# TREE SHAP OUTPUT SEMANTICS AND ADDITIVITY PILOT
#
# Read-only pilot on selected validation/test windows
# No model fitting, no file writes, no global analysis yet
# ============================================================

from pathlib import Path
import hashlib
import joblib
import shap
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

BASE = Path('/content/drive/MyDrive/Master/Thesis/Code')
RECOVERY = BASE / 'SUPERVISOR_FINAL_EXPERIMENTS_2026/03_integrity_checks/IF_RECOVERY'
SCORES = BASE / 'experiments/EXP_SPLIT_BOUNDARY_PURGE_CORRECTION'

CONFIG = {
    'UK-DALE': {
        'model': RECOVERY / 'UKDALE/ukdale_full_train_IF_FUNCTIONALLY_VERIFIED_20260928T180619_923276Z.joblib',
        'scores': SCORES / 'UKDALE/ukdale_if_scored_boundary_corrected.csv',
        'sha256': 'c1695e8c4265f6e90994be88ea7ec645acba8e10ed6240697aa73703ce3c9c5b',
        'key': 'series_key',
        'channels': [
            'building1_meter12_fridge_freezer',
            'building5_meter18_kettle'
        ]
    },
    'AMPds2': {
        'model': RECOVERY / 'AMPDS2/ampds2_full_train_IF_FUNCTIONALLY_VERIFIED_20260928T181300_250116Z.joblib',
        'scores': SCORES / 'AMPDS2/ampds2_if_scored_boundary_corrected.csv',
        'sha256': '890a47ce9a2b30902010c78c0084b643c2f7b6c41257c9655d6be6cb03e2d7ca',
        'key': 'channel_code',
        'channels': ['EQE', 'B2E']
    }
}

FEATURES = [
    'energy', 'first_last_diff', 'iqr', 'kurtosis', 'mad',
    'max', 'max_abs_diff', 'mean', 'mean_abs_diff',
    'median', 'min', 'n_points', 'peak_count',
    'q25', 'q75', 'range', 'rms', 'skew',
    'slope', 'std', 'std_diff', 'sum', 'var'
]

SCALED = [f'{feature}_z' for feature in FEATURES]

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def isolation_normalizer(n):
    """
    Independent reproduction of sklearn's average path length
    normalization for the fitted maximum sample size.
    """
    n = int(n)
    if n <= 1:
        return 0.0
    if n == 2:
        return 1.0
    return float(
        2.0 * (np.log(n - 1.0) + np.euler_gamma)
        - 2.0 * (n - 1.0) / n
    )

def select_pilot_rows(path, key, channels):
    """
    For each selected channel and each val/test split:
      - first chronological stored row
      - highest saved anomaly-score row

    Deterministic diagnostic selection, not final XAI sampling.
    """
    targets = {
        (str(channel), split)
        for channel in channels
        for split in ('val', 'test')
    }

    required = [key, 'split', 'anomaly_score'] + SCALED

    header = pd.read_csv(path, nrows=0).columns.tolist()
    missing = [c for c in required if c not in header]

    if missing:
        raise RuntimeError(f'STOP: Missing columns: {missing}')

    first = {}
    highest = {}
    row_offset = 0

    for chunk in pd.read_csv(
        path,
        usecols=required,
        chunksize=75000,
        float_precision='round_trip',
        low_memory=False
    ):
        chunk['__source_row__'] = np.arange(
            row_offset,
            row_offset + len(chunk),
            dtype=np.int64
        )
        row_offset += len(chunk)

        for (channel, split), block in chunk.groupby(
            [key, 'split'], sort=False
        ):
            target = (str(channel), str(split))

            if target not in targets:
                continue

            if target not in first:
                first[target] = block.iloc[[0]].copy()


            top = block.loc[[block['anomaly_score'].idxmax()]].copy()


            if (
                target not in highest
                or float(top['anomaly_score'].iloc[0])
                > float(highest[target]['anomaly_score'].iloc[0])
            ):
                highest[target] = top

    if set(first) != targets or set(highest) != targets:
        raise RuntimeError(
            f'STOP: Incomplete pilot group coverage. '
            f'Found {sorted(first)}'
        )

    frames = []
    for target in sorted(targets):
        frames.append(first[target])
        frames.append(highest[target])

    sample = pd.concat(frames, ignore_index=True)
    sample = sample.drop_duplicates('__source_row__')
    sample = sample.sort_values('__source_row__').reset_index(drop=True)

    return sample, row_offset, len(targets)

def independent_average_path_length(model, X):
    """
    Compute the fitted forest's mean adjusted isolation path
    directly from its stored trees and path-length metadata.
    """
    required_attributes = [
        '_decision_path_lengths',
        '_average_path_length_per_tree',
        'estimators_features_'
    ]

    missing = [
        attribute for attribute in required_attributes
        if not hasattr(model, attribute)
    ]

    if missing:
        raise RuntimeError(
            f'STOP: Required fitted path metadata missing: {missing}'
        )

    result = np.zeros(X.shape[0], dtype=np.float64)

    for i, tree in enumerate(model.estimators_):
        feature_indices = model.estimators_features_[i]
        X_tree = X[:, feature_indices]

        leaves = tree.apply(X_tree)

        result += (
            model._decision_path_lengths[i][leaves]
            + model._average_path_length_per_tree[i][leaves]
            - 1.0
        )

    return result / len(model.estimators_)

print('=' * 94)
print('D4 / C3-11 — TREE SHAP PILOT')
print('Native path-length output versus historical anomaly score')
print('=' * 94)
print('SHAP version:', shap.__version__)
print('No background supplied: tree_path_dependent explanations')
print('Model output: raw')

for dataset, cfg in CONFIG.items():

    print('\n' + '=' * 94)
    print('DATASET:', dataset)
    print('=' * 94)

    for name in ['model', 'scores']:
        if not cfg[name].is_file():
            raise RuntimeError(f'STOP: Missing {name}: {cfg[name]}')

    if sha256_file(cfg['model']) != cfg['sha256']:
        raise RuntimeError(f'STOP: {dataset} model SHA256 mismatch')

    model = joblib.load(cfg['model'])

    if not isinstance(model, IsolationForest):
        raise RuntimeError(f'STOP: Unexpected model class: {dataset}')

    if (
        len(model.estimators_) != 300
        or model.n_features_in_ != 23
        or model.max_samples_ != 256
    ):
        raise RuntimeError(f'STOP: Unexpected fitted model specification: {dataset}')

    sample, streamed_rows, group_count = select_pilot_rows(
        cfg['scores'],
        cfg['key'],
        cfg['channels']
    )

    X = sample[SCALED].to_numpy(dtype=np.float32)

    if not np.isfinite(X).all():
        raise RuntimeError(f'STOP: Nonfinite pilot inputs: {dataset}')

    historical_scores = sample['anomaly_score'].to_numpy(
        dtype=np.float64
    )

    if not np.isfinite(historical_scores).all():
        raise RuntimeError(f'STOP: Nonfinite saved scores: {dataset}')

    original_scores = -model.decision_function(X)

    scoring_error = float(
        np.max(np.abs(original_scores - historical_scores))
    )

    print('\n1. PILOT POPULATION')
    print('Historical rows streamed:', f'{streamed_rows:,}')
    print('Selected channel/split groups:', group_count)
    print('Unique pilot windows:', len(sample))
    print('Selected channels:', cfg['channels'])
    print('Historical scoring max error:', f'{scoring_error:.15g}')

    if scoring_error > 1e-10:
        raise RuntimeError(f'STOP: Historical score mismatch: {dataset}')

    # --------------------------------------------------------
    # 2. Compute TreeSHAP in its native RAW tree output space.
    # --------------------------------------------------------

    print('\n2. TREE SHAP COMPUTATION')

    explainer = shap.TreeExplainer(
        model,
        data=None,
        model_output='raw',
        feature_perturbation='tree_path_dependent'
    )

    shap_values = np.asarray(
        explainer.shap_values(
            X,
            check_additivity=False
        ),
        dtype=np.float64
    )

    expected = np.asarray(
        explainer.expected_value, dtype=np.float64
    ).reshape(-1)

    if shap_values.shape != X.shape or expected.size != 1:
        raise RuntimeError(
            f'STOP: Unexpected SHAP output shape: '
            f'{shap_values.shape}, expected={expected.shape}'
        )

    if not np.isfinite(shap_values).all():
        raise RuntimeError('STOP: Nonfinite TreeSHAP values')

    baseline = float(expected[0])

    # TreeSHAP additive reconstruction of its raw output.
    shap_path_length = baseline + shap_values.sum(axis=1)

    # --------------------------------------------------------
    # 3. Independently reproduce adjusted isolation path.
    # --------------------------------------------------------

    independent_path_length = independent_average_path_length(
        model, X
    )

    path_error = float(np.max(np.abs(
        shap_path_length - independent_path_length
    )))

    print('SHAP baseline (path-length scale):', repr(baseline))
    print('SHAP values shape:', shap_values.shape)
    print('Maximum path-length additivity error:',
          f'{path_error:.15g}')

    if path_error > 1e-7:
        raise RuntimeError(
            f'STOP: TreeSHAP path-length additivity failed: {dataset}'
        )

    # --------------------------------------------------------
    # 4. Transform raw path length to historical score.
    #
    # IF raw outlier score = 2 ** (-mean_path / c(max_samples))
    # historical anomaly_score = outlier_score + offset_
    # --------------------------------------------------------

    normalizer = isolation_normalizer(model.max_samples_)

    if normalizer <= 0:
        raise RuntimeError('STOP: Invalid isolation normalizer')

    transformed_scores = (
        np.power(2.0, -shap_path_length / normalizer)
        + float(model.offset_)
    )

    transformed_error = float(np.max(np.abs(
        transformed_scores - historical_scores
    )))

    print('\n3. HISTORICAL ANOMALY-SCORE RECONSTRUCTION')
    print('Path-length normalizer:', repr(normalizer))
    print('Model offset_:', repr(float(model.offset_)))
    print('Maximum transformed-score error:',
          f'{transformed_error:.15g}')

    if transformed_error > 1e-9:
        raise RuntimeError(
            f'STOP: TreeSHAP-to-score transformation failed: {dataset}'
        )

    # --------------------------------------------------------
    # 5. Report output orientation.
    # --------------------------------------------------------

    print('\n4. PATH LENGTH VERSUS ANOMALY SCORE')

    display = pd.DataFrame({
        'channel': sample[cfg['key']].astype(str).values,
        'split': sample['split'].astype(str).values,
        'source_row': sample['__source_row__'].values,
        'mean_path_length': independent_path_length,
        'shap_reconstructed_path': shap_path_length,
        'historical_anomaly_score': historical_scores,
        'transformed_anomaly_score': transformed_scores
    })

    print(
        display.to_string(
            index=False,
            float_format=lambda value: f'{value:.9f}'
        )
    )

    print('\nINTERPRETATION CHECK')
    print('Longer isolation path: more normal according to the forest.')
    print('Shorter isolation path: more anomalous according to the forest.')
    print('Positive TreeSHAP value: increases path length.')
    print('Negative TreeSHAP value: decreases path length.')
    print('Path-length attribution is NOT numerically an additive anomaly-score attribution.')

    print('\nDATASET PILOT: PASS')
    print('Historical scoring: PASS')
    print('Native TreeSHAP additivity: PASS')
    print('Path-to-score transformation: PASS')

    del model, sample, X, explainer, shap_values

print('\n' + '=' * 94)
print('CELL 3 COMPLETE — BOTH DATASETS PASSED')
print('The explained native output has been independently verified.')
print('Historical -decision_function(X) reproduced through path transformation.')
print('No global SHAP conclusions generated yet.')
print('No model trained or modified.')
print('No historical files changed or new files created.')
print('D4 / C3-11 remains IN PROGRESS.')
print('=' * 94)


In [ ]:

# ============================================================
# D4 / C3-11 — CELL 4
# GLOBAL TREE SHAP ANALYSIS ON RESIDENTIAL TEST WINDOWS
#
# Population: frozen, boundary-corrected test partitions
# Sampling: deterministic, stratified by channel and model flag
# Importance: population-weighted mean absolute path SHAP
# Flagged analysis: contributions shortening isolation paths
#
# READ ONLY — No fitting, threshold changes, or file writes
# ============================================================

from pathlib import Path
import hashlib
import joblib
import shap
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

BASE = Path('/content/drive/MyDrive/Master/Thesis/Code')
RECOVERY = BASE / 'SUPERVISOR_FINAL_EXPERIMENTS_2026/03_integrity_checks/IF_RECOVERY'
SCORES = BASE / 'experiments/EXP_SPLIT_BOUNDARY_PURGE_CORRECTION'

CONFIG = {
    'UK-DALE': {
        'model': RECOVERY / 'UKDALE/ukdale_full_train_IF_FUNCTIONALLY_VERIFIED_20260928T180619_923276Z.joblib',
        'scores': SCORES / 'UKDALE/ukdale_if_scored_boundary_corrected.csv',
        'sha256': 'c1695e8c4265f6e90994be88ea7ec645acba8e10ed6240697aa73703ce3c9c5b',
        'key': 'series_key',
        'channels': 16,
        'test_windows': 72780,
        'test_flags': 3551,
        'threshold': 0.058252700304033553
    },
    'AMPds2': {
        'model': RECOVERY / 'AMPDS2/ampds2_full_train_IF_FUNCTIONALLY_VERIFIED_20260928T181300_250116Z.joblib',
        'scores': SCORES / 'AMPDS2/ampds2_if_scored_boundary_corrected.csv',
        'sha256': '890a47ce9a2b30902010c78c0084b643c2f7b6c41257c9655d6be6cb03e2d7ca',
        'key': 'channel_code',
        'channels': 21,
        'test_windows': 147147,
        'test_flags': 14560,
        'threshold': 0.029990580900317053
    }
}

FEATURES = [
    'energy', 'first_last_diff', 'iqr', 'kurtosis', 'mad',
    'max', 'max_abs_diff', 'mean', 'mean_abs_diff',
    'median', 'min', 'n_points', 'peak_count',
    'q25', 'q75', 'range', 'rms', 'skew',
    'slope', 'std', 'std_diff', 'sum', 'var'
]

SCALED = [f'{name}_z' for name in FEATURES]

FEATURE_GROUPS = {
    'Consumption level': [
        'energy', 'max', 'mean', 'median', 'min',
        'q25', 'q75', 'rms', 'sum'
    ],
    'Dispersion': [
        'iqr', 'mad', 'range', 'std', 'var'
    ],
    'Successive change': [
        'first_last_diff', 'max_abs_diff',
        'mean_abs_diff', 'std_diff'
    ],
    'Trend': ['slope'],
    'Distributional shape': ['kurtosis', 'skew'],
    'Sampling and peaks': ['n_points', 'peak_count']
}

assert sorted(
    feature for group in FEATURE_GROUPS.values() for feature in group
) == sorted(FEATURES), 'STOP: Feature groups are incomplete or duplicated.'

SEED = 2026
MAX_PER_CHANNEL_FLAG_STRATUM = 48
TOL_SCORE = 1e-10
TOL_TRANSFORM = 1e-9

# Results remain in notebook memory for subsequent verification.
D4_GLOBAL_RESULTS = {}

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def stable_seed(dataset, channel, flag):
    text = f'{SEED}|{dataset}|{channel}|{int(flag)}'
    digest = hashlib.sha256(text.encode('utf-8')).digest()
    return int.from_bytes(digest[:8], byteorder='little')

def isolation_normalizer(n):
    n = int(n)
    if n <= 1:
        return 0.0
    if n == 2:
        return 1.0
    return float(
        2.0 * (np.log(n - 1.0) + np.euler_gamma)
        - 2.0 * (n - 1.0) / n
    )

def read_historical_test_rows(path, key):
    """
    Read only necessary columns, in chunks.
    Retain test windows and their original CSV row positions.
    """
    required = [
        key, 'split', 'anomaly_score',
        'predicted_anomaly_corrected'
    ] + SCALED

    columns = pd.read_csv(path, nrows=0).columns.tolist()
    missing = [c for c in required if c not in columns]

    if missing:
        raise RuntimeError(f'STOP: Missing columns: {missing}')

    parts = []
    row_offset = 0

    for chunk in pd.read_csv(
        path,
        usecols=required,
        chunksize=100000,
        float_precision='round_trip',
        low_memory=False
    ):
        chunk['__csv_row__'] = np.arange(
            row_offset,
            row_offset + len(chunk),
            dtype=np.int64
        )
        row_offset += len(chunk)

        test_chunk = chunk.loc[chunk['split'].eq('test')]

        if len(test_chunk):
            parts.append(test_chunk.copy())

    if not parts:
        raise RuntimeError('STOP: No test rows were found.')

    test = pd.concat(parts, ignore_index=True)

    return test, row_offset

def stratified_sample(test, dataset, key):
    """
    Draw up to 48 windows from every channel x flag stratum.

    This deliberately includes both flagged and unflagged
    test windows. Inverse sampling-fraction weights recover
    each stratum's relative importance in the test population.
    """
    selected_indices = []
    stratum_weights = {}
    strata_summary = []

    for (channel, flag), group in test.groupby(
        [key, 'frozen_flag'], sort=True
    ):
        channel = str(channel)
        flag = int(flag)

        population_n = len(group)
        sample_n = min(
            MAX_PER_CHANNEL_FLAG_STRATUM,
            population_n
        )

        rng = np.random.default_rng(
            stable_seed(dataset, channel, flag)
        )

        chosen = rng.choice(
            group.index.to_numpy(),
            size=sample_n,
            replace=False
        )

        selected_indices.extend(chosen.tolist())

        weight = population_n / sample_n
        stratum_weights[(channel, flag)] = weight

        strata_summary.append({
            'channel': channel,
            'flag': flag,
            'population_n': population_n,
            'sample_n': sample_n,
            'design_weight': weight
        })

    sample = (
        test.loc[selected_indices]
        .copy()
        .sort_values('__csv_row__')
        .reset_index(drop=True)
    )

    sample['design_weight'] = [
        stratum_weights[(str(channel), int(flag))]
        for channel, flag in zip(
            sample[key], sample['frozen_flag']
        )
    ]

    return sample, pd.DataFrame(strata_summary)

def weighted_feature_mean(values, weights):
    return np.average(
        values,
        axis=0,
        weights=weights
    )

def feature_group_summary(feature_table):
    rows = []

    for group_name, feature_names in FEATURE_GROUPS.items():
        selected = feature_table.set_index('feature').loc[
            feature_names
        ]

        rows.append({
            'feature_group': group_name,
            'all_test_abs': selected['all_test_abs'].sum(),
            'flagged_abs': selected['flagged_abs'].sum(),
            'flagged_shortening': selected['flagged_shortening'].sum()
        })

    result = pd.DataFrame(rows)

    for column in [
        'all_test_abs', 'flagged_abs', 'flagged_shortening'
    ]:
        total = result[column].sum()
        result[column + '_share_pct'] = (
            100.0 * result[column] / total
            if total > 0 else np.nan
        )

    return result

print('=' * 100)
print('D4 / C3-11 — GLOBAL TREE SHAP ON HELD-OUT RESIDENTIAL WINDOWS')
print('=' * 100)
print('SHAP version:', shap.__version__)
print('Sampling seed:', SEED)
print('Maximum observations per channel/flag stratum:',
      MAX_PER_CHANNEL_FLAG_STRATUM)
print('Attribution units: native isolation path length')
print('Importance estimates are weighted to test-population strata.')

for dataset, cfg in CONFIG.items():

    print('\n' + '=' * 100)
    print('DATASET:', dataset)
    print('=' * 100)

    if not cfg['model'].is_file() or not cfg['scores'].is_file():
        raise RuntimeError(f'STOP: Missing historical artifact for {dataset}')

    if sha256_file(cfg['model']) != cfg['sha256']:
        raise RuntimeError(f'STOP: Model integrity failure for {dataset}')

    model = joblib.load(cfg['model'])

    if not isinstance(model, IsolationForest):
        raise RuntimeError(f'STOP: Incorrect model class for {dataset}')

    if (
        len(model.estimators_) != 300
        or model.n_features_in_ != 23
        or model.max_samples_ != 256
    ):
        raise RuntimeError(f'STOP: Model specification mismatch for {dataset}')

    test, total_rows = read_historical_test_rows(
        cfg['scores'], cfg['key']
    )

    test['anomaly_score'] = pd.to_numeric(
        test['anomaly_score'], errors='coerce'
    )

    test['frozen_flag'] = pd.to_numeric(
        test['predicted_anomaly_corrected'],
        errors='coerce'
    )

    if len(test) != cfg['test_windows']:
        raise RuntimeError(f'STOP: Test population mismatch for {dataset}')

    if test[cfg['key']].nunique() != cfg['channels']:
        raise RuntimeError(f'STOP: Channel count mismatch for {dataset}')

    if not np.isfinite(test['anomaly_score'].to_numpy()).all():
        raise RuntimeError(f'STOP: Invalid saved scores in {dataset}')

    if not test['frozen_flag'].isin([0, 1]).all():
        raise RuntimeError(f'STOP: Invalid historical flags in {dataset}')

    test['frozen_flag'] = test['frozen_flag'].astype(np.int8)

    if int(test['frozen_flag'].sum()) != cfg['test_flags']:
        raise RuntimeError(f'STOP: Historical flagged count mismatch: {dataset}')

    recomputed_flags = (
        test['anomaly_score'].to_numpy() > cfg['threshold']
    ).astype(np.int8)

    mismatches = np.count_nonzero(
        recomputed_flags != test['frozen_flag'].to_numpy()
    )

    if mismatches:
        raise RuntimeError(f'STOP: Historical flag mismatch: {dataset}')

    # Ensure the expected flagged and unflagged strata exist
    # for every retained channel.
    stratum_count = test.groupby(
        [cfg['key'], 'frozen_flag']
    ).ngroups

    if stratum_count != 2 * cfg['channels']:
        raise RuntimeError(
            f'STOP: Unexpected channel/flag stratum count: {dataset}'
        )

    sample, strata = stratified_sample(
        test, dataset, cfg['key']
    )

    weights = sample['design_weight'].to_numpy(
        dtype=np.float64
    )

    flags = sample['frozen_flag'].to_numpy(
        dtype=np.int8
    )

    X = sample[SCALED].to_numpy(dtype=np.float32)

    if not np.isfinite(X).all() or not np.isfinite(weights).all():
        raise RuntimeError(f'STOP: Nonfinite sample inputs: {dataset}')

    # This equality is exact in principle because every
    # stratum receives its inverse sampling-fraction weight.
    reconstructed_flag_rate = np.average(
        flags, weights=weights
    )

    original_flag_rate = cfg['test_flags'] / cfg['test_windows']

    if not np.isclose(
        reconstructed_flag_rate,
        original_flag_rate,
        atol=1e-12,
        rtol=0
    ):
        raise RuntimeError(
            f'STOP: Design weights do not reproduce test flag rate: {dataset}'
        )

    print('\n1. POPULATION AND SAMPLING')
    print('Historical CSV rows streamed:', f'{total_rows:,}')
    print('Frozen test windows:', f"{len(test):,}")
    print('Test flagged windows:', f"{int(flags.sum())} sampled / {cfg['test_flags']:,} population")
    print('Channels:', cfg['channels'])
    print('Channel/flag strata:', stratum_count)
    print('Sampled test windows:', len(sample))
    print('Weighted population flagged rate:',
          f'{100 * reconstructed_flag_rate:.6f}%')
    print('Historical population flagged rate:',
          f'{100 * original_flag_rate:.6f}%')

    # Verify the recovered historical model on the entire
    # explanation sample before computing attributions.
    saved_scores = sample['anomaly_score'].to_numpy(
        dtype=np.float64
    )

    recovered_scores = -model.decision_function(X)

    score_error = float(np.max(np.abs(
        recovered_scores - saved_scores
    )))

    print('\n2. HISTORICAL SCORE REPRODUCTION')
    print('Maximum score error:', f'{score_error:.15g}')

    if score_error > TOL_SCORE:
        raise RuntimeError(
            f'STOP: Model scoring reproduction failed for {dataset}'
        )

    # --------------------------------------------------------
    # TreeSHAP — native path-length output
    # --------------------------------------------------------

    print('\n3. COMPUTING GLOBAL TREE SHAP')

    explainer = shap.TreeExplainer(
        model,
        data=None,
        model_output='raw',
        feature_perturbation='tree_path_dependent'
    )

    phi = np.asarray(
        explainer.shap_values(
            X,
            check_additivity=False
        ),
        dtype=np.float64
    )

    expected = np.asarray(
        explainer.expected_value,
        dtype=np.float64
    ).reshape(-1)

    if phi.shape != X.shape or expected.size != 1:
        raise RuntimeError(
            f'STOP: Unexpected explanation dimensions for {dataset}'
        )

    if not np.isfinite(phi).all():
        raise RuntimeError(f'STOP: Nonfinite SHAP values for {dataset}')

    baseline = float(expected[0])
    reconstructed_path = baseline + phi.sum(axis=1)

    normalizer = isolation_normalizer(model.max_samples_)

    transformed_scores = (
        np.exp2(-reconstructed_path / normalizer)
        + float(model.offset_)
    )

    transformation_error = float(np.max(np.abs(
        transformed_scores - saved_scores
    )))

    print('TreeSHAP baseline:', repr(baseline))
    print('SHAP matrix dimensions:', phi.shape)
    print('Maximum path-to-score error:',
          f'{transformation_error:.15g}')

    if transformation_error > TOL_TRANSFORM:
        raise RuntimeError(
            f'STOP: Additive explanation transformation failed: {dataset}'
        )

    # --------------------------------------------------------
    # Population-weighted global importance
    # --------------------------------------------------------

    all_mean_abs = weighted_feature_mean(
        np.abs(phi),
        weights
    )

    flagged_mask = flags == 1

    flagged_mean_abs = weighted_feature_mean(
        np.abs(phi[flagged_mask]),
        weights[flagged_mask]
    )

    # Negative path-length contributions move model output
    # toward shorter isolation paths and higher anomaly scores.
    flagged_shortening = weighted_feature_mean(
        np.maximum(-phi[flagged_mask], 0.0),
        weights[flagged_mask]
    )

    feature_table = pd.DataFrame({
        'feature': FEATURES,
        'all_test_abs': all_mean_abs,
        'flagged_abs': flagged_mean_abs,
        'flagged_shortening': flagged_shortening
    })

    group_table = feature_group_summary(feature_table)

    print('\n4. GLOBAL FEATURE IMPORTANCE — ALL TEST WINDOWS')
    print('Population-weighted mean absolute path-length SHAP')
    print(
        feature_table.sort_values(
            'all_test_abs', ascending=False
        )[['feature', 'all_test_abs']]
        .head(10)
        .to_string(index=False, float_format=lambda v: f'{v:.6f}')
    )

    print('\n5. FEATURE CONTRIBUTIONS — FLAGGED TEST WINDOWS')
    print('Population-weighted mean contribution shortening the path')
    print(
        feature_table.sort_values(
            'flagged_shortening', ascending=False
        )[['feature', 'flagged_abs', 'flagged_shortening']]
        .head(10)
        .to_string(index=False, float_format=lambda v: f'{v:.6f}')
    )

    print('\n6. ENGINEERED FEATURE GROUPS')
    print('Shares are based on summed absolute feature contributions.')
    print('They are NOT causal effects or unique physical contributions.')
    print(
        group_table[
            [
                'feature_group',
                'all_test_abs_share_pct',
                'flagged_shortening_share_pct'
            ]
        ].sort_values(
            'flagged_shortening_share_pct',
            ascending=False
        ).to_string(index=False, float_format=lambda v: f'{v:.3f}')
    )

    # Keep verified results in notebook memory only.
    D4_GLOBAL_RESULTS[dataset] = {
        'model_sha256': cfg['sha256'],
        'sample_seed': SEED,
        'max_per_stratum': MAX_PER_CHANNEL_FLAG_STRATUM,
        'historical_test_windows': len(test),
        'historical_test_flags': cfg['test_flags'],
        'sample_metadata': sample[
            [cfg['key'], 'split', '__csv_row__',
             'anomaly_score', 'frozen_flag', 'design_weight']
        ].copy(),
        'strata': strata.copy(),
        'feature_importance': feature_table.copy(),
        'feature_groups': group_table.copy(),
        'shap_values': phi.copy(),
        'baseline_path_length': baseline,
        'maximum_score_reproduction_error': score_error,
        'maximum_transformation_error': transformation_error
    }

    print('\nDATASET STATUS: PASS')
    print('Global estimates computed in native path-length units.')
    print('Original model and historical scores unchanged.')

    del test, sample, X, model, explainer, phi

print('\n' + '=' * 100)
print('CELL 4 COMPLETE — BOTH DATASETS VERIFIED')
print('Results retained only in D4_GLOBAL_RESULTS (notebook memory).')
print('Sample design: deterministic and channel/flag stratified.')
print('Population weighting applied to global importance estimates.')
print('Native path-length additivity checked through score reconstruction.')
print('No model training or retraining.')
print('No historical scores, labels, thresholds, or predictions changed.')
print('No files created or modified.')
print('D4 / C3-11 remains IN PROGRESS pending interpretation and reporting.')
print('=' * 100)


In [ ]:

# ============================================================
# D4 / C3-11 — CELL 5
# GLOBAL TREE SHAP AUDIT AND AMPds2 EQE DECOMPOSITION
#
# Uses existing D4_GLOBAL_RESULTS from Cell 4.
# No model loading, training, SHAP recomputation or file writes.
# ============================================================

import numpy as np
import pandas as pd

FEATURES = [
    'energy', 'first_last_diff', 'iqr', 'kurtosis', 'mad',
    'max', 'max_abs_diff', 'mean', 'mean_abs_diff',
    'median', 'min', 'n_points', 'peak_count',
    'q25', 'q75', 'range', 'rms', 'skew',
    'slope', 'std', 'std_diff', 'sum', 'var'
]

FEATURE_GROUPS = {
    'Consumption level': [
        'energy', 'max', 'mean', 'median', 'min',
        'q25', 'q75', 'rms', 'sum'
    ],
    'Dispersion': ['iqr', 'mad', 'range', 'std', 'var'],
    'Successive change': [
        'first_last_diff', 'max_abs_diff',
        'mean_abs_diff', 'std_diff'
    ],
    'Trend': ['slope'],
    'Distributional shape': ['kurtosis', 'skew'],
    'Sampling and peaks': ['n_points', 'peak_count']
}

EXPECTED = {
    'UK-DALE': {
        'windows': 72780,
        'flagged': 3551,
        'channels': 16,
        'key': 'series_key'
    },
    'AMPds2': {
        'windows': 147147,
        'flagged': 14560,
        'channels': 21,
        'key': 'channel_code'
    }
}

if 'D4_GLOBAL_RESULTS' not in globals():
    raise RuntimeError(
        'STOP: Cell 4 in-memory results are unavailable. '
        'Rerun Cell 4 before executing this cell.'
    )

if set(D4_GLOBAL_RESULTS) != set(EXPECTED):
    raise RuntimeError(
        'STOP: Expected both UK-DALE and AMPds2 results.'
    )

assert sorted(
    f for names in FEATURE_GROUPS.values() for f in names
) == sorted(FEATURES), 'STOP: Incomplete or duplicated feature groups.'

def weighted_mean(array, weights):
    return np.average(array, axis=0, weights=weights)

def group_summary(feature_contributions):
    """
    Aggregate already-computed feature contributions.

    Group shares use summed magnitudes.
    Per-feature group means account descriptively for
    unequal numbers of engineered features per group.
    """
    total = float(np.sum(feature_contributions))

    if not np.isfinite(total) or total <= 0:
        raise RuntimeError('STOP: Invalid attribution total.')

    rows = []

    for group, names in FEATURE_GROUPS.items():
        indices = [FEATURES.index(name) for name in names]
        magnitude = float(
            np.sum(feature_contributions[indices])
        )

        rows.append({
            'feature_group': group,
            'n_features': len(names),
            'sum_path_shortening': magnitude,
            'share_pct': 100.0 * magnitude / total,
            'mean_per_feature': magnitude / len(names)
        })

    return pd.DataFrame(rows).sort_values(
        'share_pct', ascending=False
    ).reset_index(drop=True)

def inspect_result(dataset, rec, expected):
    meta = rec['sample_metadata'].copy()
    strata = rec['strata'].copy()
    phi = np.asarray(rec['shap_values'], dtype=np.float64)
    table = rec['feature_importance'].copy()
    group_table = rec['feature_groups'].copy()

    key = expected['key']

    if phi.ndim != 2 or phi.shape != (len(meta), len(FEATURES)):
        raise RuntimeError(
            f'STOP: SHAP/sample dimension mismatch in {dataset}'
        )

    if table['feature'].tolist() != FEATURES:
        raise RuntimeError(
            f'STOP: Feature order mismatch in {dataset}'
        )

    if meta['__csv_row__'].duplicated().any():
        raise RuntimeError(
            f'STOP: Duplicate sampled source row in {dataset}'
        )

    if not meta['__csv_row__'].is_monotonic_increasing:
        raise RuntimeError(
            f'STOP: Sample order mismatch in {dataset}'
        )

    if not meta['split'].eq('test').all():
        raise RuntimeError(
            f'STOP: Non-test rows in global explanation sample: {dataset}'
        )

    weights = meta['design_weight'].to_numpy(dtype=np.float64)
    flags = meta['frozen_flag'].to_numpy(dtype=np.int8)

    if (
        not np.isfinite(phi).all()
        or not np.isfinite(weights).all()
        or np.any(weights <= 0)
        or not np.isin(flags, [0, 1]).all()
    ):
        raise RuntimeError(
            f'STOP: Invalid explanation values or weights in {dataset}'
        )

    if meta[key].nunique() != expected['channels']:
        raise RuntimeError(
            f'STOP: Channel count mismatch in {dataset}'
        )

    if not np.isclose(
        weights.sum(),
        expected['windows'],
        rtol=0, atol=1e-7
    ):
        raise RuntimeError(
            f'STOP: Total population weights mismatch in {dataset}'
        )

    if not np.isclose(
        weights[flags == 1].sum(),
        expected['flagged'],
        rtol=0, atol=1e-7
    ):
        raise RuntimeError(
            f'STOP: Flagged population weights mismatch in {dataset}'
        )

    if int(strata['population_n'].sum()) != expected['windows']:
        raise RuntimeError(
            f'STOP: Stratum population counts mismatch in {dataset}'
        )

    if int(strata.loc[
        strata['flag'].eq(1), 'population_n'
    ].sum()) != expected['flagged']:
        raise RuntimeError(
            f'STOP: Stratum flagged counts mismatch in {dataset}'
        )

    if int(strata['sample_n'].sum()) != len(meta):
        raise RuntimeError(
            f'STOP: Stratum sampled counts mismatch in {dataset}'
        )

    # Independently cross-check every stratum against the
    # source-row metadata retained in Cell 4.
    observed_counts = (
        meta.groupby([key, 'frozen_flag'])
        .size()
        .to_dict()
    )

    for item in strata.itertuples(index=False):
        stratum_key = (str(item.channel), int(item.flag))
        if observed_counts.get(stratum_key, 0) != int(item.sample_n):
            raise RuntimeError(
                f'STOP: Stratum identity/count mismatch: '
                f'{dataset}, {stratum_key}'
            )

        if not np.isclose(
            float(item.design_weight),
            item.population_n / item.sample_n,
            rtol=0, atol=1e-12
        ):
            raise RuntimeError(
                f'STOP: Invalid design weight: {dataset}, {stratum_key}'
            )

    # Reproduce the stored global feature estimates.
    all_abs = weighted_mean(np.abs(phi), weights)

    flagged = flags == 1
    flagged_abs = weighted_mean(
        np.abs(phi[flagged]),
        weights[flagged]
    )

    flagged_shortening = weighted_mean(
        np.maximum(-phi[flagged], 0.0),
        weights[flagged]
    )

    comparisons = {
        'all_test_abs': all_abs,
        'flagged_abs': flagged_abs,
        'flagged_shortening': flagged_shortening
    }

    for column, computed in comparisons.items():
        saved = table[column].to_numpy(dtype=np.float64)

        if not np.allclose(
            computed, saved, atol=1e-12, rtol=0
        ):
            raise RuntimeError(
                f'STOP: Saved importance mismatch: {dataset}, {column}'
            )

    # Verify group totals independently of Cell 4.
    all_groups = group_summary(all_abs)
    flagged_groups = group_summary(flagged_shortening)

    for group in FEATURE_GROUPS:
        expected_all = float(
            group_table.set_index('feature_group').loc[
                group, 'all_test_abs_share_pct'
            ]
        )

        expected_shortening = float(
            group_table.set_index('feature_group').loc[
                group, 'flagged_shortening_share_pct'
            ]
        )

        computed_all = float(
            all_groups.set_index('feature_group').loc[
                group, 'share_pct'
            ]
        )

        computed_shortening = float(
            flagged_groups.set_index('feature_group').loc[
                group, 'share_pct'
            ]
        )

        if not (
            np.isclose(expected_all, computed_all, atol=1e-10, rtol=0)
            and np.isclose(
                expected_shortening,
                computed_shortening,
                atol=1e-10,
                rtol=0
            )
        ):
            raise RuntimeError(
                f'STOP: Feature-group aggregation mismatch: '
                f'{dataset}, {group}'
            )

    print('\nAUDIT RESULTS')
    print('Total sample rows:', len(meta))
    print('Population-weight sum:', f'{weights.sum():.6f}')
    print('Flagged population-weight sum:',
          f'{weights[flagged].sum():.6f}')
    print('Channel/flag strata:', len(strata))
    print('Feature importance reproduction: PASS')
    print('Group importance reproduction: PASS')
    print('Source-row uniqueness and alignment: PASS')

    print('\nFLAGGED-WINDOW FEATURE GROUPS')
    print('The two importance measures have different meanings.')
    print('Group share: summed contribution across all features in group.')
    print('Mean per feature: group magnitude divided by feature count.\n')

    print(
        flagged_groups[
            [
                'feature_group',
                'n_features',
                'share_pct',
                'mean_per_feature'
            ]
        ].to_string(
            index=False,
            float_format=lambda x: f'{x:.6f}'
        )
    )

    return {
        'meta': meta,
        'phi': phi,
        'weights': weights,
        'flags': flags,
        'shortening': flagged_shortening,
        'group_summary': flagged_groups
    }

print('=' * 100)
print('D4 / C3-11 — GLOBAL TREE SHAP EVIDENCE AUDIT')
print('AND EQE-EXCLUSION SENSITIVITY')
print('=' * 100)

audited = {}

for dataset, expected in EXPECTED.items():

    print('\n' + '=' * 100)
    print('DATASET:', dataset)
    print('=' * 100)

    audited[dataset] = inspect_result(
        dataset,
        D4_GLOBAL_RESULTS[dataset],
        expected
    )

    print('\nDATASET AUDIT: PASS')

# ============================================================
# EQE decomposition of the existing AMPds2 flagged sample
# ============================================================

print('\n' + '=' * 100)
print('AMPds2 — FLAGGED-WINDOW ATTRIBUTIONS WITH AND WITHOUT EQE')
print('=' * 100)

amp = audited['AMPds2']

meta = amp['meta']
phi = amp['phi']
weights = amp['weights']
flags = amp['flags']

eqe_mask = meta['channel_code'].eq('EQE').to_numpy()
flagged_mask = flags == 1

subsets = {
    'All AMPds2 flagged': flagged_mask,
    'EQE flagged only': flagged_mask & eqe_mask,
    'AMPds2 flagged excluding EQE': flagged_mask & ~eqe_mask
}

group_tables = {}
feature_rankings = {}

for name, mask in subsets.items():

    if not mask.any():
        raise RuntimeError(
            f'STOP: Empty diagnostic subset: {name}'
        )

    # Existing TreeSHAP values, weighted within this subset.
    shortening = weighted_mean(
        np.maximum(-phi[mask], 0.0),
        weights[mask]
    )

    group_tables[name] = group_summary(shortening)

    feature_rankings[name] = (
        pd.DataFrame({
            'feature': FEATURES,
            'mean_path_shortening': shortening
        })
        .sort_values('mean_path_shortening', ascending=False)
        .reset_index(drop=True)
    )

    print('\nSUBSET:', name)
    print('Sampled windows:', int(mask.sum()))
    print('Represented population windows:',
          f'{weights[mask].sum():.6f}')

comparison = None

for name, groups in group_tables.items():
    column = (
        groups.set_index('feature_group')['share_pct']
        .rename(name)
    )

    comparison = (
        column.to_frame()
        if comparison is None
        else comparison.join(column)
    )

comparison = comparison.loc[list(FEATURE_GROUPS)]

print('\nFEATURE-GROUP COMPARISON')
print('Percentage of summed path-shortening magnitude.')
print(
    comparison.to_string(
        float_format=lambda x: f'{x:.3f}'
    )
)

print('\nTOP 10 FEATURES — AMPds2 FLAGGED EXCLUDING EQE')
print(
    feature_rankings['AMPds2 flagged excluding EQE']
    .head(10)
    .to_string(
        index=False,
        float_format=lambda x: f'{x:.6f}'
    )
)

full_top5 = set(
    feature_rankings['All AMPds2 flagged']
    .head(5)['feature']
)

excluded_top5 = set(
    feature_rankings['AMPds2 flagged excluding EQE']
    .head(5)['feature']
)

print('\nEQE CONTRIBUTION TO AMPds2 FLAGGED POPULATION')

eqe_flagged_weight = float(
    weights[eqe_mask & flagged_mask].sum()
)

all_flagged_weight = float(
    weights[flagged_mask].sum()
)

eqe_fraction = 100.0 * (
    eqe_flagged_weight / all_flagged_weight
)

print('EQE flagged population:', f'{eqe_flagged_weight:.0f}')
print('All AMPds2 flagged population:', f'{all_flagged_weight:.0f}')
print('EQE share of flagged population:', f'{eqe_fraction:.6f}%')
print('Shared top-five features after EQE exclusion:',
      len(full_top5 & excluded_top5), 'of 5')

if not np.isclose(
    eqe_flagged_weight, 5375.0, atol=1e-7, rtol=0
):
    raise RuntimeError('STOP: EQE historical flagged count mismatch.')

if not np.isclose(
    all_flagged_weight, 14560.0, atol=1e-7, rtol=0
):
    raise RuntimeError('STOP: AMPds2 flagged population mismatch.')

print('\n' + '=' * 100)
print('CELL 5 COMPLETE — ALL VERIFICATION CHECKS PASSED')
print('EQE sensitivity derived from existing global SHAP values.')
print('No new explanation computation.')
print('No model fitting, score changes, or threshold changes.')
print('No file writes.')
print('D4 / C3-11 remains IN PROGRESS pending evidence preservation')
print('and manuscript integration.')
print('=' * 100)


In [ ]:

# ============================================================
# D4 / C3-11 — CELL 6
# PRESERVE VERIFIED GLOBAL TREE SHAP EVIDENCE
#
# Uses existing D4_GLOBAL_RESULTS from Cell 4.
# No model loading, fitting, or SHAP recomputation.
# Writes NEW evidence files only; no historical overwrites.
# ============================================================

from pathlib import Path
from importlib import metadata
import hashlib
import json
import numpy as np
import pandas as pd

BASE = Path('/content/drive/MyDrive/Master/Thesis/Code')

OUTPUT_DIR = (
    BASE / 'experiments'
    / 'D4_C3_11_GLOBAL_TREESHAP_EVIDENCE_20261106'
)

FEATURES = [
    'energy', 'first_last_diff', 'iqr', 'kurtosis', 'mad',
    'max', 'max_abs_diff', 'mean', 'mean_abs_diff',
    'median', 'min', 'n_points', 'peak_count',
    'q25', 'q75', 'range', 'rms', 'skew',
    'slope', 'std', 'std_diff', 'sum', 'var'
]

FEATURE_GROUPS = {
    'Consumption level': [
        'energy', 'max', 'mean', 'median', 'min',
        'q25', 'q75', 'rms', 'sum'
    ],
    'Dispersion': ['iqr', 'mad', 'range', 'std', 'var'],
    'Successive change': [
        'first_last_diff', 'max_abs_diff',
        'mean_abs_diff', 'std_diff'
    ],
    'Trend': ['slope'],
    'Distributional shape': ['kurtosis', 'skew'],
    'Sampling and peaks': ['n_points', 'peak_count']
}

CONFIG = {
    'UK-DALE': {
        'key': 'series_key',
        'test_windows': 72780,
        'test_flags': 3551,
        'sample_windows': 1451,
        'sample_flags': 683,
        'strata': 32,
        'threshold': 0.058252700304033553,
        'source_csv': (
            BASE / 'experiments/EXP_SPLIT_BOUNDARY_PURGE_CORRECTION'
            / 'UKDALE/ukdale_if_scored_boundary_corrected.csv'
        )
    },
    'AMPds2': {
        'key': 'channel_code',
        'test_windows': 147147,
        'test_flags': 14560,
        'sample_windows': 1938,
        'sample_flags': 930,
        'strata': 42,
        'threshold': 0.029990580900317053,
        'source_csv': (
            BASE / 'experiments/EXP_SPLIT_BOUNDARY_PURGE_CORRECTION'
            / 'AMPDS2/ampds2_if_scored_boundary_corrected.csv'
        )
    }
}

if 'D4_GLOBAL_RESULTS' not in globals():
    raise RuntimeError(
        'STOP: D4_GLOBAL_RESULTS is unavailable. '
        'Do not proceed without the verified Cell 4 results.'
    )

if set(D4_GLOBAL_RESULTS) != set(CONFIG):
    raise RuntimeError('STOP: Both dataset results are required.')

if OUTPUT_DIR.exists():
    raise RuntimeError(
        'STOP: Evidence directory already exists. '
        'This cell will not overwrite prior evidence: '
        + str(OUTPUT_DIR)
    )

assert sorted(
    f for names in FEATURE_GROUPS.values() for f in names
) == sorted(FEATURES), 'STOP: Invalid feature-group mapping.'

def file_sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def version_of(package):
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return 'not installed'

def calculate_groups(contributions, dataset, subset):
    total = float(contributions.sum())

    if total <= 0 or not np.isfinite(total):
        raise RuntimeError(
            f'STOP: Invalid contribution total: {dataset}, {subset}'
        )

    rows = []

    for group, names in FEATURE_GROUPS.items():
        indices = [FEATURES.index(name) for name in names]
        magnitude = float(contributions[indices].sum())

        rows.append({
            'dataset': dataset,
            'subset': subset,
            'feature_group': group,
            'n_features': len(names),
            'sum_mean_path_shortening': magnitude,
            'share_pct': 100 * magnitude / total,
            'mean_per_feature': magnitude / len(names)
        })

    return pd.DataFrame(rows)

def build_evidence(dataset, cfg, rec):

    meta = rec['sample_metadata'].copy().reset_index(drop=True)
    phi = np.asarray(rec['shap_values'], dtype=np.float64)
    strata = rec['strata'].copy()
    features = rec['feature_importance'].copy()
    groups = rec['feature_groups'].copy()

    if phi.shape != (len(meta), 23):
        raise RuntimeError(f'STOP: SHAP shape mismatch: {dataset}')

    if len(meta) != cfg['sample_windows']:
        raise RuntimeError(f'STOP: Sample count mismatch: {dataset}')

    if features['feature'].tolist() != FEATURES:
        raise RuntimeError(f'STOP: Feature order mismatch: {dataset}')

    if len(strata) != cfg['strata']:
        raise RuntimeError(f'STOP: Stratum count mismatch: {dataset}')

    if not cfg['source_csv'].is_file():
        raise RuntimeError(f'STOP: Historical CSV missing: {dataset}')

    key = cfg['key']
    weights = meta['design_weight'].to_numpy(dtype=np.float64)
    flags = meta['frozen_flag'].to_numpy(dtype=np.int8)
    saved_scores = meta['anomaly_score'].to_numpy(dtype=np.float64)

    if not (
        np.isfinite(phi).all()
        and np.isfinite(weights).all()
        and np.isfinite(saved_scores).all()
        and (weights > 0).all()
    ):
        raise RuntimeError(f'STOP: Invalid numeric values: {dataset}')

    if not meta['split'].eq('test').all():
        raise RuntimeError(f'STOP: Non-test sample row: {dataset}')

    if meta['__csv_row__'].duplicated().any():
        raise RuntimeError(f'STOP: Duplicate sampled row: {dataset}')

    if not np.isin(flags, [0, 1]).all():
        raise RuntimeError(f'STOP: Nonbinary flags: {dataset}')

    if not np.array_equal(
        flags,
        (saved_scores > cfg['threshold']).astype(np.int8)
    ):
        raise RuntimeError(f'STOP: Historical flag mismatch: {dataset}')

    if int(flags.sum()) != cfg['sample_flags']:
        raise RuntimeError(f'STOP: Flagged sample mismatch: {dataset}')

    if not np.isclose(
        weights.sum(), cfg['test_windows'], rtol=0, atol=1e-7
    ):
        raise RuntimeError(f'STOP: Population weight mismatch: {dataset}')

    if not np.isclose(
        weights[flags == 1].sum(),
        cfg['test_flags'], rtol=0, atol=1e-7
    ):
        raise RuntimeError(f'STOP: Flagged weight mismatch: {dataset}')

    if int(strata['population_n'].sum()) != cfg['test_windows']:
        raise RuntimeError(f'STOP: Stratum population mismatch: {dataset}')

    if int(strata['sample_n'].sum()) != len(meta):
        raise RuntimeError(f'STOP: Stratum sample mismatch: {dataset}')

    # Independently reconstruct the original historical score
    # from already-computed additive TreeSHAP path contributions.
    baseline = float(rec['baseline_path_length'])
    normalizer = float(
        2 * (np.log(255) + np.euler_gamma)
        - 2 * 255 / 256
    )
    offset = -0.5

    reconstructed_path = baseline + phi.sum(axis=1)

    reconstructed_score = (
        np.exp2(-reconstructed_path / normalizer) + offset
    )

    reconstruction_error = float(np.max(np.abs(
        reconstructed_score - saved_scores
    )))

    if reconstruction_error > 1e-9:
        raise RuntimeError(
            f'STOP: Historical score reconstruction failed: {dataset}'
        )

    # Independently verify saved feature importance.
    all_abs = np.average(
        np.abs(phi), axis=0, weights=weights
    )

    flagged = flags == 1

    flagged_abs = np.average(
        np.abs(phi[flagged]),
        axis=0,
        weights=weights[flagged]
    )

    flagged_shortening = np.average(
        np.maximum(-phi[flagged], 0.0),
        axis=0,
        weights=weights[flagged]
    )

    for column, computed in [
        ('all_test_abs', all_abs),
        ('flagged_abs', flagged_abs),
        ('flagged_shortening', flagged_shortening)
    ]:
        if not np.allclose(
            features[column].to_numpy(dtype=np.float64),
            computed,
            rtol=0,
            atol=1e-12
        ):
            raise RuntimeError(
                f'STOP: Importance mismatch: {dataset}, {column}'
            )

    # Preserve one explanation row per sampled historical window.
    samples = meta.copy()
    samples['shap_baseline_path'] = baseline
    samples['shap_reconstructed_path'] = reconstructed_path
    samples['reconstructed_anomaly_score'] = reconstructed_score
    samples['score_reconstruction_abs_error'] = np.abs(
        reconstructed_score - saved_scores
    )

    for index, feature in enumerate(FEATURES):
        samples[f'shap_path__{feature}'] = phi[:, index]

    # Produce EQE decomposition without recomputing TreeSHAP.
    subset_feature_rows = []
    subset_group_frames = []

    subset_masks = {
        'all_flagged': flagged
    }

    if dataset == 'AMPds2':
        eqe = meta[key].eq('EQE').to_numpy()

        subset_masks['eqe_flagged'] = flagged & eqe
        subset_masks['flagged_excluding_eqe'] = flagged & ~eqe

        if not np.isclose(
            weights[subset_masks['eqe_flagged']].sum(),
            5375.0, atol=1e-7, rtol=0
        ):
            raise RuntimeError('STOP: EQE flagged population mismatch.')

        if not np.isclose(
            weights[subset_masks['flagged_excluding_eqe']].sum(),
            9185.0, atol=1e-7, rtol=0
        ):
            raise RuntimeError(
                'STOP: Non-EQE flagged population mismatch.'
            )

    for subset_name, mask in subset_masks.items():

        contribution = np.average(
            np.maximum(-phi[mask], 0.0),
            axis=0,
            weights=weights[mask]
        )

        for feature, value in zip(FEATURES, contribution):
            subset_feature_rows.append({
                'dataset': dataset,
                'subset': subset_name,
                'feature': feature,
                'weighted_mean_path_shortening': float(value)
            })

        subset_group_frames.append(
            calculate_groups(contribution, dataset, subset_name)
        )

    subset_features = pd.DataFrame(subset_feature_rows)
    subset_groups = pd.concat(
        subset_group_frames, ignore_index=True
    )

    # Compare the independently reproduced group shares
    # against the previous Cell 4 results.
    all_group = subset_groups.loc[
        subset_groups['subset'].eq('all_flagged')
    ].set_index('feature_group')

    reference_group = groups.set_index('feature_group')

    for group in FEATURE_GROUPS:
        original = float(
            reference_group.loc[
                group, 'flagged_shortening_share_pct'
            ]
        )
        reproduced = float(
            all_group.loc[group, 'share_pct']
        )

        if not np.isclose(
            original, reproduced, rtol=0, atol=1e-10
        ):
            raise RuntimeError(
                f'STOP: Group share mismatch: {dataset}, {group}'
            )

    # Capture all necessary methodological provenance.
    information = {
        'dataset': dataset,
        'model_sha256': rec['model_sha256'],
        'historical_csv_path': str(cfg['source_csv']),
        'model_status': 'functionally verified historical reconstruction',
        'feature_order': FEATURES,
        'input_dtype': 'float32',
        'trees': 300,
        'max_samples_per_tree': 256,
        'sampling_seed': int(rec['sample_seed']),
        'maximum_sample_per_channel_flag_stratum': int(
            rec['max_per_stratum']
        ),
        'historical_test_windows': cfg['test_windows'],
        'historical_test_flags': cfg['test_flags'],
        'original_pooled_threshold': cfg['threshold'],
        'sampled_test_windows': len(meta),
        'sampled_flagged_windows': int(flagged.sum()),
        'channel_flag_strata': len(strata),
        'tree_shap_output': 'average adjusted isolation path length',
        'tree_shap_reference': 'tree_path_dependent; no external background',
        'shap_baseline_path_length': baseline,
        'path_length_normalizer': normalizer,
        'historical_isolation_forest_offset': offset,
        'historical_score_formula':
            'anomaly_score = 2**(-mean_path/c(256)) + offset',
        'attribution_direction':
            'negative path contribution shortens isolation path and increases anomaly score',
        'importance_definition':
            'inverse-stratum-sampling-fraction weighted mean absolute path SHAP',
        'flagged_shortening_definition':
            'weighted mean of max(-SHAP_path, 0) among flagged test windows',
        'feature_group_limitation':
            'summed shares not adjusted for group size or correlated features',
        'estimated_population_limitation':
            'stratified sampled estimates, not full-population SHAP enumeration',
        'maximum_reconstructed_score_error': reconstruction_error
    }

    return {
        'samples': samples,
        'strata': strata,
        'feature_importance': features,
        'feature_groups': groups,
        'subset_feature_importance': subset_features,
        'subset_group_importance': subset_groups,
        'information': information
    }

print('=' * 96)
print('D4 / C3-11 — VERIFIED TREE SHAP EVIDENCE PRESERVATION')
print('=' * 96)

# Complete all checks before creating the evidence directory.
prepared = {}

for dataset, cfg in CONFIG.items():
    print('\nPreflight verification:', dataset)

    prepared[dataset] = build_evidence(
        dataset,
        cfg,
        D4_GLOBAL_RESULTS[dataset]
    )

    info = prepared[dataset]['information']

    print('Sample windows:', info['sampled_test_windows'])
    print('Historical flagged population:', info['historical_test_flags'])
    print(
        'Score reconstruction max error:',
        f"{info['maximum_reconstructed_score_error']:.15g}"
    )
    print('PREFLIGHT: PASS')

# All preflight checks passed; write new evidence files.
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

manifest = {
    'analysis': 'D4 / C3-11 — Global Isolation Forest TreeSHAP',
    'date': '2026-11-06',
    'primary_scoring_protocol': 'unchanged',
    'output_directory': str(OUTPUT_DIR),
    'software': {
        name: version_of(name)
        for name in ['shap', 'scikit-learn', 'numpy', 'pandas']
    },
    'datasets': {},
    'files': {}
}

for dataset, evidence in prepared.items():

    prefix = 'ukdale' if dataset == 'UK-DALE' else 'ampds2'

    for name in [
        'samples',
        'strata',
        'feature_importance',
        'feature_groups',
        'subset_feature_importance',
        'subset_group_importance'
    ]:
        filename = f'{prefix}_{name}.csv'
        destination = OUTPUT_DIR / filename

        evidence[name].to_csv(
            destination,
            index=False,
            float_format='%.17g'
        )

        manifest['files'][filename] = {
            'sha256': file_sha256(destination),
            'rows': len(evidence[name]),
            'bytes': destination.stat().st_size
        }

    manifest['datasets'][dataset] = evidence['information']

manifest_path = OUTPUT_DIR / 'D4_C3_11_EVIDENCE_MANIFEST.json'

with manifest_path.open('w', encoding='utf-8') as handle:
    json.dump(manifest, handle, indent=2, ensure_ascii=False)

# Verify every saved file against its recorded checksum.
for filename, details in manifest['files'].items():

    actual = file_sha256(OUTPUT_DIR / filename)

    if actual != details['sha256']:
        raise RuntimeError(
            f'STOP: Written evidence hash mismatch: {filename}'
        )

# Verify manifest can be read back successfully.
with manifest_path.open('r', encoding='utf-8') as handle:
    manifest_check = json.load(handle)

if set(manifest_check['datasets']) != set(CONFIG):
    raise RuntimeError('STOP: Saved manifest dataset mismatch.')

print('\n' + '=' * 96)
print('CELL 6 COMPLETE — EVIDENCE PRESERVATION: PASS')
print('=' * 96)

print('Evidence directory:', OUTPUT_DIR)
print('CSV evidence files:', len(manifest['files']))
print('Manifest:', manifest_path.name)

for filename in manifest['files']:
    print(' VERIFIED:', filename)

print('\nAll evidence checksums verified.')
print('Historical scoring artifacts remain unchanged.')
print('No model fitting or SHAP recomputation.')
print('No historical files overwritten.')
print('D4 / C3-11: NUMERICAL ANALYSIS PRESERVED.')
print('Manuscript integration remains pending.')


In [ ]:

# ============================================================
# D5 / C3-12 + C4-12 — CELL 1
# HISTORICAL SHAP / LIME ARTIFACT INVENTORY
#
# READ ONLY
# No models loaded or fitted
# No SHAP/LIME explanations computed
# No existing files changed; no new files written
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path('/content/drive/MyDrive/Master/Thesis/Code')
EXPERIMENTS = BASE / 'experiments'

UK_XAI = (
    EXPERIMENTS / 'EXP_UKDALE_Selected_Pipeline/xai'
)
AMP_XAI = (
    EXPERIMENTS / 'EXP_AMPDS2_Selected_Pipeline/xai'
)
AMP_CORRECTION = (
    EXPERIMENTS
    / 'EXP_SPLIT_BOUNDARY_PURGE_CORRECTION'
    / 'AMPDS2/XAI_LIME_CORRECTION'
)

FILES = {
    'UK-DALE': {
        'local_lime': (
            UK_XAI
            / 'ukdale_lime_explanations_selected_test_windows.csv'
        ),
        'local_shap': (
            UK_XAI
            / 'ukdale_shap_explanations_selected_test_windows.csv'
        ),
        'lime_top5': (
            UK_XAI / 'ukdale_lime_top5_per_category.csv'
        ),
        'shap_top5': (
            UK_XAI / 'ukdale_shap_top5_per_category.csv'
        )
    },
    'AMPds2': {
        'historical_lime': (
            AMP_XAI
            / 'ampds2_lime_explanations_selected_test_windows.csv'
        ),
        'historical_shap': (
            AMP_XAI
            / 'ampds2_shap_explanations_selected_test_windows.csv'
        ),
        'historical_lime_top5': (
            AMP_XAI / 'ampds2_lime_top5_per_candidate.csv'
        ),
        'historical_shap_top5': (
            AMP_XAI / 'ampds2_shap_top5_per_candidate.csv'
        ),
        'historical_shap_top10': (
            AMP_XAI / 'ampds2_shap_top10_per_candidate.csv'
        ),
        'corrected_lime': (
            AMP_CORRECTION
            / 'ampds2_lime_explanations_selected_test_windows_CORRECTED.csv'
        ),
        'corrected_lime_top5': (
            AMP_CORRECTION
            / 'ampds2_lime_top5_per_candidate_CORRECTED.csv'
        ),
        'lime_correction_comparison': (
            AMP_CORRECTION
            / 'ampds2_lime_historical_vs_corrected_top5.csv'
        )
    }
}

REQUIRED = {
    'UK-DALE': ['local_lime', 'local_shap'],
    'AMPds2': ['historical_shap', 'corrected_lime']
}

ID_HINTS = [
    'candidate', 'window', 'series', 'meter', 'channel',
    'category', 'timestamp', 'start_time', 'end_time',
    'feature', 'rank', 'weight', 'importance',
    'r2', 'fidelity', 'local_fit', 'exp_score'
]

def fidelity_columns(columns):
    """
    Look for possible local-surrogate R-squared/fidelity
    fields without assuming that anomaly_score is fidelity.
    """
    matches = []

    for column in columns:
        name = column.lower()

        if (
            'fidelity' in name
            or 'r2' in name
            or 'r_squared' in name
            or 'local_fit' in name
            or 'exp_score' in name
            or 'lime_score' in name
        ):
            matches.append(column)

    return matches

def inspect_csv(path, dataset, role):
    print('\n' + '-' * 94)
    print(f'{dataset} — {role}')
    print('-' * 94)
    print('File:', path.name)

    if not path.is_file():
        print('STATUS: MISSING')
        return None

    size = path.stat().st_size

    print('STATUS: FOUND')
    print('Size:', f'{size:,}', 'bytes')

    # All expected historical XAI files were previously
    # identified as small CSVs. Do not accidentally load
    # a newly substituted, unexpectedly large file.
    if size > 10 * 1024 * 1024:
        raise RuntimeError(
            f'STOP: Unexpectedly large XAI file: {path}'
        )

    df = pd.read_csv(path, low_memory=False)

    print('Rows:', len(df))
    print('Columns:', len(df.columns))
    print('Column names:')
    print(' ', df.columns.tolist())

    if df.columns.duplicated().any():
        raise RuntimeError(
            f'STOP: Duplicate CSV column names: {path.name}'
        )

    if df.empty:
        print('WARNING: Empty XAI artifact.')

    possible_fidelity = fidelity_columns(df.columns)

    print('Potential LIME fidelity fields:',
          possible_fidelity if possible_fidelity else 'NONE FOUND')

    preview_cols = [
        col for col in df.columns
        if any(hint in col.lower() for hint in ID_HINTS)
    ][:8]

    if not preview_cols:
        preview_cols = df.columns.tolist()[:6]

    if len(df):
        preview = df.loc[:, preview_cols].head(2).copy()

        # Prevent very long attribution representations
        # from overwhelming the console.
        for col in preview.select_dtypes(
            include=['object', 'string']
        ).columns:
            preview[col] = (
                preview[col].astype(str).str.slice(0, 110)
            )

        print('\nFirst two rows — selected identification fields:')
        print(preview.to_string(index=False))

    # Do not interpret score columns as R2 unless
    # their semantics have been established.
    return {
        'dataset': dataset,
        'role': role,
        'path': str(path),
        'rows': len(df),
        'columns': df.columns.tolist(),
        'potential_fidelity_columns': possible_fidelity
    }

print('=' * 94)
print('D5 / C3-12 + C4-12 — HISTORICAL XAI EVIDENCE INVENTORY')
print('=' * 94)

if not BASE.is_dir():
    raise RuntimeError(
        'STOP: Thesis Code directory unavailable. '
        'Verify Google Drive is mounted.'
    )

print('Research directory:', BASE)
print('Pandas version:', pd.__version__)
print('NumPy version:', np.__version__)

D5_INVENTORY = {}
missing_required = []

for dataset, roles in FILES.items():

    print('\n' + '=' * 94)
    print('DATASET:', dataset)
    print('=' * 94)

    D5_INVENTORY[dataset] = {}

    for role, path in roles.items():

        result = inspect_csv(path, dataset, role)

        D5_INVENTORY[dataset][role] = result

        if result is None and role in REQUIRED[dataset]:
            missing_required.append((dataset, role))

print('\n' + '=' * 94)
print('ADDITIONAL PROVENANCE FILES')
print('=' * 94)

# Inspect names only. No recursive full-drive search.
for directory in [UK_XAI, AMP_XAI, AMP_CORRECTION]:

    print('\nDirectory:', directory)

    if not directory.is_dir():
        print('MISSING DIRECTORY')
        continue

    supplementary = sorted(
        path for path in directory.iterdir()
        if path.is_file()
        and path.suffix.lower() in [
            '.json', '.txt', '.md', '.py', '.ipynb'
        ]
    )

    print('Metadata/code files:', len(supplementary))

    for path in supplementary[:25]:
        print(' ', path.name)

print('\n' + '=' * 94)
print('INVENTORY SUMMARY')
print('=' * 94)

for dataset, entries in D5_INVENTORY.items():

    present = sum(value is not None for value in entries.values())

    print(f'\n{dataset}: {present}/{len(entries)} expected CSVs found')

    for role, result in entries.items():
        if result is not None:
            print(
                f'  {role:28s} | '
                f'rows={result["rows"]:5d} | '
                f'fidelity_columns={result["potential_fidelity_columns"]}'
            )

if missing_required:
    print('\nREQUIRED ARTIFACTS MISSING:', missing_required)
    raise RuntimeError(
        'STOP: Required XAI evidence is missing. '
        'Do not proceed to fidelity calculations.'
    )

print('\n' + '=' * 94)
print('CELL 1 COMPLETE — REQUIRED XAI ARTIFACTS LOCATED')
print('Historical and corrected AMPds2 LIME kept separate.')
print('No claim of fidelity or SHAP/LIME agreement yet.')
print('No models loaded, fitted, or retrained.')
print('No explanations computed and no files modified.')
print('D5 remains IN PROGRESS.')
print('=' * 94)


In [ ]:

# ============================================================
# D5 / C3-12 + C4-12 — CELL 2
# SHAP–LIME WINDOW ALIGNMENT AND INTERFACE AUDIT
#
# READ ONLY
# No model fitting or explanation recomputation
# No historical files modified or created
# ============================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd

BASE = Path('/content/drive/MyDrive/Master/Thesis/Code')
EXP = BASE / 'experiments'

UK_DIR = EXP / 'EXP_UKDALE_Selected_Pipeline/xai'
AMP_DIR = EXP / 'EXP_AMPDS2_Selected_Pipeline/xai'
AMP_CORR = (
    EXP / 'EXP_SPLIT_BOUNDARY_PURGE_CORRECTION'
    / 'AMPDS2/XAI_LIME_CORRECTION'
)

CONFIG = {
    'UK-DALE': {
        'lime': UK_DIR / 'ukdale_lime_explanations_selected_test_windows.csv',
        'shap': UK_DIR / 'ukdale_shap_explanations_selected_test_windows.csv',
        'keys': ['building', 'meter', 'window_start', 'window_end'],
        'expected_windows': 5
    },
    'AMPds2': {
        'lime': AMP_CORR /
            'ampds2_lime_explanations_selected_test_windows_CORRECTED.csv',
        'shap': AMP_DIR /
            'ampds2_shap_explanations_selected_test_windows.csv',
        'keys': [
            'candidate_id', 'channel_code',
            'window_start', 'window_end'
        ],
        'expected_windows': 8
    }
}

FEATURES = [
    'energy', 'first_last_diff', 'iqr', 'kurtosis', 'mad',
    'max', 'max_abs_diff', 'mean', 'mean_abs_diff',
    'median', 'min', 'n_points', 'peak_count',
    'q25', 'q75', 'range', 'rms', 'skew',
    'slope', 'std', 'std_diff', 'sum', 'var'
]

def parse_lime_feature(condition):
    """
    Identify a canonical engineered feature in a LIME
    discretization condition.

    Match complete feature tokens, with or without '_z'.
    Do not mistake 'mean' for 'mean_abs_diff', etc.
    Return None for ambiguous or unmatched conditions.
    """
    condition = str(condition)

    matches = []

    for feature in FEATURES:
        expression = (
            r'(?<![A-Za-z0-9_])'
            + re.escape(feature)
            + r'(?:_z)?(?![A-Za-z0-9_])'
        )

        if re.search(expression, condition):
            matches.append(feature)

    return matches[0] if len(matches) == 1 else None

def make_groups(df, keys):
    return {
        tuple(key): group.copy()
        for key, group in df.groupby(
            keys, sort=True, dropna=False
        )
    }

def audit_dataset(dataset, cfg):

    print('\n' + '=' * 100)
    print('DATASET:', dataset)
    print('=' * 100)

    if not cfg['lime'].is_file() or not cfg['shap'].is_file():
        raise RuntimeError(
            f'STOP: Missing required explanation file: {dataset}'
        )

    lime = pd.read_csv(cfg['lime'], low_memory=False)
    shap = pd.read_csv(cfg['shap'], low_memory=False)

    required_lime = (
        cfg['keys']
        + ['anomaly_score', 'rank',
           'lime_condition', 'lime_weight']
    )

    required_shap = (
        cfg['keys']
        + ['anomaly_score', 'feature',
           'shap_value', 'abs_shap_value']
    )

    missing_lime = [
        col for col in required_lime
        if col not in lime.columns
    ]

    missing_shap = [
        col for col in required_shap
        if col not in shap.columns
    ]

    if missing_lime or missing_shap:
        raise RuntimeError(
            f'STOP: Missing columns for {dataset}. '
            f'LIME: {missing_lime}, SHAP: {missing_shap}'
        )

    lime_groups = make_groups(lime, cfg['keys'])
    shap_groups = make_groups(shap, cfg['keys'])

    lime_keys = set(lime_groups)
    shap_keys = set(shap_groups)

    print('\n1. WINDOW COVERAGE')
    print('LIME rows:', len(lime))
    print('SHAP rows:', len(shap))
    print('Unique LIME windows:', len(lime_keys))
    print('Unique SHAP windows:', len(shap_keys))
    print('Expected windows:', cfg['expected_windows'])
    print('LIME-only windows:', len(lime_keys - shap_keys))
    print('SHAP-only windows:', len(shap_keys - lime_keys))

    issues = []

    if (
        len(lime_keys) != cfg['expected_windows']
        or len(shap_keys) != cfg['expected_windows']
        or lime_keys != shap_keys
    ):
        issues.append('Window identity or coverage mismatch')

    if lime_keys != shap_keys:
        for key in sorted(lime_keys - shap_keys, key=str)[:5]:
            print('LIME-only identity:', key)

        for key in sorted(shap_keys - lime_keys, key=str)[:5]:
            print('SHAP-only identity:', key)

    print('\n2. LIME CONFIGURATION METADATA')

    if dataset == 'AMPds2':
        for column in [
            'lime_mode',
            'lime_label',
            'lime_num_features',
            'lime_random_state',
            'threshold_provenance'
        ]:
            if column in lime.columns:
                print(
                    column, ':',
                    lime[column].value_counts(
                        dropna=False
                    ).to_dict()
                )
            else:
                print(column, ': NOT RECORDED')
    else:
        print(
            'UK-DALE LIME mode is not explicitly recorded '
            'in the inspected explanation CSV.'
        )
        print(
            'The historical implementation must be '
            'verified separately before a fidelity claim.'
        )

    print('\n3. PAIRED EXPLANATION VERIFICATION')

    paired_records = []
    unmatched_conditions = []
    duplicate_feature_windows = []
    max_score_difference = 0.0

    for key in sorted(lime_keys & shap_keys, key=str):

        ldf = lime_groups[key].copy()
        sdf = shap_groups[key].copy()

        if len(ldf) != 10:
            issues.append(
                f'LIME row count {len(ldf)} for {key}'
            )

        if len(sdf) != 23:
            issues.append(
                f'SHAP row count {len(sdf)} for {key}'
            )

        # Verify unique canonical SHAP feature coverage.
        shap_features = sdf['feature'].astype(str).tolist()

        if (
            len(set(shap_features)) != len(FEATURES)
            or set(shap_features) != set(FEATURES)
        ):
            issues.append(
                f'SHAP feature coverage mismatch for {key}'
            )

        # Verify LIME's original 1-based feature ordering.
        ranks = pd.to_numeric(
            ldf['rank'], errors='coerce'
        )

        if (
            ranks.isna().any()
            or sorted(ranks.tolist()) != list(range(1, 11))
        ):
            issues.append(
                f'Unexpected LIME ranks for {key}'
            )

        ldf['rank_numeric'] = ranks
        ldf = ldf.sort_values('rank_numeric')

        ldf['parsed_feature'] = (
            ldf['lime_condition']
            .map(parse_lime_feature)
        )

        for _, item in ldf.iterrows():
            if item['parsed_feature'] is None:
                unmatched_conditions.append({
                    'dataset': dataset,
                    'identity': str(key),
                    'rank': item['rank'],
                    'condition': item['lime_condition']
                })

        parsed = ldf['parsed_feature'].dropna()

        if parsed.duplicated().any():
            duplicate_feature_windows.append(str(key))

        lime_scores = pd.to_numeric(
            ldf['anomaly_score'], errors='coerce'
        ).to_numpy(dtype=np.float64)

        shap_scores = pd.to_numeric(
            sdf['anomaly_score'], errors='coerce'
        ).to_numpy(dtype=np.float64)

        if (
            not np.isfinite(lime_scores).all()
            or not np.isfinite(shap_scores).all()
        ):
            issues.append(
                f'Nonfinite saved anomaly score: {key}'
            )
            continue

        score_diff = float(
            np.max(np.abs(
                lime_scores[:, None] - shap_scores[None, :]
            ))
        )

        max_score_difference = max(
            max_score_difference, score_diff
        )

        if score_diff > 1e-9:
            issues.append(
                f'SHAP/LIME saved score mismatch for {key}: '
                f'{score_diff:.12g}'
            )

        shap_top5 = (
            sdf.sort_values(
                'abs_shap_value', ascending=False
            )['feature']
            .astype(str)
            .head(5)
            .tolist()
        )

        lime_top5 = (
            ldf['parsed_feature']
            .head(5)
            .tolist()
        )

        paired_records.append({
            'dataset': dataset,
            'identity': str(key),
            'lime_score': float(lime_scores[0]),
            'shap_score': float(shap_scores[0]),
            'lime_top5': lime_top5,
            'shap_top5': shap_top5,
            'lime_all_ranked_features':
                ldf['parsed_feature'].tolist(),
            'shap_all_ranked_features':
                sdf.sort_values(
                    'abs_shap_value', ascending=False
                )['feature'].astype(str).tolist()
        })

    print('Paired windows:', len(paired_records))
    print(
        'Maximum paired anomaly-score discrepancy:',
        f'{max_score_difference:.15g}'
    )
    print(
        'Unmatched LIME conditions:',
        len(unmatched_conditions)
    )
    print(
        'Windows with repeated parsed LIME features:',
        len(duplicate_feature_windows)
    )

    if unmatched_conditions:
        print('\nUNMATCHED LIME CONDITIONS')
        for item in unmatched_conditions[:15]:
            print(item)

    if duplicate_feature_windows:
        print('\nREPEATED LIME FEATURE IDENTITIES')
        for item in duplicate_feature_windows[:10]:
            print(item)

    print('\n4. PAIRED TOP-FIVE FEATURE LISTS')
    print('For identity inspection only — no agreement statistics.')

    for record in paired_records:
        print('\nWindow:', record['identity'])
        print('Saved anomaly score:',
              f"{record['shap_score']:.9f}")
        print('SHAP top five:',
              ', '.join(record['shap_top5']))
        print('LIME top five:',
              ', '.join(str(v) for v in record['lime_top5']))

    if unmatched_conditions:
        issues.append(
            'One or more LIME condition features could not be parsed'
        )

    if duplicate_feature_windows:
        issues.append(
            'Repeated parsed LIME features require review'
        )

    if issues:
        print('\nDATASET STATUS: REVIEW')
        for issue in issues[:25]:
            print(' -', issue)

        raise RuntimeError(
            f'STOP: {dataset} explanation alignment requires review. '
            'Do not calculate agreement metrics yet.'
        )

    print('\nDATASET ALIGNMENT: PASS')

    return pd.DataFrame(paired_records)

print('=' * 100)
print('D5 / C3-12 + C4-12 — LOCAL EXPLANATION PAIRING AUDIT')
print('=' * 100)

D5_PAIRED = {}

for dataset, config in CONFIG.items():
    D5_PAIRED[dataset] = audit_dataset(dataset, config)

print('\n' + '=' * 100)
print('CELL 2 COMPLETE — SHAP/LIME PAIRING VERIFIED')
print('Total aligned explanation windows:',
      sum(len(df) for df in D5_PAIRED.values()))
print('All compared explanations retain historical scores.')
print('No fidelity R-squared has been inferred or invented.')
print('No agreement statistics calculated yet.')
print('No model fitting or explanation recomputation.')
print('No files written or modified.')
print('D5 remains IN PROGRESS.')
print('=' * 100)


In [ ]:

# ============================================================
# D5 / C4-12 — CELL 3
# EXISTING SHAP–LIME RANKING AGREEMENT: DIAGNOSTIC BASELINE
#
# Reuses verified D5_PAIRED from Cell 2.
# No model or explainer computations.
# No file writes.
# NOT final same-target comparison for AMPds2.
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

EXPECTED_WINDOWS = {
    'UK-DALE': 5,
    'AMPds2': 8
}

EXPECTED_FEATURES = [
    'energy', 'first_last_diff', 'iqr', 'kurtosis', 'mad',
    'max', 'max_abs_diff', 'mean', 'mean_abs_diff',
    'median', 'min', 'n_points', 'peak_count',
    'q25', 'q75', 'range', 'rms', 'skew',
    'slope', 'std', 'std_diff', 'sum', 'var'
]

if 'D5_PAIRED' not in globals():
    raise RuntimeError(
        'STOP: Run Cell 2 first; D5_PAIRED is unavailable.'
    )

if set(D5_PAIRED) != set(EXPECTED_WINDOWS):
    raise RuntimeError(
        'STOP: Expected UK-DALE and AMPds2 pairing records.'
    )

def validate_ranked_features(features, expected_count, context):
    if not isinstance(features, (list, tuple)):
        raise RuntimeError(
            f'STOP: Ranked features are not a list: {context}'
        )

    if (
        len(features) != expected_count
        or len(set(features)) != expected_count
        or any(feature not in EXPECTED_FEATURES for feature in features)
    ):
        raise RuntimeError(
            f'STOP: Invalid ranked feature list: {context}'
        )

def calculate_agreement(row, dataset):
    identity = str(row['identity'])

    context = f'{dataset}: {identity}'

    lime_top10 = list(row['lime_all_ranked_features'])
    shap_top23 = list(row['shap_all_ranked_features'])

    validate_ranked_features(lime_top10, 10, context)
    validate_ranked_features(shap_top23, 23, context)

    if set(shap_top23) != set(EXPECTED_FEATURES):
        raise RuntimeError(
            f'STOP: Incomplete SHAP feature universe: {context}'
        )

    lime_top5 = lime_top10[:5]
    shap_top5 = shap_top23[:5]

    # Ensure the stored top-five lists agree with their
    # independently paired full ranked lists.
    if (
        lime_top5 != list(row['lime_top5'])
        or shap_top5 != list(row['shap_top5'])
    ):
        raise RuntimeError(
            f'STOP: Inconsistent top-five feature lists: {context}'
        )

    lime_score = float(row['lime_score'])
    shap_score = float(row['shap_score'])

    if (
        not np.isfinite(lime_score)
        or not np.isfinite(shap_score)
        or abs(lime_score - shap_score) > 1e-10
    ):
        raise RuntimeError(
            f'STOP: Saved score mismatch: {context}'
        )

    shared = set(lime_top5) & set(shap_top5)
    union = set(lime_top5) | set(shap_top5)

    overlap_count = len(shared)
    jaccard = overlap_count / len(union)

    # Rank consistency is evaluated only among the ten
    # features explicitly ranked by LIME.
    #
    # SHAP ranks refer to the full 23-feature ordering.
    # No artificial ranks are assigned to omitted LIME
    # features.
    shap_rank = {
        feature: rank
        for rank, feature in enumerate(shap_top23, start=1)
    }

    lime_ranks = np.arange(1, 11, dtype=np.float64)

    corresponding_shap_ranks = np.array(
        [shap_rank[f] for f in lime_top10],
        dtype=np.float64
    )

    rho = float(
        spearmanr(
            lime_ranks,
            corresponding_shap_ranks
        ).statistic
    )

    if not np.isfinite(rho):
        raise RuntimeError(
            f'STOP: Undefined rank correlation: {context}'
        )

    # Independent Jaccard identity for two size-five sets:
    # union size = 10 - intersection size.
    independent_jaccard = overlap_count / (
        10 - overlap_count
    )

    if not np.isclose(
        jaccard, independent_jaccard,
        atol=1e-15, rtol=0
    ):
        raise RuntimeError(
            f'STOP: Jaccard consistency check failed: {context}'
        )

    return {
        'dataset': dataset,
        'identity': identity,
        'anomaly_score': shap_score,
        'top5_overlap_count': overlap_count,
        'top5_jaccard': jaccard,
        'spearman_top10_lime_features': rho,
        'shared_top5': ', '.join(
            f for f in shap_top5 if f in shared
        ),
        'shap_top5': ', '.join(shap_top5),
        'lime_top5': ', '.join(lime_top5),
        'comparison_status': (
            'historical_diagnostic_classification_lime'
            if dataset == 'AMPds2'
            else 'historical_diagnostic_uk_lime'
        )
    }

print('=' * 100)
print('D5 / C4-12 — HISTORICAL SHAP–LIME AGREEMENT')
print('DIAGNOSTIC BASELINE — NOT FINAL SAME-TARGET EVALUATION')
print('=' * 100)

records = []

for dataset, expected_n in EXPECTED_WINDOWS.items():

    paired = D5_PAIRED[dataset]

    if len(paired) != expected_n:
        raise RuntimeError(
            f'STOP: Expected {expected_n} windows in {dataset}, '
            f'found {len(paired)}'
        )

    for _, row in paired.iterrows():
        records.append(calculate_agreement(row, dataset))

D5_DIAGNOSTIC_AGREEMENT = pd.DataFrame(records)

if len(D5_DIAGNOSTIC_AGREEMENT) != 13:
    raise RuntimeError(
        'STOP: Expected exactly 13 explanation comparisons.'
    )

print('\n1. WINDOW-LEVEL RESULTS')

for dataset, group in D5_DIAGNOSTIC_AGREEMENT.groupby(
    'dataset', sort=False
):
    print('\nDATASET:', dataset)

    for _, item in group.iterrows():
        print('\nWindow:', item['identity'])
        print('Shared top-five features:',
              int(item['top5_overlap_count']))
        print('Top-five Jaccard:',
              f"{item['top5_jaccard']:.6f}")
        print('Spearman rho (LIME-selected ten):',
              f"{item['spearman_top10_lime_features']:.6f}")
        print('Shared features:',
              item['shared_top5'] or '(none)')

print('\n2. DATASET-LEVEL DESCRIPTIVE SUMMARY')

summary_rows = []

for dataset, group in D5_DIAGNOSTIC_AGREEMENT.groupby(
    'dataset', sort=False
):
    summary_rows.append({
        'dataset': dataset,
        'n_windows': len(group),
        'mean_top5_overlap': group[
            'top5_overlap_count'
        ].mean(),
        'mean_jaccard': group[
            'top5_jaccard'
        ].mean(),
        'median_jaccard': group[
            'top5_jaccard'
        ].median(),
        'min_jaccard': group[
            'top5_jaccard'
        ].min(),
        'max_jaccard': group[
            'top5_jaccard'
        ].max(),
        'mean_spearman': group[
            'spearman_top10_lime_features'
        ].mean(),
        'median_spearman': group[
            'spearman_top10_lime_features'
        ].median()
    })

D5_DIAGNOSTIC_SUMMARY = pd.DataFrame(summary_rows)

print(
    D5_DIAGNOSTIC_SUMMARY.to_string(
        index=False,
        float_format=lambda x: f'{x:.6f}'
    )
)

print('\n3. ALL 13 ILLUSTRATIVE WINDOWS')

print('Mean top-five Jaccard:',
      f"{D5_DIAGNOSTIC_AGREEMENT['top5_jaccard'].mean():.6f}")

print('Mean Spearman rho:',
      f"{D5_DIAGNOSTIC_AGREEMENT['spearman_top10_lime_features'].mean():.6f}")

print('\n4. INTERPRETATION RESTRICTIONS')
print('These are descriptive comparisons on selected windows.')
print('AMPds2 LIME used a classification pseudo-probability.')
print('UK-DALE LIME interface still requires provenance verification.')
print('Spearman rho uses only the ten LIME-reported features.')
print('No ranks were imputed for the other thirteen LIME features.')
print('These values do NOT measure local LIME R-squared fidelity.')
print('No statistical significance or explanation stability inferred.')

print('\n' + '=' * 100)
print('CELL 3 COMPLETE — DIAGNOSTIC AGREEMENT CALCULATED')
print('Both datasets: 13/13 original windows compared.')
print('Results stored only in notebook memory.')
print('No historical artifacts changed.')
print('C3-12 remains OPEN.')
print('C4-12 remains OPEN pending same-target explanation review.')
print('=' * 100)


In [ ]:

# ============================================================
# D5 / C3-12 — CELL 4
# HISTORICAL LIME IMPLEMENTATION PROVENANCE AUDIT
#
# READ ONLY
# No model loading or fitting
# No LIME/SHAP computation
# No files written or modified
# ============================================================

from pathlib import Path
import os
import json
import re

BASE = Path('/content/drive/MyDrive/Master/Thesis/Code')
EXP = BASE / 'experiments'

SEARCH_ROOTS = [
    ('UK selected pipeline',
     EXP / 'EXP_UKDALE_Selected_Pipeline', 4),
    ('AMPds2 selected pipeline',
     EXP / 'EXP_AMPDS2_Selected_Pipeline', 4),
    ('AMPds2 LIME correction',
     EXP / 'EXP_SPLIT_BOUNDARY_PURGE_CORRECTION'
     / 'AMPDS2', 4),
    ('Thesis code root',
     BASE, 1),
    ('Thesis notebooks',
     BASE / 'notebooks', 3)
]

ALLOWED_SUFFIXES = {
    '.py', '.ipynb', '.json', '.md', '.txt',
    '.yaml', '.yml'
}

SKIP_DIRECTORIES = {
    '.git', '.ipynb_checkpoints', '__pycache__',
    'node_modules', '.venv', 'venv',
    'raw_data', 'processed_data'
}

MAX_SOURCE_BYTES = 15 * 1024 * 1024
MAX_FILES_PER_ROOT = 250
MAX_MATCHED_FILES_SHOWN = 20
MAX_MATCHED_LINES_PER_FILE = 14

PATTERN = re.compile(
    r'LimeTabularExplainer'
    r'|lime_tabular'
    r'|\.explain_instance\s*\('
    r'|(?:exp|explanation)\.score\b'
    r'|lime_mode'
    r'|lime_random_state'
    r'|predict_proba'
    r'|pseudo.probab'
    r'|discretize_continuous'
    r'|kernel_width'
    r'|num_samples\s*=',
    re.IGNORECASE
)

D5_CODE_AUDIT = []

def discover_source_files(root, max_depth):
    if not root.is_dir():
        return [], False

    found = []
    capped = False

    for current, dirs, files in os.walk(root):
        current = Path(current)
        depth = len(current.relative_to(root).parts)

        dirs[:] = sorted(
            d for d in dirs
            if d not in SKIP_DIRECTORIES
        )

        if depth >= max_depth:
            dirs[:] = []

        for filename in sorted(files):
            path = current / filename

            if path.suffix.lower() not in ALLOWED_SUFFIXES:
                continue

            try:
                size = path.stat().st_size
            except OSError:
                continue

            if size > MAX_SOURCE_BYTES:
                continue

            found.append(path)

            if len(found) >= MAX_FILES_PER_ROOT:
                capped = True
                return found, capped

    return found, capped

def read_source_lines(path):
    try:
        if path.suffix.lower() == '.ipynb':
            with path.open('r', encoding='utf-8') as f:
                notebook = json.load(f)

            lines = []

            for index, cell in enumerate(
                notebook.get('cells', [])
            ):
                if cell.get('cell_type') != 'code':
                    continue

                source = cell.get('source', [])

                if isinstance(source, list):
                    source = ''.join(source)

                lines.append(
                    f'# --- NOTEBOOK CELL {index} ---'
                )
                lines.extend(str(source).splitlines())

            return lines

        with path.open(
            'r', encoding='utf-8', errors='replace'
        ) as f:
            return f.read().splitlines()

    except (OSError, ValueError, UnicodeError) as exc:
        return [f'[READ ERROR: {type(exc).__name__}]']

def matching_snippets(lines):
    hit_indices = [
        i for i, line in enumerate(lines)
        if PATTERN.search(line)
    ]

    snippets = []

    for index in hit_indices[:MAX_MATCHED_LINES_PER_FILE]:
        start = max(0, index - 2)
        end = min(len(lines), index + 3)

        excerpt = []

        for j in range(start, end):
            text = lines[j].strip()

            if len(text) > 220:
                text = text[:220] + ' ...'

            excerpt.append(f'{j + 1:5d}: {text}')

        snippets.append('\n'.join(excerpt))

    return hit_indices, snippets

print('=' * 100)
print('D5 / C3-12 — HISTORICAL LIME CODE AND CONFIGURATION AUDIT')
print('=' * 100)
print('Search is restricted to specified thesis directories.')
print('Existing notebooks and source files will NOT be executed.')

all_paths = {}
root_summaries = []

for label, root, max_depth in SEARCH_ROOTS:
    files, capped = discover_source_files(
        root, max_depth
    )

    print('\nSEARCH ROOT:', label)
    print('Directory:', root)
    print('Exists:', root.is_dir())
    print('Candidate source files:', len(files))
    print('File-search cap reached:', capped)

    root_summaries.append({
        'root': label,
        'exists': root.is_dir(),
        'candidate_files': len(files),
        'capped': capped
    })

    for path in files:
        all_paths[str(path)] = path

print('\nUnique candidate source files:', len(all_paths))

for path in sorted(all_paths.values(), key=str):
    lines = read_source_lines(path)
    hits, snippets = matching_snippets(lines)

    if not hits:
        continue

    D5_CODE_AUDIT.append({
        'path': str(path),
        'hit_count': len(hits),
        'snippets': snippets
    })

print('\n' + '=' * 100)
print('FILES CONTAINING RELEVANT LIME CONFIGURATION CODE')
print('=' * 100)
print('Matched source files:', len(D5_CODE_AUDIT))

for record in D5_CODE_AUDIT[:MAX_MATCHED_FILES_SHOWN]:
    print('\n' + '-' * 100)
    print('FILE:', record['path'])
    print('Matching lines:', record['hit_count'])

    for snippet in record['snippets']:
        print('\n' + snippet)

if len(D5_CODE_AUDIT) > MAX_MATCHED_FILES_SHOWN:
    print(
        '\nAdditional matching files not printed:',
        len(D5_CODE_AUDIT) - MAX_MATCHED_FILES_SHOWN
    )

print('\n' + '=' * 100)
print('PROVENANCE AUDIT SUMMARY')
print('=' * 100)

if D5_CODE_AUDIT:
    print('STATUS: SOURCE REFERENCES FOUND')
    print('Review snippets for LIME mode, data background,')
    print('prediction function, random state and num_samples.')
else:
    print('STATUS: HISTORICAL SOURCE NOT LOCATED IN SEARCH ROOTS')
    print('Do not claim an exact historical LIME reproduction.')
    print('A new explicitly configured, frozen-model')
    print('regression-mode LIME analysis may still be conducted.')

print('\nNo model trained, fitted, or loaded.')
print('No explanation calculation performed.')
print('No files created or modified.')
print('C3-12 remains IN PROGRESS.')
print('=' * 100)


In [ ]:

# ============================================================
# D5 / C3-12 — CELL 5
# ORIGINAL EXPLANATION INPUT RECOVERY
#
# Verify selected-window identities, scores and 23 scaled
# features against frozen historical scoring CSVs.
#
# READ ONLY — no model fitting, SHAP, LIME or file writes.
# ============================================================

from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd

BASE = Path('/content/drive/MyDrive/Master/Thesis/Code')
EXP = BASE / 'experiments'

FEATURES = [
    'energy', 'first_last_diff', 'iqr', 'kurtosis', 'mad',
    'max', 'max_abs_diff', 'mean', 'mean_abs_diff',
    'median', 'min', 'n_points', 'peak_count',
    'q25', 'q75', 'range', 'rms', 'skew',
    'slope', 'std', 'std_diff', 'sum', 'var'
]

SCALED = [f'{feature}_z' for feature in FEATURES]

CONFIG = {
    'UK-DALE': {
        'score_file': (
            EXP / 'EXP_SPLIT_BOUNDARY_PURGE_CORRECTION'
            / 'UKDALE/ukdale_if_scored_boundary_corrected.csv'
        ),
        'shap_file': (
            EXP / 'EXP_UKDALE_Selected_Pipeline/xai'
            / 'ukdale_shap_explanations_selected_test_windows.csv'
        ),
        'keys': [
            'building', 'meter',
            'window_start', 'window_end'
        ],
        'scaled_feature_column': 'feature_scaled',
        'expected_windows': 5,
        'expected_score_rows': 363909,
        'expected_train_rows': 218358
    },
    'AMPds2': {
        'score_file': (
            EXP / 'EXP_SPLIT_BOUNDARY_PURGE_CORRECTION'
            / 'AMPDS2/ampds2_if_scored_boundary_corrected.csv'
        ),
        'shap_file': (
            EXP / 'EXP_AMPDS2_Selected_Pipeline/xai'
            / 'ampds2_shap_explanations_selected_test_windows.csv'
        ),
        'keys': [
            'candidate_id', 'channel_code',
            'window_start', 'window_end'
        ],
        'scaled_feature_column': 'scaled_feature',
        'expected_windows': 8,
        'expected_score_rows': 735777,
        'expected_train_rows': 441483
    }
}

def time_key(value):
    """
    Compare existing UTC timestamp representations without
    modifying the underlying historical timestamp.
    The first 19 characters represent YYYY-MM-DD HH:MM:SS.
    """
    return str(value)[:19].replace('T', ' ')

def build_targets(shap, cfg):
    targets = []

    for identity, group in shap.groupby(
        cfg['keys'], sort=False, dropna=False
    ):
        group = group.copy()

        if len(group) != 23:
            raise RuntimeError(
                f'STOP: Expected 23 SHAP features: {identity}'
            )

        if set(group['feature']) != set(FEATURES):
            raise RuntimeError(
                f'STOP: SHAP feature universe mismatch: {identity}'
            )

        original_scaled_names = (
            group[cfg['scaled_feature_column']]
            .astype(str)
            .tolist()
        )

        if set(original_scaled_names) != set(SCALED):
            raise RuntimeError(
                f'STOP: Scaled-feature names mismatch: {identity}'
            )

        values = (
            group.set_index('feature')
            .loc[FEATURES, 'feature_value_scaled']
            .to_numpy(dtype=np.float64)
        )

        if not np.isfinite(values).all():
            raise RuntimeError(
                f'STOP: Nonfinite original feature: {identity}'
            )

        saved_scores = group['anomaly_score'].to_numpy(
            dtype=np.float64
        )

        if not np.isfinite(saved_scores).all():
            raise RuntimeError(
                f'STOP: Invalid original score: {identity}'
            )

        if not np.allclose(
            saved_scores, saved_scores[0],
            rtol=0, atol=1e-12
        ):
            raise RuntimeError(
                f'STOP: Original SHAP score inconsistent: {identity}'
            )

        metadata = dict(zip(cfg['keys'], identity))

        targets.append({
            'identity': identity,
            'metadata': metadata,
            'start_key': time_key(metadata['window_start']),
            'end_key': time_key(metadata['window_end']),
            'anomaly_score': float(saved_scores[0]),
            'scaled_features': values,
            'candidates': []
        })

    return targets

print('=' * 100)
print('D5 / C3-12 — FROZEN EXPLANATION INPUT RECOVERY')
print('=' * 100)

D5_INPUT_AUDIT = {}
all_pass = True

for dataset, cfg in CONFIG.items():

    print('\n' + '=' * 100)
    print('DATASET:', dataset)
    print('=' * 100)

    if not cfg['score_file'].is_file():
        raise RuntimeError(
            'STOP: Missing frozen scoring file: '
            + str(cfg['score_file'])
        )

    if not cfg['shap_file'].is_file():
        raise RuntimeError(
            'STOP: Missing SHAP source: '
            + str(cfg['shap_file'])
        )

    shap = pd.read_csv(cfg['shap_file'], low_memory=False)
    targets = build_targets(shap, cfg)

    if len(targets) != cfg['expected_windows']:
        raise RuntimeError(
            f'STOP: Incorrect selected window count: {dataset}'
        )

    header = pd.read_csv(
        cfg['score_file'], nrows=0
    ).columns.tolist()

    print('\n1. SOURCE SCHEMA')
    print('Frozen scoring CSV:', cfg['score_file'].name)
    print('Columns:', header)

    required = [
        'window_start', 'anomaly_score', 'split'
    ] + SCALED

    missing = [
        column for column in required
        if column not in header
    ]

    if missing:
        print('\nSTATUS: REVIEW')
        print('Missing required fields:', missing)
        print('Timestamp-like columns:', [
            c for c in header
            if 'time' in c.lower()
            or 'date' in c.lower()
            or 'start' in c.lower()
        ])
        raise RuntimeError(
            'STOP: Source schema must be mapped before '
            'recovering explanation inputs.'
        )

    optional = [
        column for column in [
            'window_end',
            'building', 'meter', 'series_key',
            'channel_code', 'candidate_id',
            'predicted_anomaly_corrected'
        ]
        if column in header
    ]

    usecols = list(dict.fromkeys(
        required + optional
    ))

    print('\n2. FROZEN POPULATION SCAN')

    start_keys = {
        target['start_key'] for target in targets
    }

    split_counts = Counter()
    total_rows = 0

    for chunk in pd.read_csv(
        cfg['score_file'],
        usecols=usecols,
        chunksize=75000,
        float_precision='round_trip',
        low_memory=False
    ):
        total_rows += len(chunk)

        split_counts.update(
            chunk['split'].astype(str).value_counts().to_dict()
        )

        matching = chunk.loc[
            chunk['window_start']
            .astype(str)
            .str.slice(0, 19)
            .str.replace('T', ' ', regex=False)
            .isin(start_keys)
        ]

        if matching.empty:
            continue

        for _, row in matching.iterrows():

            start = time_key(row['window_start'])
            end = (
                time_key(row['window_end'])
                if 'window_end' in matching.columns
                else None
            )

            score = float(row['anomaly_score'])

            if not np.isfinite(score):
                continue

            for target in targets:

                if start != target['start_key']:
                    continue

                if (
                    end is not None
                    and end != target['end_key']
                ):
                    continue

                if abs(
                    score - target['anomaly_score']
                ) > 1e-9:
                    continue

                # Check any identifiers shared by both
                # the original SHAP file and scoring CSV.
                shared_identity_fields = [
                    field for field in [
                        'building', 'meter',
                        'channel_code', 'candidate_id'
                    ]
                    if (
                        field in row.index
                        and field in target['metadata']
                    )
                ]

                identity_ok = all(
                    str(row[field]) == str(
                        target['metadata'][field]
                    )
                    for field in shared_identity_fields
                )

                recovered_features = row[SCALED].to_numpy(
                    dtype=np.float64
                )

                max_feature_diff = float(
                    np.max(np.abs(
                        recovered_features
                        - target['scaled_features']
                    ))
                )

                target['candidates'].append({
                    'split': str(row['split']),
                    'score_difference': abs(
                        score - target['anomaly_score']
                    ),
                    'max_feature_difference': max_feature_diff,
                    'identity_ok': identity_ok,
                    'verified_identity_fields':
                        shared_identity_fields,
                    'series_key': (
                        str(row['series_key'])
                        if 'series_key' in row.index
                        else None
                    ),
                    'features': recovered_features
                })

    print('Total scanned rows:', f'{total_rows:,}')
    print('Rows by original split:', dict(split_counts))
    print('Expected total rows:',
          cfg['expected_score_rows'])
    print('Expected training rows:',
          cfg['expected_train_rows'])

    population_ok = (
        total_rows == cfg['expected_score_rows']
        and split_counts.get('train', 0)
        == cfg['expected_train_rows']
    )

    print('Population count verification:',
          'PASS' if population_ok else 'REVIEW')

    print('\n3. SELECTED EXPLANATION WINDOWS')

    recovered = []
    dataset_ok = population_ok

    for target in targets:

        candidates = target['candidates']

        valid = [
            candidate for candidate in candidates
            if (
                candidate['split'] == 'test'
                and candidate['identity_ok']
                and candidate['max_feature_difference'] <= 1e-5
            )
        ]

        print('\nWindow:', target['identity'])
        print('Candidate rows with matching time/score:',
              len(candidates))
        print('Verified test candidates:', len(valid))

        if candidates:
            best = min(
                candidates,
                key=lambda candidate:
                candidate['max_feature_difference']
            )

            print('Best score discrepancy:',
                  f"{best['score_difference']:.12g}")
            print('Best scaled-feature discrepancy:',
                  f"{best['max_feature_difference']:.12g}")
            print('Matched identity fields:',
                  best['verified_identity_fields'])

            if best['series_key'] is not None:
                print('Source series_key:',
                      best['series_key'])

        if len(valid) != 1:
            dataset_ok = False
            print('WINDOW STATUS: REVIEW')
        else:
            recovered.append({
                'identity': target['identity'],
                'original_score': target['anomaly_score'],
                'scaled_features': valid[0]['features'].copy(),
                'max_feature_difference':
                    valid[0]['max_feature_difference']
            })

            print('WINDOW STATUS: PASS')

    print('\n4. DATASET RESULT')
    print('Recovered windows:',
          len(recovered), '/', len(targets))
    print('Source/input consistency:',
          'PASS' if dataset_ok else 'REVIEW')

    D5_INPUT_AUDIT[dataset] = {
        'status': 'PASS' if dataset_ok else 'REVIEW',
        'recovered_windows': recovered,
        'training_rows': split_counts.get('train', 0),
        'total_rows': total_rows,
        'score_file': str(cfg['score_file'])
    }

    all_pass = all_pass and dataset_ok

print('\n' + '=' * 100)

if all_pass:
    print('CELL 5 COMPLETE — ALL 13 ORIGINAL INPUTS VERIFIED')
    print('The frozen scoring files contain matching')
    print('23-dimensional explanation inputs.')
    print('No anomaly detector or explainer was fitted.')
    print('No files were created or modified.')
    print('D5 remains IN PROGRESS.')
else:
    print('CELL 5 COMPLETE — INPUT RECOVERY REQUIRES REVIEW')
    print('Do not compute new explanations until the')
    print('source representation discrepancies are resolved.')

print('=' * 100)


In [ ]:

# ============================================================
# D5 / C3-12 — CELL 6
# FROZEN MODEL VERIFICATION + TRAINING-ONLY LIME REFERENCE
#
# READ ONLY:
# - Verify recovered Isolation Forest model SHA-256
# - Reproduce 13 historical anomaly scores
# - Select 8,192 training-only reference windows per dataset
# - Preserve matrices in notebook memory
#
# NO fitting, SHAP/LIME computations or file writes.
# ============================================================

from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import joblib
import sklearn
from sklearn.ensemble import IsolationForest

BASE = Path('/content/drive/MyDrive/Master/Thesis/Code')

RECOVERY = (
    BASE / 'SUPERVISOR_FINAL_EXPERIMENTS_2026'
    / '03_integrity_checks/IF_RECOVERY'
)

FEATURES_D5 = [
    'energy', 'first_last_diff', 'iqr', 'kurtosis', 'mad',
    'max', 'max_abs_diff', 'mean', 'mean_abs_diff',
    'median', 'min', 'n_points', 'peak_count',
    'q25', 'q75', 'range', 'rms', 'skew',
    'slope', 'std', 'std_diff', 'sum', 'var'
]

SCALED_D5 = [f'{f}_z' for f in FEATURES_D5]

REFERENCE_SIZE = 8192
REFERENCE_SEED = 2026
SCORE_TOLERANCE = 1e-9

D5_C6_CONFIG = {
    'UK-DALE': {
        'model_path': (
            RECOVERY / 'UKDALE'
            / 'ukdale_full_train_IF_FUNCTIONALLY_VERIFIED_'
              '20260928T180619_923276Z.joblib'
        ),
        'expected_sha256':
            'c1695e8c4265f6e90994be88ea7ec645'
            'acba8e10ed6240697aa73703ce3c9c5b',
        'expected_train': 218358,
        'expected_selected': 5,
        'reference_seed': REFERENCE_SEED
    },
    'AMPds2': {
        'model_path': (
            RECOVERY / 'AMPDS2'
            / 'ampds2_full_train_IF_FUNCTIONALLY_VERIFIED_'
              '20260928T181300_250116Z.joblib'
        ),
        'expected_sha256':
            '890a47ce9a2b30902010c78c0084b643'
            'c2f7b6c41257c9655d6be6cb03e2d7ca',
        'expected_train': 441483,
        'expected_selected': 8,
        'reference_seed': REFERENCE_SEED + 1
    }
}

def file_sha256(path):
    digest = hashlib.sha256()

    with path.open('rb') as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024), b''
        ):
            digest.update(block)

    return digest.hexdigest()

def verify_original_model(model, dataset):
    if not isinstance(model, IsolationForest):
        raise RuntimeError(
            f'STOP: Unexpected model type for {dataset}: '
            f'{type(model)}'
        )

    checks = {
        '23 input features':
            getattr(model, 'n_features_in_', None) == 23,
        '300 fitted trees':
            len(model.estimators_) == 300,
        '256 fitted samples per tree':
            int(model.max_samples_) == 256,
        'historical offset -0.5':
            np.isclose(
                float(model.offset_), -0.5,
                atol=1e-12, rtol=0
            )
    }

    print('\nModel configuration:')
    for name, passed in checks.items():
        print(f'  {name}:',
              'PASS' if passed else 'FAIL')

    if not all(checks.values()):
        raise RuntimeError(
            f'STOP: Model configuration mismatch: {dataset}'
        )

def extract_training_reference(
    score_file,
    expected_training_rows,
    sample_size,
    seed
):
    """
    Deterministically select training-only rows by their
    ordinal position within the frozen training population.

    The sample is uniform without replacement. The
    historical scoring file is read but never modified.
    """

    if not score_file.is_file():
        raise RuntimeError(
            f'STOP: Missing scoring file: {score_file}'
        )

    if sample_size > expected_training_rows:
        raise RuntimeError(
            'STOP: Reference sample larger than training set.'
        )

    rng = np.random.default_rng(seed)

    positions = np.sort(
        rng.choice(
            expected_training_rows,
            size=sample_size,
            replace=False
        )
    )

    pieces = []
    training_seen = 0

    for chunk in pd.read_csv(
        score_file,
        usecols=['split'] + SCALED_D5,
        chunksize=75000,
        float_precision='round_trip',
        low_memory=False
    ):
        train_mask = (
            chunk['split'].astype(str) == 'train'
        ).to_numpy()

        n_train = int(train_mask.sum())

        if n_train == 0:
            continue

        lower = np.searchsorted(
            positions, training_seen, side='left'
        )
        upper = np.searchsorted(
            positions, training_seen + n_train,
            side='left'
        )

        if upper > lower:
            local_positions = (
                positions[lower:upper] - training_seen
            )

            training_chunk = chunk.loc[
                train_mask, SCALED_D5
            ]

            selected = training_chunk.iloc[
                local_positions
            ].to_numpy(dtype=np.float32)

            pieces.append(selected)

        training_seen += n_train

    if training_seen != expected_training_rows:
        raise RuntimeError(
            'STOP: Training population count mismatch: '
            f'{training_seen} != {expected_training_rows}'
        )

    if not pieces:
        raise RuntimeError(
            'STOP: No reference windows recovered.'
        )

    reference = np.vstack(pieces)

    if reference.shape != (sample_size, 23):
        raise RuntimeError(
            'STOP: Incorrect reference matrix shape: '
            f'{reference.shape}'
        )

    if not np.isfinite(reference).all():
        raise RuntimeError(
            'STOP: Nonfinite reference feature values.'
        )

    if np.unique(positions).size != sample_size:
        raise RuntimeError(
            'STOP: Duplicate reference row positions.'
        )

    return reference, positions, training_seen

print('=' * 100)
print('D5 / C3-12 — FROZEN MODEL AND LIME REFERENCE AUDIT')
print('=' * 100)

if 'D5_INPUT_AUDIT' not in globals():
    raise RuntimeError(
        'STOP: Cell 5 is required before Cell 6.'
    )

print('scikit-learn:', sklearn.__version__)
print('Reference sample size:', REFERENCE_SIZE)
print('Reference selection: uniform training-only sample')
print('Models will be loaded for inference only.')

prepared = {}

for dataset, cfg in D5_C6_CONFIG.items():

    print('\n' + '=' * 100)
    print('DATASET:', dataset)
    print('=' * 100)

    audit = D5_INPUT_AUDIT.get(dataset)

    if audit is None or audit.get('status') != 'PASS':
        raise RuntimeError(
            f'STOP: Cell 5 not passed for {dataset}'
        )

    recovered = audit['recovered_windows']

    if len(recovered) != cfg['expected_selected']:
        raise RuntimeError(
            f'STOP: Selected-window count mismatch: {dataset}'
        )

    model_path = cfg['model_path']

    if not model_path.is_file():
        raise RuntimeError(
            f'STOP: Frozen model file missing: {model_path}'
        )

    print('Model:', model_path.name)

    original_hash = file_sha256(model_path)

    print('SHA-256:', original_hash)

    if original_hash != cfg['expected_sha256']:
        raise RuntimeError(
            f'STOP: Historical model SHA-256 mismatch: {dataset}'
        )

    # Only load this locally verified, trusted model artifact.
    # joblib files must never be loaded from untrusted sources.
    model = joblib.load(model_path)

    verify_original_model(model, dataset)

    X_selected = np.vstack([
        record['scaled_features']
        for record in recovered
    ]).astype(np.float32)

    historical_scores = np.array([
        record['original_score']
        for record in recovered
    ], dtype=np.float64)

    if (
        X_selected.shape !=
        (cfg['expected_selected'], 23)
        or not np.isfinite(X_selected).all()
        or not np.isfinite(historical_scores).all()
    ):
        raise RuntimeError(
            f'STOP: Invalid selected feature matrix: {dataset}'
        )

    # Original Isolation Forest continuous anomaly score:
    # anomaly_score = -decision_function(X).
    reproduced_scores = (
        -model.decision_function(X_selected)
    )

    score_errors = np.abs(
        reproduced_scores - historical_scores
    )

    max_error = float(score_errors.max())

    print('\n1. HISTORICAL SCORE REPRODUCTION')
    print('Original windows:', len(recovered))
    print('Selected matrix:', X_selected.shape)
    print('Maximum absolute score error:',
          f'{max_error:.16g}')

    for i, record in enumerate(recovered):
        print(
            f'  Window {i+1}: '
            f'historical={historical_scores[i]:.12f}  '
            f'reproduced={reproduced_scores[i]:.12f}  '
            f'error={score_errors[i]:.3e}'
        )

    if max_error > SCORE_TOLERANCE:
        raise RuntimeError(
            f'STOP: Historical score mismatch: {dataset}'
        )

    print('SCORE REPRODUCTION: PASS')

    print('\n2. TRAINING-ONLY REFERENCE EXTRACTION')

    reference, positions, n_training = (
        extract_training_reference(
            Path(audit['score_file']),
            cfg['expected_train'],
            REFERENCE_SIZE,
            cfg['reference_seed']
        )
    )

    reference_hash = hashlib.sha256(
        reference.tobytes(order='C')
    ).hexdigest()

    position_hash = hashlib.sha256(
        positions.astype('<i8').tobytes()
    ).hexdigest()

    print('Training rows scanned:', f'{n_training:,}')
    print('Reference rows selected:', len(reference))
    print('Reference shape:', reference.shape)
    print('Reference dtype:', reference.dtype)
    print('Reference seed:', cfg['reference_seed'])
    print('Reference-matrix SHA-256:', reference_hash)
    print('Sample-position SHA-256:', position_hash)
    print('First five training positions:',
          positions[:5].tolist())

    # Confirm no duplicate feature vectors are being
    # introduced by the sampling procedure itself.
    # Identical source observations may naturally exist.
    print('Finite feature values:',
          bool(np.isfinite(reference).all()))

    if file_sha256(model_path) != original_hash:
        raise RuntimeError(
            f'STOP: Model artifact changed: {dataset}'
        )

    prepared[dataset] = {
        'model': model,
        'model_path': str(model_path),
        'model_sha256': original_hash,
        'score_file': str(audit['score_file']),
        'feature_names': FEATURES_D5.copy(),
        'scaled_feature_names': SCALED_D5.copy(),
        'X_selected': X_selected.copy(),
        'historical_scores': historical_scores.copy(),
        'identities': [
            str(record['identity'])
            for record in recovered
        ],
        'reference': reference.copy(),
        'reference_seed': cfg['reference_seed'],
        'reference_size': REFERENCE_SIZE,
        'reference_sha256': reference_hash,
        'reference_positions_sha256': position_hash,
        'max_score_error': max_error
    }

    print('\nDATASET PREPARATION: PASS')

if set(prepared) != set(D5_C6_CONFIG):
    raise RuntimeError(
        'STOP: Not all datasets were prepared.'
    )

# Publish the complete in-memory result only after
# both dataset audits have passed.
D5_REGRESSION_PREP = prepared

print('\n' + '=' * 100)
print('CELL 6 COMPLETE — FROZEN REGRESSION INPUTS READY')
print('=' * 100)
print('13 historical test windows verified by frozen models.')
print('Training-only LIME reference matrices prepared.')
print('Model files and source scoring artifacts unchanged.')
print('No Isolation Forest fitting performed.')
print('No LIME or SHAP explanations computed.')
print('No files written or overwritten.')
print('C3-12 and C4-12 remain IN PROGRESS.')
print('=' * 100)


In [ ]:

%pip install -q "lime==0.2.0.1"


In [ ]:

from importlib.metadata import version
from lime.lime_tabular import LimeTabularExplainer
import sklearn
import numpy as np

print("LIME version:", version("lime"))
print("scikit-learn version:", sklearn.__version__)
print("NumPy version:", np.__version__)
print("LIME IMPORT: PASS")


In [ ]:

# ============================================================
# D5 / C3-12 — CELL 7
# REGRESSION-MODE LIME PILOT: ONE WINDOW PER DATASET
#
# Uses verified D5_REGRESSION_PREP from Cell 6.
# Historical Isolation Forest models remain frozen.
# LIME fits local surrogate models only.
#
# No historical files modified.
# No new files written.
# ============================================================

import numpy as np
from importlib.metadata import version, PackageNotFoundError

try:
    from lime.lime_tabular import LimeTabularExplainer
except ImportError as exc:
    raise RuntimeError(
        "STOP: LIME is not installed in this Colab runtime. "
        "Report the import error before continuing."
    ) from exc

if 'D5_REGRESSION_PREP' not in globals():
    raise RuntimeError(
        'STOP: Run Cell 6 first.'
    )

try:
    LIME_VERSION = version('lime')
except PackageNotFoundError:
    LIME_VERSION = 'unknown'

NUM_SAMPLES = 5000
NUM_FEATURES = 10
LIME_SEED = 42
N_FEATURES = 23
KERNEL_WIDTH = 0.75 * np.sqrt(N_FEATURES)

def make_score_predictor(model):
    """
    Continuous anomaly-score interface.
    Higher values mean greater anomaly evidence.

    Isolation Forest itself is not refitted.
    """
    def predict(X):
        X = np.asarray(X, dtype=np.float32)

        if X.ndim == 1:
            X = X.reshape(1, -1)

        if X.ndim != 2 or X.shape[1] != N_FEATURES:
            raise ValueError(
                f'Expected an N x {N_FEATURES} matrix, '
                f'got {X.shape}'
            )

        if not np.isfinite(X).all():
            raise ValueError(
                'Nonfinite input supplied to scoring function.'
            )

        scores = -model.decision_function(X)
        scores = np.asarray(scores, dtype=np.float64)

        if scores.shape != (len(X),):
            raise RuntimeError(
                f'Unexpected score shape: {scores.shape}'
            )

        if not np.isfinite(scores).all():
            raise RuntimeError(
                'Nonfinite Isolation Forest scores.'
            )

        return scores

    return predict

print('=' * 100)
print('D5 / C3-12 — REGRESSION LIME PILOT')
print('=' * 100)
print('LIME version:', LIME_VERSION)
print('Mode: regression')
print('Target: negative Isolation Forest decision_function')
print('Features:', N_FEATURES)
print('Training reference rows: 8192 per dataset')
print('LIME samples per explanation:', NUM_SAMPLES)
print('Reported features:', NUM_FEATURES)
print('Discretization: quartile')
print('Kernel width:', f'{KERNEL_WIDTH:.12f}')
print('Random seed:', LIME_SEED)

D5_LIME_PILOT = {}

for dataset in ['UK-DALE', 'AMPds2']:

    print('\n' + '=' * 100)
    print('DATASET:', dataset)
    print('=' * 100)

    prep = D5_REGRESSION_PREP[dataset]

    model = prep['model']
    reference = np.asarray(
        prep['reference'], dtype=np.float32
    )
    selected = np.asarray(
        prep['X_selected'], dtype=np.float32
    )

    if (
        reference.shape != (8192, 23)
        or selected.shape[1] != 23
        or not np.isfinite(reference).all()
        or not np.isfinite(selected).all()
    ):
        raise RuntimeError(
            f'STOP: Invalid verified input matrices: {dataset}'
        )

    feature_names = list(prep['scaled_feature_names'])

    if len(feature_names) != len(set(feature_names)):
        raise RuntimeError(
            f'STOP: Duplicate feature names: {dataset}'
        )

    predict_fn = make_score_predictor(model)

    # Check the score interface before running LIME.
    expected_score = float(prep['historical_scores'][0])
    actual_score = float(
        predict_fn(selected[:1])[0]
    )

    score_error = abs(actual_score - expected_score)

    print('\n1. HISTORICAL SCORE TARGET CHECK')
    print('Selected window:', prep['identities'][0])
    print('Historical score:', f'{expected_score:.15f}')
    print('Prediction function:', f'{actual_score:.15f}')
    print('Absolute discrepancy:', f'{score_error:.3e}')

    if score_error > 1e-9:
        raise RuntimeError(
            f'STOP: Anomaly-score target mismatch: {dataset}'
        )

    # Explicit, consistent configuration.
    explainer = LimeTabularExplainer(
        training_data=reference,
        feature_names=feature_names,
        mode='regression',
        discretize_continuous=True,
        discretizer='quartile',
        categorical_features=[],
        feature_selection='auto',
        kernel_width=KERNEL_WIDTH,
        sample_around_instance=False,
        random_state=LIME_SEED
    )

    print('\n2. FITTING LIME LOCAL SURROGATE')

    explanation = explainer.explain_instance(
        data_row=selected[0],
        predict_fn=predict_fn,
        num_features=NUM_FEATURES,
        num_samples=NUM_SAMPLES
    )

    fidelity_r2 = float(explanation.score)

    if not np.isfinite(fidelity_r2):
        raise RuntimeError(
            f'STOP: Nonfinite local R-squared: {dataset}'
        )

    # R2 can be negative when the local surrogate
    # performs worse than its weighted mean baseline.
    # A low or negative value is evidence to report,
    # not a reason to conceal or overwrite an outcome.
    if fidelity_r2 > 1.0 + 1e-8:
        raise RuntimeError(
            f'STOP: Invalid R-squared above 1: {dataset}'
        )

    local_map = explanation.as_map()

    if 1 not in local_map:
        raise RuntimeError(
            f'STOP: Regression explanation label 1 absent: '
            f'{dataset}, available keys={list(local_map)}'
        )

    ranked = local_map[1]

    if len(ranked) != NUM_FEATURES:
        raise RuntimeError(
            f'STOP: Expected {NUM_FEATURES} LIME coefficients, '
            f'got {len(ranked)} for {dataset}'
        )

    ranked_features = []

    for index, coefficient in ranked:
        index = int(index)
        coefficient = float(coefficient)

        if not 0 <= index < N_FEATURES:
            raise RuntimeError(
                f'STOP: Invalid LIME feature index: {index}'
            )

        if not np.isfinite(coefficient):
            raise RuntimeError(
                'STOP: Nonfinite LIME coefficient.'
            )

        ranked_features.append({
            'feature': prep['feature_names'][index],
            'feature_scaled': feature_names[index],
            'coefficient': coefficient,
            'abs_coefficient': abs(coefficient)
        })

    if len({
        item['feature']
        for item in ranked_features
    }) != NUM_FEATURES:
        raise RuntimeError(
            f'STOP: Duplicate LIME feature identities: {dataset}'
        )

    print('Local surrogate R-squared:',
          f'{fidelity_r2:.9f}')

    predicted_value = getattr(
        explanation, 'predicted_value', None
    )

    if predicted_value is not None:
        predicted_array = np.asarray(
            predicted_value, dtype=np.float64
        ).reshape(-1)

        print('LIME original target prediction:',
              predicted_array.tolist())

        if (
            predicted_array.size != 1
            or not np.isclose(
                predicted_array[0],
                actual_score,
                atol=1e-9,
                rtol=0
            )
        ):
            raise RuntimeError(
                f'STOP: LIME original target mismatch: {dataset}'
            )

    local_prediction = getattr(
        explanation, 'local_pred', None
    )

    if local_prediction is not None:
        local_array = np.asarray(
            local_prediction, dtype=np.float64
        ).reshape(-1)

        print('Local surrogate prediction:',
              local_array.tolist())

    print('\n3. TOP TEN REGRESSION-MODE LIME FEATURES')

    for rank, item in enumerate(ranked_features, start=1):
        print(
            f"{rank:2d}. "
            f"{item['feature']:19s} "
            f"weight={item['coefficient']:+.9f}"
        )

    D5_LIME_PILOT[dataset] = {
        'identity': prep['identities'][0],
        'mode': 'regression',
        'target': '-IsolationForest.decision_function',
        'num_samples': NUM_SAMPLES,
        'num_features': NUM_FEATURES,
        'seed': LIME_SEED,
        'kernel_width': KERNEL_WIDTH,
        'historical_score': expected_score,
        'score_error': score_error,
        'local_r2': fidelity_r2,
        'ranked_features': ranked_features
    }

    print('\nPILOT STATUS: PASS')

if set(D5_LIME_PILOT) != {'UK-DALE', 'AMPds2'}:
    raise RuntimeError(
        'STOP: Both dataset pilots are required.'
    )

print('\n' + '=' * 100)
print('CELL 7 COMPLETE — REGRESSION LIME PILOT PASSED')
print('=' * 100)

for dataset, result in D5_LIME_PILOT.items():
    print(
        f"{dataset}: "
        f"R2={result['local_r2']:.9f}, "
        f"score_error={result['score_error']:.3e}"
    )

print('\nOriginal Isolation Forest models were not refitted.')
print('Only new local LIME surrogates were fitted.')
print('Historical artifacts unchanged.')
print('No results written to disk.')
print('C3-12 and C4-12 remain IN PROGRESS.')
print('=' * 100)


In [ ]:

# ============================================================
# D5 / C3-12 — CELL 8
# COMPLETE REGRESSION-MODE LIME FIDELITY EVALUATION
#
# 13 original test windows:
#   UK-DALE: 5
#   AMPds2:  8
#
# Frozen Isolation Forest models: inference only
# LIME: new local linear surrogates
# No historical artifacts modified
# No files written
# ============================================================

import hashlib
import numpy as np
import pandas as pd
from lime.lime_tabular import LimeTabularExplainer

if 'D5_REGRESSION_PREP' not in globals():
    raise RuntimeError(
        'STOP: Cell 6 preparation is unavailable.'
    )

if 'D5_LIME_PILOT' not in globals():
    raise RuntimeError(
        'STOP: Run Cell 7 before Cell 8.'
    )

N_FEATURES = 23
NUM_FEATURES = 10
NUM_SAMPLES = 5000
LIME_SEED = 42
KERNEL_WIDTH = 0.75 * np.sqrt(N_FEATURES)

EXPECTED_WINDOWS = {
    'UK-DALE': 5,
    'AMPds2': 8
}

def d5_predict(model, X):
    X = np.asarray(X, dtype=np.float32)

    if X.ndim == 1:
        X = X.reshape(1, -1)

    if X.ndim != 2 or X.shape[1] != 23:
        raise RuntimeError(
            f'Invalid prediction input shape: {X.shape}'
        )

    if not np.isfinite(X).all():
        raise RuntimeError(
            'Nonfinite input to Isolation Forest.'
        )

    scores = np.asarray(
        -model.decision_function(X),
        dtype=np.float64
    )

    if scores.shape != (len(X),):
        raise RuntimeError(
            'Unexpected anomaly-score output shape.'
        )

    if not np.isfinite(scores).all():
        raise RuntimeError(
            'Nonfinite anomaly scores.'
        )

    return scores


def explain_one_window(prep, index):

    X = np.asarray(
        prep['X_selected'], dtype=np.float32
    )

    reference = np.asarray(
        prep['reference'], dtype=np.float32
    )

    model = prep['model']

    feature_names = list(
        prep['scaled_feature_names']
    )

    if reference.shape != (8192, 23):
        raise RuntimeError(
            'Training reference shape mismatch.'
        )

    reference_sha = hashlib.sha256(
        reference.tobytes(order='C')
    ).hexdigest()

    if reference_sha != prep['reference_sha256']:
        raise RuntimeError(
            'STOP: Training-reference fingerprint changed.'
        )

    if X.shape[1] != 23:
        raise RuntimeError(
            'Selected feature dimension mismatch.'
        )

    target = float(
        prep['historical_scores'][index]
    )

    direct_score = float(
        d5_predict(model, X[index:index+1])[0]
    )

    if abs(target - direct_score) > 1e-9:
        raise RuntimeError(
            'STOP: Frozen-model score mismatch.'
        )

    # Each window receives an independently initialized
    # explainer using the same deterministic seed.
    # This makes results independent of execution order.
    explainer = LimeTabularExplainer(
        training_data=reference,
        feature_names=feature_names,
        mode='regression',
        discretize_continuous=True,
        discretizer='quartile',
        categorical_features=[],
        feature_selection='auto',
        kernel_width=KERNEL_WIDTH,
        sample_around_instance=False,
        random_state=LIME_SEED
    )

    explanation = explainer.explain_instance(
        data_row=X[index],
        predict_fn=lambda values: d5_predict(
            model, values
        ),
        num_features=NUM_FEATURES,
        num_samples=NUM_SAMPLES
    )

    local_r2 = float(explanation.score)

    if (
        not np.isfinite(local_r2)
        or local_r2 > 1.0 + 1e-8
    ):
        raise RuntimeError(
            'STOP: Invalid local R-squared.'
        )

    original_prediction = np.asarray(
        explanation.predicted_value,
        dtype=np.float64
    ).reshape(-1)

    surrogate_prediction = np.asarray(
        explanation.local_pred,
        dtype=np.float64
    ).reshape(-1)

    if (
        original_prediction.size != 1
        or surrogate_prediction.size != 1
        or not np.isfinite(original_prediction).all()
        or not np.isfinite(surrogate_prediction).all()
    ):
        raise RuntimeError(
            'STOP: Invalid explanation prediction.'
        )

    if abs(original_prediction[0] - target) > 1e-9:
        raise RuntimeError(
            'STOP: LIME explained an unexpected target.'
        )

    local_map = explanation.as_map()

    if 1 not in local_map:
        raise RuntimeError(
            'STOP: Missing LIME regression label.'
        )

    coefficients = local_map[1]

    if len(coefficients) != NUM_FEATURES:
        raise RuntimeError(
            'STOP: Unexpected LIME feature count.'
        )

    ranked = []

    for rank, (feature_index, weight) in enumerate(
        coefficients, start=1
    ):
        feature_index = int(feature_index)
        weight = float(weight)

        if not 0 <= feature_index < N_FEATURES:
            raise RuntimeError(
                'STOP: Invalid feature index.'
            )

        if not np.isfinite(weight):
            raise RuntimeError(
                'STOP: Nonfinite LIME weight.'
            )

        ranked.append({
            'rank': rank,
            'feature': prep['feature_names'][feature_index],
            'feature_scaled':
                feature_names[feature_index],
            'weight': weight,
            'abs_weight': abs(weight)
        })

    selected_features = [
        item['feature'] for item in ranked
    ]

    if len(set(selected_features)) != NUM_FEATURES:
        raise RuntimeError(
            'STOP: Duplicate LIME feature identity.'
        )

    # LIME's local_exp entries should be ordered by
    # descending coefficient magnitude.
    magnitudes = [
        item['abs_weight'] for item in ranked
    ]

    if any(
        magnitudes[j] + 1e-12 < magnitudes[j + 1]
        for j in range(len(magnitudes) - 1)
    ):
        raise RuntimeError(
            'STOP: Unexpected LIME feature ranking order.'
        )

    predicted_locally = float(
        surrogate_prediction[0]
    )

    return {
        'identity': prep['identities'][index],
        'historical_score': target,
        'lime_target_prediction':
            float(original_prediction[0]),
        'surrogate_prediction': predicted_locally,
        'local_r2': local_r2,
        'absolute_local_error':
            abs(predicted_locally - target),
        'ranked_features': ranked,
        'top5': selected_features[:5],
        'top10': selected_features,
        'seed': LIME_SEED,
        'num_samples': NUM_SAMPLES,
        'num_features': NUM_FEATURES
    }


print('=' * 100)
print('D5 / C3-12 — COMPLETE REGRESSION LIME FIDELITY')
print('=' * 100)

print('Target: -IsolationForest.decision_function')
print('Mode: regression')
print('Reference: 8192 training windows per dataset')
print('Samples per explanation:', NUM_SAMPLES)
print('Feature count:', NUM_FEATURES)
print('Seed per independent explanation:', LIME_SEED)
print('Kernel width:', f'{KERNEL_WIDTH:.12f}')
print('Output: in memory only')

full_results = {}
summary_records = []

for dataset, expected_count in EXPECTED_WINDOWS.items():

    print('\n' + '=' * 100)
    print('DATASET:', dataset)
    print('=' * 100)

    prep = D5_REGRESSION_PREP[dataset]

    if len(prep['identities']) != expected_count:
        raise RuntimeError(
            f'STOP: Unexpected window count for {dataset}'
        )

    records = []

    for index in range(expected_count):

        result = explain_one_window(prep, index)
        records.append(result)

        print(
            f"\nWindow {index + 1}/{expected_count}: "
            f"{result['identity']}"
        )
        print(
            f"  R2: {result['local_r2']:.9f}"
        )
        print(
            "  Detector score: "
            f"{result['historical_score']:.9f}"
        )
        print(
            "  Surrogate prediction: "
            f"{result['surrogate_prediction']:.9f}"
        )
        print(
            "  Absolute local error: "
            f"{result['absolute_local_error']:.9f}"
        )
        print(
            "  Top five: "
            + ', '.join(result['top5'])
        )

    # Validate independent rerun against the pilot.
    pilot = D5_LIME_PILOT[dataset]

    pilot_r2_error = abs(
        records[0]['local_r2']
        - float(pilot['local_r2'])
    )

    print('\nPILOT REPRODUCIBILITY CHECK')
    print('R2 discrepancy:',
          f'{pilot_r2_error:.12g}')

    if pilot_r2_error > 1e-8:
        raise RuntimeError(
            f'STOP: Pilot R2 not reproduced: {dataset}'
        )

    if (
        records[0]['top10']
        != [
            row['feature']
            for row in pilot['ranked_features']
        ]
    ):
        raise RuntimeError(
            f'STOP: Pilot feature ranking not reproduced: {dataset}'
        )

    full_results[dataset] = records

    r2_values = np.array([
        r['local_r2'] for r in records
    ])

    errors = np.array([
        r['absolute_local_error'] for r in records
    ])

    summary_records.append({
        'dataset': dataset,
        'n_windows': len(records),
        'mean_r2': float(np.mean(r2_values)),
        'median_r2': float(np.median(r2_values)),
        'min_r2': float(np.min(r2_values)),
        'max_r2': float(np.max(r2_values)),
        'mean_absolute_local_error':
            float(np.mean(errors)),
        'max_absolute_local_error':
            float(np.max(errors)),
        'count_r2_below_0_5':
            int(np.sum(r2_values < 0.5)),
        'count_negative_r2':
            int(np.sum(r2_values < 0))
    })

    print('\nDATASET FIDELITY SUMMARY')
    print(
        f"Mean R2: {np.mean(r2_values):.6f}"
    )
    print(
        f"Median R2: {np.median(r2_values):.6f}"
    )
    print(
        f"R2 range: "
        f"{np.min(r2_values):.6f} to "
        f"{np.max(r2_values):.6f}"
    )
    print(
        "Mean absolute local prediction error: "
        f"{np.mean(errors):.6f}"
    )
    print(
        "Windows with R2 below 0.5: "
        f"{int(np.sum(r2_values < 0.5))}/{len(records)}"
    )

if (
    len(full_results['UK-DALE']) != 5
    or len(full_results['AMPds2']) != 8
):
    raise RuntimeError(
        'STOP: Missing explanations.'
    )

D5_LIME_FULL = full_results
D5_LIME_FIDELITY_SUMMARY = pd.DataFrame(
    summary_records
)

print('\n' + '=' * 100)
print('COMBINED FIDELITY SUMMARY')
print('=' * 100)

print(
    D5_LIME_FIDELITY_SUMMARY.to_string(
        index=False,
        float_format=lambda v: f'{v:.6f}'
    )
)

all_records = (
    D5_LIME_FULL['UK-DALE']
    + D5_LIME_FULL['AMPds2']
)

print('\nOverall mean R2:',
      f"{np.mean([r['local_r2'] for r in all_records]):.6f}")

print('Overall mean absolute local prediction error:',
      f"{np.mean([r['absolute_local_error'] for r in all_records]):.6f}")

print('\n' + '=' * 100)
print('CELL 8 COMPLETE — 13 REGRESSION LIME EXPLANATIONS')
print('=' * 100)
print('All 13 local fidelity scores computed.')
print('All 13 pointwise surrogate errors computed.')
print('Pilot reproducibility verified for both datasets.')
print('Original Isolation Forest models not fitted.')
print('No historical artifacts modified.')
print('No files written.')
print('C3-12 remains OPEN pending interpretation/evidence preservation.')
print('C4-12 remains OPEN pending same-window comparison.')
print('=' * 100)


In [ ]:

# ============================================================
# D5 / C4-12 — CELL 9
# NEW REGRESSION LIME vs HISTORICAL LOCAL KERNEL SHAP
#
# Window-matched comparison:
#   - Top-five Jaccard similarity
#   - Spearman rank correlation on the ten LIME features
#   - Old-versus-new LIME ranking change (diagnostic)
#
# READ ONLY / MEMORY ONLY
# No model fitting, explainer runs or file writes
#
# IMPORTANT: Kernel SHAP target provenance is pending.
# These comparisons are PROVISIONAL, not final results.
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

if 'D5_LIME_FULL' not in globals():
    raise RuntimeError('STOP: Run Cell 8 first.')

if 'D5_PAIRED' not in globals():
    raise RuntimeError('STOP: Run Cell 2 first.')

EXPECTED = {'UK-DALE': 5, 'AMPds2': 8}

def top5_jaccard(a, b):
    a, b = set(a), set(b)

    if len(a) != 5 or len(b) != 5:
        raise RuntimeError('STOP: Invalid top-five feature set.')

    return len(a & b) / len(a | b)

def validate_features(values, expected_n, context):
    values = list(values)

    if (
        len(values) != expected_n
        or len(set(values)) != expected_n
        or not all(isinstance(f, str) for f in values)
    ):
        raise RuntimeError(
            f'STOP: Invalid feature ordering for {context}'
        )

    return values

print('=' * 100)
print('D5 / C4-12 — REGRESSION LIME / KERNEL SHAP AGREEMENT')
print('PROVISIONAL: HISTORICAL KERNEL SHAP TARGET AUDIT PENDING')
print('=' * 100)

results = []

for dataset, expected_n in EXPECTED.items():

    print('\n' + '=' * 100)
    print('DATASET:', dataset)
    print('=' * 100)

    new_records = D5_LIME_FULL[dataset]
    historical = D5_PAIRED[dataset]

    if len(new_records) != expected_n or len(historical) != expected_n:
        raise RuntimeError(
            f'STOP: Incorrect window count for {dataset}'
        )

    # Match by the complete verified window identity,
    # never by array position or category.
    historical_map = {
        str(row['identity']): row
        for _, row in historical.iterrows()
    }

    new_map = {
        str(row['identity']): row
        for row in new_records
    }

    if (
        len(historical_map) != expected_n
        or len(new_map) != expected_n
        or set(historical_map) != set(new_map)
    ):
        raise RuntimeError(
            f'STOP: Window alignment mismatch for {dataset}'
        )

    for identity in sorted(new_map):

        new = new_map[identity]
        old = historical_map[identity]

        score_difference = abs(
            float(new['historical_score'])
            - float(old['shap_score'])
        )

        if score_difference > 1e-9:
            raise RuntimeError(
                f'STOP: Score mismatch for {identity}'
            )

        new_top10 = validate_features(
            new['top10'], 10, identity
        )

        shap_top23 = validate_features(
            old['shap_all_ranked_features'],
            23,
            identity
        )

        old_lime_top10 = validate_features(
            old['lime_all_ranked_features'],
            10,
            identity
        )

        new_top5 = new_top10[:5]
        shap_top5 = shap_top23[:5]
        old_lime_top5 = old_lime_top10[:5]

        if new_top5 != list(new['top5']):
            raise RuntimeError(
                f'STOP: New LIME rank mismatch: {identity}'
            )

        if shap_top5 != list(old['shap_top5']):
            raise RuntimeError(
                f'STOP: Historical SHAP rank mismatch: {identity}'
            )

        if not set(new_top10).issubset(set(shap_top23)):
            raise RuntimeError(
                f'STOP: Incompatible feature universes: {identity}'
            )

        # Full 23-feature SHAP ranking.
        shap_rank = {
            feature: rank
            for rank, feature in enumerate(
                shap_top23, start=1
            )
        }

        # Compare SHAP positions for the ten
        # features explicitly ranked by new LIME.
        lime_rank_positions = np.arange(
            1, 11, dtype=np.float64
        )

        shap_rank_positions = np.array(
            [shap_rank[f] for f in new_top10],
            dtype=np.float64
        )

        rho = float(
            spearmanr(
                lime_rank_positions,
                shap_rank_positions
            ).statistic
        )

        if not np.isfinite(rho):
            raise RuntimeError(
                f'STOP: Undefined rank correlation: {identity}'
            )

        shared = set(new_top5) & set(shap_top5)

        new_jaccard = top5_jaccard(
            new_top5, shap_top5
        )

        # Changes introduced by the new LIME protocol.
        old_vs_new_jaccard = top5_jaccard(
            old_lime_top5, new_top5
        )

        row = {
            'dataset': dataset,
            'identity': identity,
            'anomaly_score':
                float(new['historical_score']),
            'lime_local_r2':
                float(new['local_r2']),
            'lime_absolute_local_error':
                float(new['absolute_local_error']),
            'shared_top5_count': len(shared),
            'top5_jaccard': new_jaccard,
            'spearman_top10': rho,
            'historical_vs_new_lime_jaccard':
                old_vs_new_jaccard,
            'shap_top5': ', '.join(shap_top5),
            'new_lime_top5': ', '.join(new_top5),
            'historical_lime_top5':
                ', '.join(old_lime_top5),
            'shared_features':
                ', '.join(sorted(shared)),
            'interpretation_status':
                'provisional_kernel_shap_target_pending'
        }

        results.append(row)

        print('\nWindow:', identity)
        print('  Local LIME R2:',
              f"{row['lime_local_r2']:.6f}")
        print('  Shared top-five features:',
              row['shared_top5_count'])
        print('  New SHAP/LIME Jaccard:',
              f'{new_jaccard:.6f}')
        print('  New SHAP/LIME Spearman rho:',
              f'{rho:.6f}')
        print('  Historical/new LIME Jaccard:',
              f'{old_vs_new_jaccard:.6f}')
        print('  SHAP top five:', row['shap_top5'])
        print('  Regression LIME top five:',
              row['new_lime_top5'])

D5_REGRESSION_AGREEMENT = pd.DataFrame(results)

if len(D5_REGRESSION_AGREEMENT) != 13:
    raise RuntimeError(
        'STOP: Exactly 13 comparisons required.'
    )

print('\n' + '=' * 100)
print('DATASET-LEVEL DESCRIPTIVE SUMMARY')
print('=' * 100)

summary = (
    D5_REGRESSION_AGREEMENT
    .groupby('dataset', sort=False)
    .agg(
        n_windows=('identity', 'count'),
        mean_jaccard=('top5_jaccard', 'mean'),
        median_jaccard=('top5_jaccard', 'median'),
        min_jaccard=('top5_jaccard', 'min'),
        max_jaccard=('top5_jaccard', 'max'),
        mean_spearman=('spearman_top10', 'mean'),
        median_spearman=('spearman_top10', 'median'),
        mean_local_r2=('lime_local_r2', 'mean'),
        mean_historical_vs_new_lime_jaccard=(
            'historical_vs_new_lime_jaccard', 'mean'
        )
    )
    .reset_index()
)

D5_REGRESSION_AGREEMENT_SUMMARY = summary

print(summary.to_string(
    index=False,
    float_format=lambda value: f'{value:.6f}'
))

print('\n' + '=' * 100)
print('COMBINED 13-WINDOW SUMMARY')
print('=' * 100)

print('Mean Jaccard:',
      f"{D5_REGRESSION_AGREEMENT['top5_jaccard'].mean():.6f}")

print('Mean Spearman:',
      f"{D5_REGRESSION_AGREEMENT['spearman_top10'].mean():.6f}")

print('Zero-overlap windows:',
      int((D5_REGRESSION_AGREEMENT['shared_top5_count'] == 0).sum()))

print('Negative rank correlations:',
      int((D5_REGRESSION_AGREEMENT['spearman_top10'] < 0).sum()))

print('\nINTERPRETATION RESTRICTIONS')
print('- Selected illustrative windows, not random samples.')
print('- Absolute attribution magnitude rankings only.')
print('- Spearman uses ten LIME-selected feature identities.')
print('- No explanation stability or significance inferred.')
print('- LIME local fidelity is reported separately.')
print('- Kernel SHAP prediction target still requires verification.')
print('- Global TreeSHAP path-length attributions are NOT used.')

print('\n' + '=' * 100)
print('CELL 9 COMPLETE — PROVISIONAL AGREEMENT COMPUTED')
print('13/13 original windows compared.')
print('No historical results modified or overwritten.')
print('No models or explainers fitted.')
print('No files written.')
print('C3-12 and C4-12 remain OPEN.')
print('=' * 100)


In [ ]:

# ============================================================
# D5 / C4-12 — CELL 10
# HISTORICAL KERNEL SHAP OUTPUT / BASELINE AUDIT
#
# NO SHAP OR LIME RECOMPUTATION
# NO MODEL TRAINING
# NO FILE WRITES
#
# Tests:
#  1. Saved SHAP attribution additivity
#  2. Implied baseline consistency across original windows
#  3. Independent 100-window training background
#  4. Compatibility with continuous anomaly-score target
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

BASE = Path('/content/drive/MyDrive/Master/Thesis/Code')
EXP = BASE / 'experiments'

if 'D5_REGRESSION_PREP' not in globals():
    raise RuntimeError('STOP: Cell 6 is required.')

SHAP_CONFIG = {
    'UK-DALE': {
        'path': (
            EXP / 'EXP_UKDALE_Selected_Pipeline/xai'
            / 'ukdale_shap_explanations_selected_test_windows.csv'
        ),
        'keys': [
            'building', 'meter',
            'window_start', 'window_end'
        ],
        'expected_windows': 5,
        'expected_training': 218358
    },
    'AMPds2': {
        'path': (
            EXP / 'EXP_AMPDS2_Selected_Pipeline/xai'
            / 'ampds2_shap_explanations_selected_test_windows.csv'
        ),
        'keys': [
            'candidate_id', 'channel_code',
            'window_start', 'window_end'
        ],
        'expected_windows': 8,
        'expected_training': 441483
    }
}

BACKGROUND_SIZE = 100
BACKGROUND_SEED = 42

# Numerical criteria are diagnostic, not permission to
# rewrite or adjust historical attribution values.
BASELINE_TOLERANCE = 1e-6
BACKGROUND_TOLERANCE = 1e-6

def get_documented_background(
    score_file,
    feature_columns,
    expected_training
):
    rng = np.random.default_rng(BACKGROUND_SEED)

    positions = np.sort(
        rng.choice(
            expected_training,
            size=BACKGROUND_SIZE,
            replace=False
        )
    )

    selected_parts = []
    training_seen = 0

    for chunk in pd.read_csv(
        score_file,
        usecols=['split'] + feature_columns,
        chunksize=75000,
        float_precision='round_trip',
        low_memory=False
    ):

        train = chunk.loc[
            chunk['split'].astype(str).eq('train'),
            feature_columns
        ]

        count = len(train)

        low = np.searchsorted(
            positions, training_seen, side='left'
        )

        high = np.searchsorted(
            positions, training_seen + count,
            side='left'
        )

        if high > low:
            selected_parts.append(
                train.iloc[
                    positions[low:high] - training_seen
                ].to_numpy(dtype=np.float32)
            )

        training_seen += count

    if training_seen != expected_training:
        raise RuntimeError(
            'STOP: Training population count mismatch.'
        )

    background = np.vstack(selected_parts)

    if background.shape != (BACKGROUND_SIZE, 23):
        raise RuntimeError(
            f'STOP: Invalid background shape {background.shape}'
        )

    if not np.isfinite(background).all():
        raise RuntimeError(
            'STOP: Nonfinite background values.'
        )

    return background

D5_KERNEL_SHAP_AUDIT = {}

print('=' * 100)
print('D5 / C4-12 — HISTORICAL KERNEL SHAP BASELINE AUDIT')
print('=' * 100)

for dataset, cfg in SHAP_CONFIG.items():

    print('\n' + '=' * 100)
    print('DATASET:', dataset)
    print('=' * 100)

    prep = D5_REGRESSION_PREP[dataset]
    shap_path = cfg['path']

    if not shap_path.is_file():
        raise RuntimeError(
            f'STOP: Missing historical SHAP CSV: {shap_path}'
        )

    shap = pd.read_csv(
        shap_path,
        float_precision='round_trip',
        low_memory=False
    )

    necessary = (
        cfg['keys']
        + ['feature', 'shap_value', 'anomaly_score']
    )

    missing = [
        column for column in necessary
        if column not in shap.columns
    ]

    if missing:
        raise RuntimeError(
            f'STOP: SHAP columns missing: {missing}'
        )

    # Verify every original explanation has one value
    # for each of the canonical 23 engineered features.
    records = []

    for identity, group in shap.groupby(
        cfg['keys'], sort=False, dropna=False
    ):

        if len(group) != 23:
            raise RuntimeError(
                f'STOP: Expected 23 SHAP rows for {identity}'
            )

        if set(group['feature']) != set(
            prep['feature_names']
        ):
            raise RuntimeError(
                f'STOP: SHAP feature mismatch for {identity}'
            )

        shap_values = pd.to_numeric(
            group['shap_value'],
            errors='raise'
        ).to_numpy(dtype=np.float64)

        scores = pd.to_numeric(
            group['anomaly_score'],
            errors='raise'
        ).to_numpy(dtype=np.float64)

        if (
            not np.isfinite(shap_values).all()
            or not np.isfinite(scores).all()
        ):
            raise RuntimeError(
                f'STOP: Nonfinite SHAP or score for {identity}'
            )

        if np.ptp(scores) > 1e-12:
            raise RuntimeError(
                f'STOP: Inconsistent score for {identity}'
            )

        score = float(scores[0])
        phi_sum = float(np.sum(shap_values))

        # If these SHAP values explain this score, the
        # implied expected output is score - sum(phi).
        implied_baseline = score - phi_sum

        records.append({
            'identity': str(identity),
            'score': score,
            'shap_sum': phi_sum,
            'implied_baseline': implied_baseline
        })

    if len(records) != cfg['expected_windows']:
        raise RuntimeError(
            f'STOP: Unexpected SHAP window count for {dataset}'
        )

    implied = np.array(
        [row['implied_baseline'] for row in records],
        dtype=np.float64
    )

    baseline_mean = float(np.mean(implied))
    baseline_range = float(np.ptp(implied))

    print('\n1. SAVED SHAP ADDITIVITY')
    for row in records:
        print('\nWindow:', row['identity'])
        print('  Saved detector score:',
              f"{row['score']:.12f}")
        print('  Sum of SHAP values:',
              f"{row['shap_sum']:.12f}")
        print('  Implied SHAP baseline:',
              f"{row['implied_baseline']:.12f}")

    print('\nMean implied baseline:',
          f'{baseline_mean:.12f}')
    print('Implied baseline range:',
          f'{baseline_range:.12g}')

    baseline_consistent = (
        baseline_range <= BASELINE_TOLERANCE
    )

    print('Baseline consistency:',
          'PASS' if baseline_consistent else 'REVIEW')

    print('\n2. DOCUMENTED BACKGROUND RECONSTRUCTION')
    print('Sampling: 100 training windows without replacement')
    print('Random seed: 42')
    print('Prediction function: -IF.decision_function')

    background = get_documented_background(
        Path(prep['score_file']),
        list(prep['scaled_feature_names']),
        cfg['expected_training']
    )

    background_scores = np.asarray(
        -prep['model'].decision_function(background),
        dtype=np.float64
    )

    if (
        background_scores.shape != (BACKGROUND_SIZE,)
        or not np.isfinite(background_scores).all()
    ):
        raise RuntimeError(
            f'STOP: Invalid background scores for {dataset}'
        )

    independently_expected = float(
        np.mean(background_scores)
    )

    baseline_difference = abs(
        baseline_mean - independently_expected
    )

    print('Background shape:', background.shape)
    print('Background mean anomaly score:',
          f'{independently_expected:.12f}')
    print('Implied SHAP expected value:',
          f'{baseline_mean:.12f}')
    print('Absolute expectation difference:',
          f'{baseline_difference:.12g}')

    background_agrees = (
        baseline_difference <= BACKGROUND_TOLERANCE
    )

    print('Background agreement:',
          'PASS' if background_agrees else 'REVIEW')

    # Distinguish independent agreement from merely
    # having a constant implied baseline.
    if baseline_consistent and background_agrees:
        status = 'STRONG_NUMERICAL_SUPPORT'
    elif baseline_consistent:
        status = 'BASELINE_CONSISTENT_BACKGROUND_UNCONFIRMED'
    else:
        status = 'REVIEW_REQUIRED'

    D5_KERNEL_SHAP_AUDIT[dataset] = {
        'status': status,
        'n_windows': len(records),
        'implied_expected_value': baseline_mean,
        'implied_baseline_range': baseline_range,
        'reconstructed_background_expected_value':
            independently_expected,
        'background_difference': baseline_difference,
        'records': records
    }

    print('\nDATASET STATUS:', status)

print('\n' + '=' * 100)
print('CELL 10 COMPLETE — NUMERICAL PROVENANCE AUDIT')
print('=' * 100)

for dataset, result in D5_KERNEL_SHAP_AUDIT.items():
    print(
        f"{dataset}: {result['status']} "
        f"| baseline range="
        f"{result['implied_baseline_range']:.3e} "
        f"| background difference="
        f"{result['background_difference']:.3e}"
    )

print('\nInterpretation:')
print('A strong match supports the documented continuous-score')
print('Kernel SHAP target and sampled background.')
print('A mismatch requires investigation; it does not by itself')
print('prove the historical SHAP explanation is incorrect.')
print('Numerical consistency does not recover original source code.')
print('No historical explanations or models were changed.')
print('No evidence files written.')
print('C3-12 and C4-12 remain OPEN.')
print('=' * 100)


In [ ]:

# ============================================================
# D5 / C4-12 — CELL 11
# UK-DALE HISTORICAL SHAP BACKGROUND INVESTIGATION
#
# Tests documented seed 42 under plausible historical
# sampling procedures and training-row orders.
#
# READ ONLY
# No model fitting, SHAP/LIME execution, or file writes.
# An exact numerical match is evidence, NOT proof
# of the unrecovered historical implementation.
# ============================================================

from pathlib import Path
import os
import numpy as np
import pandas as pd

if 'D5_REGRESSION_PREP' not in globals():
    raise RuntimeError('STOP: Cell 6 is required.')

if 'D5_KERNEL_SHAP_AUDIT' not in globals():
    raise RuntimeError('STOP: Cell 10 is required.')

prep = D5_REGRESSION_PREP['UK-DALE']
audit = D5_KERNEL_SHAP_AUDIT['UK-DALE']

SCORE_FILE = Path(prep['score_file'])
EXPECTED_TRAIN = 218358
BACKGROUND_N = 100
SEED = 42

historical_baseline = float(
    audit['implied_expected_value']
)

cell10_baseline = float(
    audit['reconstructed_background_expected_value']
)

print('=' * 100)
print('D5 / C4-12 — UK-DALE BACKGROUND INVESTIGATION')
print('=' * 100)
print('Historical implied baseline:',
      f'{historical_baseline:.15f}')
print('Cell 10 reconstructed baseline:',
      f'{cell10_baseline:.15f}')
print('Discrepancy:',
      f'{abs(historical_baseline-cell10_baseline):.12g}')

# ------------------------------------------------------------
# 1. READ ORIGINAL FROZEN SCORING RECORDS
# ------------------------------------------------------------

header = pd.read_csv(
    SCORE_FILE, nrows=0
).columns.tolist()

required = ['split', 'anomaly_score']

missing = [c for c in required if c not in header]

if missing:
    raise RuntimeError(
        f'STOP: Missing required scoring columns: {missing}'
    )

optional = [
    c for c in [
        'window_start',
        'series_key',
        'original_source_row_id'
    ]
    if c in header
]

df = pd.read_csv(
    SCORE_FILE,
    usecols=required + optional,
    float_precision='round_trip',
    low_memory=False
)

train = (
    df.loc[df['split'].astype(str).eq('train')]
    .copy()
    .reset_index(drop=True)
)

if len(train) != EXPECTED_TRAIN:
    raise RuntimeError(
        f'STOP: Unexpected training population {len(train)}'
    )

scores = train['anomaly_score'].to_numpy(
    dtype=np.float64
)

if not np.isfinite(scores).all():
    raise RuntimeError(
        'STOP: Nonfinite historical training scores.'
    )

print('\n1. HISTORICAL SCORING POPULATION')
print('Training rows:', len(train))
print('Training score mean:', f'{scores.mean():.12f}')
print('Training score minimum:', f'{scores.min():.12f}')
print('Training score maximum:', f'{scores.max():.12f}')

# ------------------------------------------------------------
# 2. PREDEFINED CANDIDATE TRAINING-ROW ORDERS
# ------------------------------------------------------------

orders = {
    'scoring_csv_order': np.arange(
        EXPECTED_TRAIN, dtype=np.int64
    )
}

if 'original_source_row_id' in train.columns:

    source_ids = pd.to_numeric(
        train['original_source_row_id'],
        errors='coerce'
    )

    print('\nOriginal-source identifiers:')
    print('  Missing/non-numeric:',
          int(source_ids.isna().sum()))
    print('  Unique:',
          int(source_ids.nunique(dropna=True)))

    if (
        source_ids.notna().all()
        and source_ids.is_unique
    ):
        orders['source_row_id_order'] = (
            np.argsort(
                source_ids.to_numpy(),
                kind='stable'
            )
        )
    else:
        print(
            '  Source-ID ordering skipped: '
            'identifiers are not unique and complete.'
        )

if (
    'series_key' in train.columns
    and 'window_start' in train.columns
):
    orders['series_then_time'] = (
        train.sort_values(
            ['series_key', 'window_start'],
            kind='stable'
        ).index.to_numpy(dtype=np.int64)
    )

    orders['time_then_series'] = (
        train.sort_values(
            ['window_start', 'series_key'],
            kind='stable'
        ).index.to_numpy(dtype=np.int64)
    )

print('\n2. CANDIDATE ROW ORDERS')

for name, order in orders.items():

    if (
        order.shape != (EXPECTED_TRAIN,)
        or len(np.unique(order)) != EXPECTED_TRAIN
    ):
        raise RuntimeError(
            f'STOP: Invalid row order: {name}'
        )

    print(
        name,
        '| first five source positions:',
        order[:5].tolist(),
        '| differs from CSV order:',
        not np.array_equal(
            order, orders['scoring_csv_order']
        )
    )

# ------------------------------------------------------------
# 3. PREDEFINED SEED-42 SAMPLING PROCEDURES
# ------------------------------------------------------------

def sampled_positions(method, count):

    if method == 'generator_choice':
        return np.random.default_rng(SEED).choice(
            count,
            size=BACKGROUND_N,
            replace=False
        )

    if method == 'legacy_randomstate_choice':
        return np.random.RandomState(SEED).choice(
            count,
            size=BACKGROUND_N,
            replace=False
        )

    if method == 'generator_permutation':
        return np.random.default_rng(SEED).permutation(
            count
        )[:BACKGROUND_N]

    raise ValueError(
        f'Unknown sampling method: {method}'
    )

sampling_methods = [
    'generator_choice',
    'legacy_randomstate_choice',
    'generator_permutation'
]

# Verify the documented, as-stored construction first.
original_positions = sampled_positions(
    'generator_choice',
    EXPECTED_TRAIN
)

reproduced_cell10 = float(
    np.mean(scores[original_positions])
)

print('\n3. CELL 10 REPRODUCTION')
print('From saved training anomaly scores:',
      f'{reproduced_cell10:.15f}')
print('From Cell 10 model scoring:',
      f'{cell10_baseline:.15f}')

if abs(reproduced_cell10 - cell10_baseline) > 1e-9:
    raise RuntimeError(
        'STOP: Saved scores do not reproduce the '
        'Cell 10 reference expectation.'
    )

print('Reproduction: PASS')

# ------------------------------------------------------------
# 4. EVALUATE PREDEFINED HYPOTHESES
# ------------------------------------------------------------

print('\n4. BACKGROUND SAMPLING HYPOTHESES')
print('Seed fixed at 42 for every hypothesis.')
print('No seed search or numerical optimization.')

records = []

for order_name, order in orders.items():

    for sampling_method in sampling_methods:

        indices = sampled_positions(
            sampling_method,
            EXPECTED_TRAIN
        )

        source_positions = order[indices]

        candidate_mean = float(
            np.mean(scores[source_positions])
        )

        difference = abs(
            candidate_mean - historical_baseline
        )

        records.append({
            'row_order': order_name,
            'sampling_method': sampling_method,
            'background_mean': candidate_mean,
            'difference_from_historical': difference,
            'exact_within_1e_9':
                difference <= 1e-9
        })

results = pd.DataFrame(records)

print(
    results.sort_values(
        'difference_from_historical'
    ).to_string(
        index=False,
        float_format=lambda x: f'{x:.12f}'
    )
)

exact_matches = results.loc[
    results['exact_within_1e_9']
]

print('\nExact numerical matches:',
      len(exact_matches))

if len(exact_matches):
    print(
        exact_matches[
            ['row_order', 'sampling_method']
        ].to_string(index=False)
    )
    print(
        'IMPORTANT: Matching a sampled mean does not '
        'recover the original explainer implementation.'
    )
else:
    print(
        'No exact baseline match in the predefined '
        'sampling/order hypotheses.'
    )

# ------------------------------------------------------------
# 5. INVENTORY POSSIBLE HISTORICAL UK-DALE SOURCES
# ------------------------------------------------------------

SEARCH_ROOTS = [
    Path(
        '/content/drive/MyDrive/Master/Thesis/Code/'
        'experiments/EXP_UKDALE_Selected_Pipeline'
    ),
    Path(
        '/content/drive/MyDrive/Master/Thesis/Code/'
        'experiments/EXP_SPLIT_BOUNDARY_PURGE_CORRECTION/'
        'UKDALE'
    )
]

print('\n5. POSSIBLE HISTORICAL SOURCE ARTIFACTS')

for root in SEARCH_ROOTS:

    print('\nDirectory:', root)
    print('Exists:', root.is_dir())

    if not root.is_dir():
        continue

    candidates = []

    for current, dirs, filenames in os.walk(root):
        current = Path(current)

        depth = len(current.relative_to(root).parts)

        if depth >= 3:
            dirs[:] = []
        else:
            dirs[:] = sorted(dirs)

        for filename in filenames:

            lower = filename.lower()

            if not any(
                token in lower
                for token in [
                    'train', 'score', 'shap',
                    'background', 'manifest',
                    'feature', 'model'
                ]
            ):
                continue

            path = current / filename

            if path.is_file():
                candidates.append(
                    (str(path.relative_to(root)),
                     path.stat().st_size)
                )

    print('Matching artifact names:', len(candidates))

    for name, size in sorted(candidates)[:35]:
        print(f'  {name} | {size:,} bytes')

    if len(candidates) > 35:
        print('  Additional names omitted:',
              len(candidates) - 35)

# Publish results only in notebook memory.
D5_UK_BACKGROUND_HYPOTHESES = results.copy()

print('\n' + '=' * 100)
print('CELL 11 COMPLETE — UK-DALE BACKGROUND INVESTIGATED')
print('=' * 100)
print('Saved historical SHAP values remain unchanged.')
print('No historical model or explainer fitted.')
print('No files written or modified.')
print('C4-12 remains OPEN until provenance is resolved.')
print('=' * 100)


In [ ]:

# ============================================================
# D5 / C4-12 — CELL 12
# FINAL KERNEL SHAP TARGET / BACKGROUND VERIFICATION
#
# UK-DALE: original_source_row_id ordering
# AMPds2:  frozen scoring CSV ordering
#
# Seed: 42; background size: 100
# Target: -IsolationForest.decision_function
#
# READ ONLY — NO model fitting, SHAP/LIME runs or writes.
# ============================================================

import hashlib
from pathlib import Path
import numpy as np
import pandas as pd

REQUIRED_OBJECTS = [
    'D5_REGRESSION_PREP',
    'D5_KERNEL_SHAP_AUDIT',
    'D5_UK_BACKGROUND_HYPOTHESES',
    'D5_LIME_FULL',
    'D5_REGRESSION_AGREEMENT'
]

for name in REQUIRED_OBJECTS:
    if name not in globals():
        raise RuntimeError(
            f'STOP: Required previous-cell result missing: {name}'
        )

CONFIG = {
    'UK-DALE': {
        'expected_train': 218358,
        'expected_windows': 5,
        'order': 'original_source_row_id'
    },
    'AMPds2': {
        'expected_train': 441483,
        'expected_windows': 8,
        'order': 'scoring_csv'
    }
}

N_BACKGROUND = 100
SEED = 42
TOL = 1e-9

print('=' * 100)
print('D5 — FINAL KERNEL SHAP NUMERICAL PROVENANCE')
print('=' * 100)

confirmed = {}

for dataset, cfg in CONFIG.items():

    print('\n' + '=' * 100)
    print('DATASET:', dataset)
    print('=' * 100)

    prep = D5_REGRESSION_PREP[dataset]
    previous = D5_KERNEL_SHAP_AUDIT[dataset]

    file_path = Path(prep['score_file'])
    feature_columns = list(prep['scaled_feature_names'])

    if len(feature_columns) != 23:
        raise RuntimeError(
            f'STOP: Invalid feature schema: {dataset}'
        )

    read_columns = ['split', 'anomaly_score'] + feature_columns

    if cfg['order'] == 'original_source_row_id':
        read_columns.append('original_source_row_id')

    data = pd.read_csv(
        file_path,
        usecols=read_columns,
        float_precision='round_trip',
        low_memory=False
    )

    train = data.loc[
        data['split'].astype(str).eq('train')
    ].copy()

    if len(train) != cfg['expected_train']:
        raise RuntimeError(
            f'STOP: Training count mismatch: {dataset}'
        )

    if cfg['order'] == 'original_source_row_id':

        ids = pd.to_numeric(
            train['original_source_row_id'],
            errors='coerce'
        )

        if ids.isna().any() or not ids.is_unique:
            raise RuntimeError(
                'STOP: Original source-row IDs must be '
                'complete and unique.'
            )

        train['_source_id_numeric'] = ids.to_numpy()

        train = train.sort_values(
            '_source_id_numeric',
            kind='stable'
        )

        ordered_source_ids = train[
            '_source_id_numeric'
        ].to_numpy().copy()

        print('Training order: original source-row ID')
        print('Original source IDs unique: PASS')
        print('First five source IDs:',
              ordered_source_ids[:5].tolist())

    else:
        ordered_source_ids = None
        print('Training order: frozen scoring CSV')

    X_train = train[
        feature_columns
    ].to_numpy(dtype=np.float32)

    historical_train_scores = train[
        'anomaly_score'
    ].to_numpy(dtype=np.float64)

    if (
        X_train.shape != (cfg['expected_train'], 23)
        or not np.isfinite(X_train).all()
        or not np.isfinite(historical_train_scores).all()
    ):
        raise RuntimeError(
            f'STOP: Invalid training representation: {dataset}'
        )

    rng = np.random.default_rng(SEED)

    positions = rng.choice(
        len(X_train),
        size=N_BACKGROUND,
        replace=False
    )

    background = X_train[positions].copy()

    historical_background_scores = (
        historical_train_scores[positions]
    )

    # Score the reconstructed background directly
    # using the previously SHA-256-verified frozen IF.
    model_background_scores = np.asarray(
        -prep['model'].decision_function(background),
        dtype=np.float64
    )

    if (
        model_background_scores.shape != (100,)
        or not np.isfinite(model_background_scores).all()
    ):
        raise RuntimeError(
            f'STOP: Invalid model background scores: {dataset}'
        )

    max_background_score_error = float(
        np.max(np.abs(
            model_background_scores
            - historical_background_scores
        ))
    )

    model_background_mean = float(
        np.mean(model_background_scores)
    )

    historical_shap_baseline = float(
        previous['implied_expected_value']
    )

    baseline_error = abs(
        model_background_mean
        - historical_shap_baseline
    )

    print('\n1. INDEPENDENT BACKGROUND MODEL CHECK')
    print('Training rows:', len(X_train))
    print('Background rows:', len(background))
    print('Seed:', SEED)
    print('First five sampled ordinals:',
          positions[:5].tolist())
    print('Max saved-vs-model score discrepancy:',
          f'{max_background_score_error:.16g}')
    print('Frozen-model background expectation:',
          f'{model_background_mean:.15f}')
    print('Saved-SHAP implied expectation:',
          f'{historical_shap_baseline:.15f}')
    print('Expectation discrepancy:',
          f'{baseline_error:.16g}')

    if max_background_score_error > TOL:
        raise RuntimeError(
            f'STOP: Background model-score mismatch: {dataset}'
        )

    if baseline_error > TOL:
        raise RuntimeError(
            f'STOP: SHAP expectation mismatch: {dataset}'
        )

    print('BACKGROUND EXPECTATION: PASS')

    print('\n2. PER-WINDOW SHAP ADDITIVITY')

    records = previous['records']

    if len(records) != cfg['expected_windows']:
        raise RuntimeError(
            f'STOP: Incorrect SHAP explanation count: {dataset}'
        )

    residuals = []

    for record in records:

        reconstructed_score = (
            model_background_mean
            + float(record['shap_sum'])
        )

        historical_score = float(
            record['score']
        )

        residual = abs(
            reconstructed_score - historical_score
        )

        residuals.append(residual)

        print(
            record['identity'],
            '| reconstruction error:',
            f'{residual:.3e}'
        )

    max_additivity_error = float(
        np.max(residuals)
    )

    print('Maximum additive reconstruction error:',
          f'{max_additivity_error:.16g}')

    if max_additivity_error > TOL:
        raise RuntimeError(
            f'STOP: SHAP additivity mismatch: {dataset}'
        )

    print('SHAP ADDITIVITY: PASS')

    background_hash = hashlib.sha256(
        background.tobytes(order='C')
    ).hexdigest()

    positions_hash = hashlib.sha256(
        np.asarray(
            positions, dtype='<i8'
        ).tobytes()
    ).hexdigest()

    confirmed[dataset] = {
        'status': 'NUMERICALLY_CONFIRMED',
        'training_order': cfg['order'],
        'background_size': N_BACKGROUND,
        'sampling_seed': SEED,
        'background_sha256': background_hash,
        'sample_positions_sha256': positions_hash,
        'expected_value': model_background_mean,
        'historical_implied_baseline':
            historical_shap_baseline,
        'baseline_error': baseline_error,
        'max_background_score_error':
            max_background_score_error,
        'max_additivity_error':
            max_additivity_error,
        'n_shap_windows': len(records)
    }

    print('\n3. REPRODUCIBILITY FINGERPRINTS')
    print('Background SHA-256:', background_hash)
    print('Sample-position SHA-256:', positions_hash)
    print('DATASET STATUS: NUMERICALLY_CONFIRMED')

print('\n' + '=' * 100)
print('CROSS-CHECK: D5 EXPLANATION COMPLETENESS')
print('=' * 100)

n_lime = sum(
    len(D5_LIME_FULL[d])
    for d in CONFIG
)

n_agreement = len(
    D5_REGRESSION_AGREEMENT
)

n_shap = sum(
    result['n_shap_windows']
    for result in confirmed.values()
)

print('Regression LIME explanations:', n_lime)
print('SHAP windows numerically verified:', n_shap)
print('Quantitative comparison records:', n_agreement)

if (n_lime, n_shap, n_agreement) != (13, 13, 13):
    raise RuntimeError(
        'STOP: D5 result populations do not match.'
    )

if set(confirmed) != set(CONFIG):
    raise RuntimeError(
        'STOP: Both datasets must pass.'
    )

D5_SHAP_TARGET_CONFIRMED = confirmed

print('\n' + '=' * 100)
print('CELL 12 COMPLETE — NUMERICAL TARGET AUDIT PASSED')
print('=' * 100)
print('Both historical SHAP baselines independently reproduced.')
print('All 13 local SHAP additive decompositions verified.')
print('Frozen Isolation Forest score function confirmed.')
print('Historical source code is still not recovered.')
print('No SHAP or LIME attributions recomputed.')
print('No models trained or modified.')
print('No files written.')
print('D5 ready for evidence preservation.')
print('=' * 100)


In [ ]:

# ============================================================
# D5 / C3-12 + C4-12 — CELL 13
# PRESERVE VERIFIED LOCAL XAI EVIDENCE
#
# Outputs:
#   1. lime_fidelity_13_windows.csv
#   2. shap_lime_agreement_13_windows.csv
#   3. regression_lime_coefficients_130_rows.csv
#   4. selected_scaled_inputs_13_windows.csv
#   5. dataset_summary.csv
#   6. shap_background_verification.csv
#   7. historical_snapshots/
#   8. manifest.json
#   9. README.md
#
# No model training or new explanation computations.
# Original evidence files are never overwritten.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import hashlib
import json
import shutil
import sys
import numpy as np
import pandas as pd
import sklearn

REQUIRED_D5_OBJECTS = [
    'D5_INPUT_AUDIT',
    'D5_REGRESSION_PREP',
    'D5_LIME_FULL',
    'D5_REGRESSION_AGREEMENT',
    'D5_SHAP_TARGET_CONFIRMED'
]

for name in REQUIRED_D5_OBJECTS:
    if name not in globals():
        raise RuntimeError(
            f'STOP: Missing previous-cell object: {name}'
        )

BASE_D5 = Path('/content/drive/MyDrive/Master/Thesis/Code')
EXP_D5 = BASE_D5 / 'experiments'

DATASETS_D5 = {
    'UK-DALE': {
        'prefix': 'UKDALE',
        'n': 5,
        'keys': [
            'building', 'meter',
            'window_start', 'window_end'
        ]
    },
    'AMPds2': {
        'prefix': 'AMPDS2',
        'n': 8,
        'keys': [
            'candidate_id', 'channel_code',
            'window_start', 'window_end'
        ]
    }
}

SNAPSHOT_SOURCES = {
    'ukdale_historical_shap.csv':
        EXP_D5 /
        'EXP_UKDALE_Selected_Pipeline/xai/'
        'ukdale_shap_explanations_selected_test_windows.csv',

    'ukdale_historical_lime.csv':
        EXP_D5 /
        'EXP_UKDALE_Selected_Pipeline/xai/'
        'ukdale_lime_explanations_selected_test_windows.csv',

    'ampds2_historical_shap.csv':
        EXP_D5 /
        'EXP_AMPDS2_Selected_Pipeline/xai/'
        'ampds2_shap_explanations_selected_test_windows.csv',

    'ampds2_historical_classification_lime.csv':
        EXP_D5 /
        'EXP_AMPDS2_Selected_Pipeline/xai/'
        'ampds2_lime_explanations_selected_test_windows.csv',

    'ampds2_corrected_classification_lime.csv':
        EXP_D5 /
        'EXP_SPLIT_BOUNDARY_PURGE_CORRECTION/'
        'AMPDS2/XAI_LIME_CORRECTION/'
        'ampds2_lime_explanations_selected_test_windows_CORRECTED.csv'
}

def sha256_file_d5(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024), b''
        ):
            digest.update(block)
    return digest.hexdigest()

def python_scalar_d5(value):
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    return value

# ------------------------------------------------------------
# 1. VALIDATE EXISTING SOURCES BEFORE ANY WRITES
# ------------------------------------------------------------

print('=' * 100)
print('D5 / C3-12 + C4-12 — EVIDENCE PRESERVATION')
print('=' * 100)

if not EXP_D5.is_dir():
    raise RuntimeError(
        'STOP: Thesis experiments directory missing.'
    )

for name, source in SNAPSHOT_SOURCES.items():
    if not source.is_file():
        raise RuntimeError(
            f'STOP: Missing original artifact {name}: {source}'
        )

for dataset, cfg in DATASETS_D5.items():

    prep = D5_REGRESSION_PREP[dataset]
    audit = D5_INPUT_AUDIT[dataset]
    confirmed = D5_SHAP_TARGET_CONFIRMED[dataset]

    if (
        audit['status'] != 'PASS'
        or confirmed['status'] != 'NUMERICALLY_CONFIRMED'
        or len(D5_LIME_FULL[dataset]) != cfg['n']
        or len(audit['recovered_windows']) != cfg['n']
        or confirmed['n_shap_windows'] != cfg['n']
    ):
        raise RuntimeError(
            f'STOP: Unverified D5 population: {dataset}'
        )

    if (
        confirmed['baseline_error'] > 1e-9
        or confirmed['max_additivity_error'] > 1e-9
    ):
        raise RuntimeError(
            f'STOP: SHAP baseline/additivity failed: {dataset}'
        )

    if sha256_file_d5(prep['model_path']) != prep['model_sha256']:
        raise RuntimeError(
            f'STOP: Frozen model fingerprint changed: {dataset}'
        )

    reference_hash = hashlib.sha256(
        np.asarray(
            prep['reference'], dtype=np.float32
        ).tobytes(order='C')
    ).hexdigest()

    if reference_hash != prep['reference_sha256']:
        raise RuntimeError(
            f'STOP: LIME reference fingerprint changed: {dataset}'
        )

print('Required historical files: FOUND')
print('Frozen model fingerprints: PASS')
print('Training-reference fingerprints: PASS')
print('Kernel SHAP numerical provenance: PASS')

# ------------------------------------------------------------
# 2. BUILD 13-WINDOW TABLES
# ------------------------------------------------------------

fidelity_rows = []
agreement_rows = []
coefficient_rows = []
input_rows = []
background_rows = []

for dataset, cfg in DATASETS_D5.items():

    prep = D5_REGRESSION_PREP[dataset]
    originals = D5_INPUT_AUDIT[dataset]['recovered_windows']
    explanations = D5_LIME_FULL[dataset]

    dataset_agreement = (
        D5_REGRESSION_AGREEMENT.loc[
            D5_REGRESSION_AGREEMENT['dataset'].eq(dataset)
        ].copy()
    )

    if (
        len(dataset_agreement) != cfg['n']
        or dataset_agreement['identity'].duplicated().any()
    ):
        raise RuntimeError(
            f'STOP: Incorrect agreement records: {dataset}'
        )

    agreement_map = {
        str(row['identity']): row
        for _, row in dataset_agreement.iterrows()
    }

    feature_names = list(prep['feature_names'])
    scaled_names = list(prep['scaled_feature_names'])

    if len(feature_names) != 23 or len(scaled_names) != 23:
        raise RuntimeError(
            f'STOP: Incorrect feature schema: {dataset}'
        )

    for i in range(cfg['n']):

        original = originals[i]
        result = explanations[i]
        identity = str(original['identity'])
        window_id = f"{cfg['prefix']}_{i+1:02d}"

        if (
            identity != str(result['identity'])
            or identity != str(prep['identities'][i])
            or identity not in agreement_map
        ):
            raise RuntimeError(
                f'STOP: Window identity mismatch: {window_id}'
            )

        comparison = agreement_map[identity]

        score = float(original['original_score'])
        lime_r2 = float(result['local_r2'])

        if (
            abs(score - result['historical_score']) > 1e-9
            or abs(score - comparison['anomaly_score']) > 1e-9
            or abs(lime_r2 - comparison['lime_local_r2']) > 1e-9
        ):
            raise RuntimeError(
                f'STOP: Fidelity/agreement mismatch: {window_id}'
            )

        metadata = {
            key: python_scalar_d5(value)
            for key, value in zip(
                cfg['keys'], original['identity']
            )
        }

        common = {
            'window_id': window_id,
            'dataset': dataset,
            'identity': identity,
            **metadata
        }

        fidelity_rows.append({
            **common,
            'anomaly_score': score,
            'lime_target_prediction':
                float(result['lime_target_prediction']),
            'lime_surrogate_prediction':
                float(result['surrogate_prediction']),
            'lime_local_r2': lime_r2,
            'absolute_local_prediction_error':
                float(result['absolute_local_error']),
            'lime_mode': 'regression',
            'lime_target':
                '-IsolationForest.decision_function',
            'lime_num_samples': 5000,
            'lime_num_features': 10,
            'lime_random_state': 42,
            'lime_top5': ', '.join(result['top5'])
        })

        agreement_rows.append({
            **common,
            'anomaly_score': score,
            'top5_overlap_count':
                int(comparison['shared_top5_count']),
            'top5_jaccard':
                float(comparison['top5_jaccard']),
            'spearman_lime_selected_top10':
                float(comparison['spearman_top10']),
            'lime_local_r2': lime_r2,
            'shared_features':
                str(comparison['shared_features']),
            'shap_top5':
                str(comparison['shap_top5']),
            'regression_lime_top5':
                str(comparison['new_lime_top5']),
            'historical_lime_top5':
                str(comparison['historical_lime_top5']),
            'historical_vs_new_lime_jaccard':
                float(
                    comparison[
                        'historical_vs_new_lime_jaccard'
                    ]
                )
        })

        weights = result['ranked_features']

        if len(weights) != 10:
            raise RuntimeError(
                f'STOP: Incorrect coefficient count: {window_id}'
            )

        for j, item in enumerate(weights, start=1):
            if int(item['rank']) != j:
                raise RuntimeError(
                    f'STOP: LIME rank inconsistency: {window_id}'
                )

            coefficient_rows.append({
                'window_id': window_id,
                'dataset': dataset,
                'rank': j,
                'feature': str(item['feature']),
                'feature_scaled':
                    str(item['feature_scaled']),
                'lime_weight':
                    float(item['weight']),
                'abs_lime_weight':
                    float(item['abs_weight']),
                'lime_mode': 'regression'
            })

        X = np.asarray(
            prep['X_selected'][i],
            dtype=np.float32
        )

        if (
            X.shape != (23,)
            or not np.isfinite(X).all()
        ):
            raise RuntimeError(
                f'STOP: Invalid selected input: {window_id}'
            )

        input_rows.append({
            **common,
            'anomaly_score': score,
            **{
                feature: float(value)
                for feature, value in zip(scaled_names, X)
            }
        })

    verified = D5_SHAP_TARGET_CONFIRMED[dataset]

    background_rows.append({
        'dataset': dataset,
        'status': verified['status'],
        'training_order': verified['training_order'],
        'background_size':
            int(verified['background_size']),
        'sampling_seed':
            int(verified['sampling_seed']),
        'model_sha256': prep['model_sha256'],
        'lime_reference_sha256':
            prep['reference_sha256'],
        'shap_background_sha256':
            verified['background_sha256'],
        'shap_background_positions_sha256':
            verified['sample_positions_sha256'],
        'shap_background_expected_value':
            float(verified['expected_value']),
        'historical_shap_implied_baseline':
            float(verified['historical_implied_baseline']),
        'baseline_absolute_error':
            float(verified['baseline_error']),
        'max_background_score_error':
            float(verified['max_background_score_error']),
        'max_shap_additivity_error':
            float(verified['max_additivity_error']),
        'n_shap_windows':
            int(verified['n_shap_windows'])
    })

fidelity_df = pd.DataFrame(fidelity_rows)
agreement_df = pd.DataFrame(agreement_rows)
coefficient_df = pd.DataFrame(coefficient_rows)
inputs_df = pd.DataFrame(input_rows)
background_df = pd.DataFrame(background_rows)

if (
    len(fidelity_df),
    len(agreement_df),
    len(coefficient_df),
    len(inputs_df),
    len(background_df)
) != (13, 13, 130, 13, 2):
    raise RuntimeError(
        'STOP: Evidence table sizes are inconsistent.'
    )

if (
    fidelity_df['window_id'].duplicated().any()
    or agreement_df['window_id'].duplicated().any()
    or inputs_df['window_id'].duplicated().any()
):
    raise RuntimeError(
        'STOP: Duplicate window identifiers.'
    )

summary_df = (
    fidelity_df
    .groupby('dataset', sort=False)
    .agg(
        n_windows=('window_id', 'count'),
        mean_local_r2=('lime_local_r2', 'mean'),
        median_local_r2=('lime_local_r2', 'median'),
        min_local_r2=('lime_local_r2', 'min'),
        max_local_r2=('lime_local_r2', 'max'),
        mean_pointwise_absolute_error=(
            'absolute_local_prediction_error',
            'mean'
        )
    )
    .reset_index()
)

agreement_summary = (
    agreement_df
    .groupby('dataset', sort=False)
    .agg(
        mean_top5_jaccard=('top5_jaccard', 'mean'),
        mean_spearman=('spearman_lime_selected_top10', 'mean')
    )
    .reset_index()
)

summary_df = summary_df.merge(
    agreement_summary,
    on='dataset',
    validate='one_to_one'
)

print('\nValidated evidence tables:')
print('Fidelity records:', len(fidelity_df))
print('Agreement records:', len(agreement_df))
print('New LIME coefficients:', len(coefficient_df))
print('Original scaled input vectors:', len(inputs_df))
print('SHAP background records:', len(background_df))
print('\nSummary:')
print(
    summary_df.to_string(
        index=False,
        float_format=lambda x: f'{x:.6f}'
    )
)

# ------------------------------------------------------------
# 3. CREATE A NEW, VERSIONED EVIDENCE DIRECTORY
# ------------------------------------------------------------

run_stamp = datetime.now(timezone.utc).strftime(
    '%Y%m%dT%H%M%S_%fZ'
)

folder_name = (
    'D5_C3_12_C4_12_LOCAL_XAI_EVIDENCE_'
    + run_stamp
)

final_directory = EXP_D5 / folder_name
staging_directory = EXP_D5 / (folder_name + '_INCOMPLETE')

if (
    final_directory.exists()
    or staging_directory.exists()
):
    raise RuntimeError(
        'STOP: A destination directory already exists.'
    )

staging_directory.mkdir(
    parents=False,
    exist_ok=False
)

tables = {
    'lime_fidelity_13_windows.csv': fidelity_df,
    'shap_lime_agreement_13_windows.csv': agreement_df,
    'regression_lime_coefficients_130_rows.csv':
        coefficient_df,
    'selected_scaled_inputs_13_windows.csv': inputs_df,
    'dataset_summary.csv': summary_df,
    'shap_background_verification.csv': background_df
}

for filename, dataframe in tables.items():
    dataframe.to_csv(
        staging_directory / filename,
        index=False,
        float_format='%.17g'
    )

snapshot_directory = (
    staging_directory / 'historical_snapshots'
)

snapshot_directory.mkdir(exist_ok=False)

source_manifest = {}

for snapshot_name, source in SNAPSHOT_SOURCES.items():

    destination = snapshot_directory / snapshot_name

    shutil.copy2(source, destination)

    source_hash = sha256_file_d5(source)
    copied_hash = sha256_file_d5(destination)

    if source_hash != copied_hash:
        raise RuntimeError(
            'STOP: Historical snapshot hash mismatch: '
            + snapshot_name
        )

    source_manifest[snapshot_name] = {
        'original_path': str(source),
        'source_sha256': source_hash,
        'copied_sha256': copied_hash
    }

# ------------------------------------------------------------
# 4. REPRODUCIBILITY MANIFEST AND README
# ------------------------------------------------------------

manifest = {
    'study': (
        'Explainable Anomaly Detection in '
        'Smart Home Energy Systems'
    ),
    'revision': 'D5 / C3-12 + C4-12',
    'created_utc': datetime.now(
        timezone.utc
    ).isoformat(),
    'status': 'EVIDENCE_PRESERVED_MANUSCRIPT_PENDING',
    'n_windows': 13,
    'dataset_window_counts': {
        'UK-DALE': 5,
        'AMPds2': 8
    },
    'environment': {
        'python': sys.version.split()[0],
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__,
        'lime': version('lime')
    },
    'prediction_target': (
        '-IsolationForest.decision_function(X)'
    ),
    'lime_protocol': {
        'mode': 'regression',
        'feature_representation':
            '23 normalized engineered features',
        'training_reference_size': 8192,
        'training_reference_sampling':
            'uniform without replacement',
        'reference_seeds': {
            'UK-DALE': 2026,
            'AMPds2': 2027
        },
        'num_samples': 5000,
        'num_reported_features': 10,
        'random_state_per_window': 42,
        'feature_selection': 'auto',
        'discretize_continuous': True,
        'discretizer': 'quartile',
        'sample_around_instance': False,
        'kernel_width': float(
            0.75 * np.sqrt(23)
        )
    },
    'metrics': {
        'local_r2': (
            'LIME weighted local-surrogate R-squared '
            'on generated perturbations; not held-out fidelity'
        ),
        'pointwise_absolute_error': (
            'Absolute difference between the fitted '
            'LIME local prediction and frozen IF score '
            'at the selected original window'
        ),
        'jaccard': (
            'Intersection-over-union of absolute-magnitude '
            'top-five feature identity sets'
        ),
        'spearman': (
            'Spearman rho between LIME ranks 1-10 and '
            'their positions in the complete 23-feature '
            'historical Kernel SHAP ranking'
        )
    },
    'model_and_background': {
        dataset: {
            'model_path':
                D5_REGRESSION_PREP[dataset]['model_path'],
            'model_sha256':
                D5_REGRESSION_PREP[dataset]['model_sha256'],
            'scoring_file':
                D5_REGRESSION_PREP[dataset]['score_file'],
            'training_reference_sha256':
                D5_REGRESSION_PREP[dataset]['reference_sha256'],
            'training_position_sha256':
                D5_REGRESSION_PREP[dataset][
                    'reference_positions_sha256'
                ],
            'shap_background_verification':
                {
                    key: python_scalar_d5(value)
                    for key, value in
                    D5_SHAP_TARGET_CONFIRMED[dataset].items()
                }
        }
        for dataset in DATASETS_D5
    },
    'historical_source_snapshots': source_manifest,
    'interpretation_limits': [
        'Only 13 purposively selected test windows',
        'No anomaly-ground-truth verification from XAI',
        'No statistical inference or explanation stability test',
        'Local R-squared values indicate limited fidelity',
        'Original Kernel SHAP source code not recovered',
        'SHAP background and target supported numerically',
        'Global TreeSHAP path-length analysis is separate',
        'Historical AMPds2 classification-mode LIME '
        'must not be presented as the new regression analysis'
    ]
}

readme = """# D5 — Local XAI Fidelity and Agreement Evidence

## Scope
Supervisor revision items C3-12 and C4-12.
Five UK-DALE and eight AMPds2 original selected test windows.

## Prediction target
Continuous Isolation Forest anomaly score:
`-model.decision_function(X)`.

Both models are the verified, frozen historical Isolation
Forest models. No anomaly detector was retrained.

## Regression-mode LIME
A new, consistently configured regression-mode LIME analysis
was conducted on all 13 selected windows. It is not claimed
to be an exact reproduction of the unrecovered historical
LIME implementation.

Each dataset used 8,192 training-only reference rows.
Each explanation used 5,000 perturbation samples and
reported 10 features. The configuration is in manifest.json.

LIME local R-squared is an in-neighborhood weighted
surrogate-fit statistic, not an out-of-sample evaluation.
Pointwise prediction error is also reported.

## Historical Kernel SHAP
The original per-window Kernel SHAP feature values were
retained. A 100-row, seed-42 training-background expectation
was reproduced numerically against the frozen anomaly-score
function for both datasets.

UK-DALE required original_source_row_id ordering.
AMPds2 used the scoring CSV order.

This supports the documented continuous-score target,
but the original Kernel SHAP source code was not recovered.

## Agreement
Top-five Jaccard uses absolute-attribution ranking.
Spearman correlation compares the ten LIME-reported
features with their ranks among all 23 SHAP features.

The statistics are descriptive and do not establish
explanation correctness or physical faults.

## Source preservation
The historical_snapshots folder contains copies of the
original SHAP and LIME explanation CSVs, including the
historical and corrected AMPds2 classification-mode
LIME files. Their SHA-256 checksums are in manifest.json.

## Restrictions
Do not merge these results with the separate global
TreeSHAP native path-length explanation analysis.

Do not mark C3-12/C4-12 closed before manuscript changes
are compiled and checked.
"""

(staging_directory / 'README.md').write_text(
    readme,
    encoding='utf-8'
)

# Hash all data and snapshot files before the manifest.
artifact_hashes = {}

for path in sorted(staging_directory.rglob('*')):
    if path.is_file():
        relative = str(
            path.relative_to(staging_directory)
        )
        artifact_hashes[relative] = sha256_file_d5(path)

manifest['artifact_sha256'] = artifact_hashes

(staging_directory / 'manifest.json').write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False
    ),
    encoding='utf-8'
)

# ------------------------------------------------------------
# 5. VERIFY WRITTEN OUTPUTS AND FINALIZE DIRECTORY
# ------------------------------------------------------------

for filename, expected in tables.items():

    reread = pd.read_csv(
        staging_directory / filename,
        low_memory=False
    )

    if len(reread) != len(expected):
        raise RuntimeError(
            f'STOP: Written row-count mismatch: {filename}'
        )

for relative, recorded_hash in artifact_hashes.items():

    actual = sha256_file_d5(
        staging_directory / relative
    )

    if actual != recorded_hash:
        raise RuntimeError(
            f'STOP: Evidence checksum mismatch: {relative}'
        )

with (staging_directory / 'manifest.json').open(
    'r', encoding='utf-8'
) as handle:
    verified_manifest = json.load(handle)

if verified_manifest['n_windows'] != 13:
    raise RuntimeError(
        'STOP: Manifest verification failed.'
    )

if final_directory.exists():
    raise RuntimeError(
        'STOP: Final directory already exists.'
    )

staging_directory.rename(final_directory)

if not (
    final_directory / 'manifest.json'
).is_file():
    raise RuntimeError(
        'STOP: Final evidence folder verification failed.'
    )

D5_EVIDENCE_DIRECTORY = str(final_directory)

print('\n' + '=' * 100)
print('CELL 13 COMPLETE — D5 EVIDENCE PRESERVED')
print('=' * 100)
print('Evidence directory:', D5_EVIDENCE_DIRECTORY)
print('Fidelity windows:', len(fidelity_df))
print('Agreement windows:', len(agreement_df))
print('New regression-mode LIME coefficients:',
      len(coefficient_df))
print('Scaled input vectors:', len(inputs_df))
print('Historical explanation snapshots:',
      len(SNAPSHOT_SOURCES))
print('Checksum verification: PASS')
print('Historical models and source files unchanged.')
print('No new models or explanations fitted.')
print('C3-12/C4-12: EVIDENCE SAVED, MANUSCRIPT PENDING')
print('=' * 100)


In [ ]:

# ============================================================
# THESIS — NEW COLAB SESSION
# CELL 1: GOOGLE DRIVE MOUNT AND PATH VERIFICATION
# ============================================================

from google.colab import drive
from pathlib import Path

# Reconnect Google Drive after runtime disconnection
drive.mount('/content/drive', force_remount=True)

# Thesis directories
THESIS_ROOT = Path('/content/drive/MyDrive/Master/Thesis')
CODE_ROOT = THESIS_ROOT / 'Code'
EXPERIMENTS_ROOT = CODE_ROOT / 'experiments'

# Verify required directories
required_paths = {
    'Google Drive': Path('/content/drive/MyDrive'),
    'Thesis Root': THESIS_ROOT,
    'Code Root': CODE_ROOT,
    'Experiments Root': EXPERIMENTS_ROOT
}

print('\n' + '=' * 75)
print('THESIS — GOOGLE DRIVE VERIFICATION')
print('=' * 75)

all_passed = True

for name, path in required_paths.items():
    exists = path.is_dir()
    status = 'PASS' if exists else 'FAIL'
    print(f'{name}: {status}')
    print(f'  {path}')

    if not exists:
        all_passed = False

if not all_passed:
    raise RuntimeError(
        'STOP: One or more thesis directories are unavailable.'
    )

# Confirm previous D5 experimental evidence remains intact
d5_folders = sorted(
    EXPERIMENTS_ROOT.glob(
        'D5_C3_12_C4_12_LOCAL_XAI_EVIDENCE_*'
    )
)

print('\nD5 evidence folders found:', len(d5_folders))

for folder in d5_folders:
    manifest = folder / 'manifest.json'
    print(f'  {folder.name}')
    print(
        '  Manifest:',
        'FOUND' if manifest.is_file() else 'MISSING'
    )

print('\n' + '=' * 75)
print('CELL 1 COMPLETE — GOOGLE DRIVE CONNECTED')
print('=' * 75)
print('Thesis directories verified.')
print('Fresh Colab session initialized.')
print('No models trained.')
print('No experimental files modified.')
print('=' * 75)


In [ ]:

# ============================================================
# D7 / C3-4 — CELL 2
# FEATURE DATA AND EXPERIMENT SOURCE INVENTORY
#
# READ ONLY
# No model fitting
# No feature transformations
# No files written or modified
# ============================================================

from pathlib import Path
import os
import pandas as pd

THESIS_ROOT = Path(
    '/content/drive/MyDrive/Master/Thesis'
)
CODE_ROOT = THESIS_ROOT / 'Code'
EXP_ROOT = CODE_ROOT / 'experiments'
PROCESSED_ROOT = CODE_ROOT / 'processed_data'

print('=' * 90)
print('D7 / C3-4 — FEATURE REDUNDANCY SOURCE AUDIT')
print('=' * 90)

if not EXP_ROOT.is_dir():
    raise RuntimeError(
        'STOP: Experiments directory unavailable.'
    )

# ------------------------------------------------------------
# 1. VERIFY FROZEN RESIDENTIAL SCORING SOURCES
# ------------------------------------------------------------

residential_sources = {
    'UK-DALE': (
        EXP_ROOT
        / 'EXP_SPLIT_BOUNDARY_PURGE_CORRECTION'
        / 'UKDALE'
        / 'ukdale_if_scored_boundary_corrected.csv'
    ),
    'AMPds2': (
        EXP_ROOT
        / 'EXP_SPLIT_BOUNDARY_PURGE_CORRECTION'
        / 'AMPDS2'
        / 'ampds2_if_scored_boundary_corrected.csv'
    )
}

print('\n1. FROZEN RESIDENTIAL SOURCES')

for dataset, path in residential_sources.items():

    print('\nDataset:', dataset)
    print('File:', path.name)
    print('Exists:', path.is_file())

    if not path.is_file():
        raise RuntimeError(
            f'STOP: Missing frozen source for {dataset}'
        )

    columns = pd.read_csv(
        path, nrows=0
    ).columns.tolist()

    scaled = [
        column for column in columns
        if column.endswith('_z')
    ]

    print('Total CSV columns:', len(columns))
    print('Normalized feature columns:', len(scaled))
    print('Has n_points:', 'n_points' in columns)
    print('Has n_points_z:', 'n_points_z' in columns)
    print('Has split:', 'split' in columns)
    print('Has series_key:', 'series_key' in columns)

    print('Normalized features:')
    print(', '.join(scaled))

    if (
        len(scaled) != 23
        or 'split' not in columns
        or 'n_points' not in columns
    ):
        raise RuntimeError(
            f'STOP: Unexpected scoring schema: {dataset}'
        )

print('\nRESIDENTIAL SCHEMA: PASS')

# ------------------------------------------------------------
# 2. FIND EXISTING BLUED EXPERIMENT SOURCES
# ------------------------------------------------------------

print('\n' + '=' * 90)
print('2. BLUED EXPERIMENT DIRECTORY INVENTORY')
print('=' * 90)

experiment_dirs = sorted(
    [
        path for path in EXP_ROOT.iterdir()
        if (
            path.is_dir()
            and 'blued' in path.name.lower()
        )
    ],
    key=lambda path: path.name
)

print('BLUED-related experiment directories:',
      len(experiment_dirs))

for path in experiment_dirs:
    print(' ', path.name)

# Restrict file searches to BLUED-related directories.
# Limit recursion depth and the number of examined
# files to keep the inventory manageable.

MAX_FILES_PER_ROOT = 1500
MAX_DEPTH = 3
MAX_PRINT = 35

tokens = [
    'feature',
    'scored',
    'window',
    'model',
    'manifest',
    'configuration',
    'train'
]

for root in experiment_dirs:

    examined = 0
    candidates = []

    for current, directories, filenames in os.walk(root):

        current = Path(current)

        depth = len(
            current.relative_to(root).parts
        )

        if depth >= MAX_DEPTH:
            directories[:] = []
        else:
            directories[:] = sorted(directories)

        for filename in sorted(filenames):

            examined += 1

            if examined > MAX_FILES_PER_ROOT:
                break

            name = filename.lower()

            if (
                any(token in name for token in tokens)
                and name.endswith((
                    '.csv',
                    '.parquet',
                    '.joblib',
                    '.json',
                    '.pkl'
                ))
            ):
                path = current / filename

                candidates.append(
                    (
                        str(path.relative_to(root)),
                        path.stat().st_size
                    )
                )

        if examined > MAX_FILES_PER_ROOT:
            break

    print('\nDirectory:', root.name)
    print('Files examined:', min(
        examined, MAX_FILES_PER_ROOT
    ))
    print('Potential sources:', len(candidates))

    for name, size in candidates[:MAX_PRINT]:
        print(
            f'  {name} | {size:,} bytes'
        )

    if len(candidates) > MAX_PRINT:
        print(
            '  Additional candidates:',
            len(candidates) - MAX_PRINT
        )

    if examined > MAX_FILES_PER_ROOT:
        print(
            '  NOTE: Search cap reached.'
        )

# ------------------------------------------------------------
# 3. CHECK PROCESSED DATA LOCATIONS
# ------------------------------------------------------------

print('\n' + '=' * 90)
print('3. PROCESSED DATA LOCATIONS')
print('=' * 90)

for phase in ['PH2', 'PH3', 'PH4']:

    directory = PROCESSED_ROOT / phase

    print(
        f'{phase}:',
        'FOUND' if directory.is_dir()
        else 'NOT FOUND'
    )

    if not directory.is_dir():
        continue

    entries = sorted(
        directory.iterdir(),
        key=lambda path: path.name
    )

    print('Immediate entries:', len(entries))

    for path in entries[:12]:
        print(
            '  ',
            '[DIR]' if path.is_dir() else '[FILE]',
            path.name
        )

    if len(entries) > 12:
        print(
            '  Additional entries:',
            len(entries) - 12
        )

print('\n' + '=' * 90)
print('CELL 2 COMPLETE — SOURCE INVENTORY')
print('=' * 90)
print('Residential feature schemas verified.')
print('Existing BLUED source candidates inventoried.')
print('No datasets modified.')
print('No models loaded or fitted.')
print('No files written.')
print('C3-4 remains OPEN.')
print('=' * 90)


In [ ]:

# ============================================================
# D7 / C3-4 — CELL 3
# BLUED FEATURE SCHEMA AND SPLIT-PROVENANCE AUDIT
#
# READ ONLY
# No fitting or model loading
# No correlation analysis yet
# No files written or modified
# ============================================================

from pathlib import Path
import pandas as pd

ROOT = Path(
    '/content/drive/MyDrive/Master/Thesis/Code/experiments'
)

BLUED_ROOT = ROOT / 'EXP_BLUED_FIVE_MODEL_LABEL_BLIND'
COMMON = BLUED_ROOT / 'common'

SOURCES = {
    'Feature matrix': (
        COMMON / 'blued_features_1min_69features.csv'
    ),
    'Feature normalization': (
        COMMON / 'blued_feature_normalization.csv'
    ),
    'Model manifest': (
        COMMON / 'blued_1min_stride30s_model_manifest.csv'
    ),
    'Segment manifest': (
        COMMON / 'blued_two_block_segment_manifest.csv'
    ),
    'Isolation Forest test scores': (
        BLUED_ROOT
        / 'controlled_feature_models'
        / 'isolation_forest'
        / 'test_window_scores.csv'
    )
}

print('=' * 100)
print('D7 / C3-4 — BLUED FEATURE PROVENANCE AUDIT')
print('=' * 100)

for name, path in SOURCES.items():

    if not path.is_file():
        raise RuntimeError(
            f'STOP: Required source missing: {name}\n{path}'
        )

print('All five source files found: PASS')

# ------------------------------------------------------------
# 1. INSPECT CSV STRUCTURES
# ------------------------------------------------------------

schemas = {}

for name, path in SOURCES.items():

    print('\n' + '=' * 100)
    print('SOURCE:', name)
    print('=' * 100)

    header = pd.read_csv(path, nrows=0)
    columns = header.columns.tolist()

    schemas[name] = columns

    print('Filename:', path.name)
    print('Size (bytes):', path.stat().st_size)
    print('Number of columns:', len(columns))

    print('\nCOLUMN NAMES:')
    for i, column in enumerate(columns, start=1):
        print(f'{i:3d}. {column}')

    sample = pd.read_csv(
        path,
        nrows=4,
        low_memory=False
    )

    # Display only a compact sample to avoid flooding output.
    priority_tokens = (
        'split', 'feature', 'series', 'window',
        'segment', 'block', 'label', 'event',
        'n_points', 'mean', 'sum', 'energy',
        'variance', 'std', 'score'
    )

    priority_cols = [
        c for c in columns
        if any(token in c.lower()
               for token in priority_tokens)
    ]

    if not priority_cols:
        priority_cols = columns[:8]

    display_cols = priority_cols[:10]

    print('\nFIRST FOUR ROWS — SELECTED COLUMNS:')
    print(
        sample[display_cols].to_string(
            index=False,
            max_colwidth=35
        )
    )

# ------------------------------------------------------------
# 2. LOOK FOR PARTITION / FEATURE IDENTIFIERS
# ------------------------------------------------------------

print('\n' + '=' * 100)
print('2. PARTITION AND FEATURE IDENTIFIER AUDIT')
print('=' * 100)

for source_name in [
    'Feature matrix',
    'Feature normalization',
    'Model manifest',
    'Segment manifest'
]:

    columns = schemas[source_name]

    split_fields = [
        c for c in columns
        if any(token in c.lower() for token in [
            'split', 'partition', 'train',
            'test', 'validation', 'fold'
        ])
    ]

    feature_fields = [
        c for c in columns
        if any(token in c.lower() for token in [
            'feature', 'column', 'selected',
            'normalization', 'scaler'
        ])
    ]

    print('\nSource:', source_name)
    print('Partition-related columns:', split_fields)
    print('Feature-related columns:', feature_fields)

# ------------------------------------------------------------
# 3. INSPECT DIRECTORIES FOR MODEL METADATA
# ------------------------------------------------------------

print('\n' + '=' * 100)
print('3. EXISTING MODEL METADATA FILES')
print('=' * 100)

MODEL_ROOT = (
    BLUED_ROOT / 'controlled_feature_models'
)

for model_name in [
    'isolation_forest',
    'local_outlier_factor',
    'one_class_svm'
]:

    folder = MODEL_ROOT / model_name

    print('\nModel:', model_name)
    print('Directory exists:', folder.is_dir())

    if not folder.is_dir():
        continue

    files = sorted(
        (
            p for p in folder.iterdir()
            if p.is_file()
        ),
        key=lambda p: p.name
    )

    print('Immediate files:', len(files))

    for path in files[:20]:
        print(
            f'  {path.name}'
            f' | {path.stat().st_size:,} bytes'
        )

    if len(files) > 20:
        print(
            '  Additional files:',
            len(files) - 20
        )

# ------------------------------------------------------------
# 4. REPORT EXACT COLUMN OVERLAPS
# ------------------------------------------------------------

print('\n' + '=' * 100)
print('4. FEATURE MATRIX / MODEL MANIFEST ALIGNMENT')
print('=' * 100)

feature_columns = set(
    schemas['Feature matrix']
)

manifest_columns = set(
    schemas['Model manifest']
)

overlap = sorted(
    feature_columns & manifest_columns
)

print('Shared column names:', len(overlap))
print('Shared columns:', overlap)

D7_BLUED_SCHEMAS = schemas
D7_BLUED_SOURCE_PATHS = {
    name: str(path)
    for name, path in SOURCES.items()
}

print('\n' + '=' * 100)
print('CELL 3 COMPLETE — BLUED SOURCE SCHEMAS INSPECTED')
print('=' * 100)
print('CSV schemas and sample records inspected.')
print('Training partition has NOT yet been assumed.')
print('Constant features have NOT yet been inferred.')
print('No model fitted or loaded.')
print('No files written or modified.')
print('C3-4 remains OPEN.')
print('=' * 100)


In [ ]:

# ============================================================
# D7 / C3-4 — CELL 4
# BLUED TRAINING-FEATURE VARIANCE AUDIT
#
# READ ONLY
# No model fitting or loading
# No features removed
# No normalization recomputation for model use
# No files written or modified
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path(
    '/content/drive/MyDrive/Master/Thesis/Code/'
    'experiments/EXP_BLUED_FIVE_MODEL_LABEL_BLIND/common'
)

FEATURE_FILE = ROOT / 'blued_features_1min_69features.csv'
MANIFEST_FILE = ROOT / 'blued_1min_stride30s_model_manifest.csv'
NORMALIZATION_FILE = ROOT / 'blued_feature_normalization.csv'

for path in [
    FEATURE_FILE,
    MANIFEST_FILE,
    NORMALIZATION_FILE
]:
    if not path.is_file():
        raise RuntimeError(
            f'STOP: Missing source: {path}'
        )

print('=' * 100)
print('D7 / C3-4 — BLUED TRAINING-FEATURE VARIANCE')
print('=' * 100)

# ------------------------------------------------------------
# 1. LOAD EXISTING FROZEN FEATURE AND SPLIT RECORDS
# ------------------------------------------------------------

features = pd.read_csv(
    FEATURE_FILE,
    low_memory=False
)

manifest = pd.read_csv(
    MANIFEST_FILE,
    usecols=[
        'model_window_id',
        'split',
        'recording_block',
        'observed_points',
        'expected_points',
        'coverage_ratio'
    ],
    low_memory=False
)

normalization = pd.read_csv(
    NORMALIZATION_FILE,
    low_memory=False
)

feature_cols = [
    c for c in features.columns
    if c != 'model_window_id'
]

if len(feature_cols) != 69:
    raise RuntimeError(
        f'STOP: Expected 69 features, found {len(feature_cols)}'
    )

if features['model_window_id'].isna().any():
    raise RuntimeError(
        'STOP: Missing feature-matrix window identifiers.'
    )

if manifest['model_window_id'].isna().any():
    raise RuntimeError(
        'STOP: Missing manifest window identifiers.'
    )

if features['model_window_id'].duplicated().any():
    raise RuntimeError(
        'STOP: Duplicate feature-matrix window identifiers.'
    )

if manifest['model_window_id'].duplicated().any():
    raise RuntimeError(
        'STOP: Duplicate manifest window identifiers.'
    )

print('\n1. SOURCE RECORDS')
print('Feature-matrix rows:', len(features))
print('Manifest rows:', len(manifest))
print('Engineered features:', len(feature_cols))
print('Feature IDs unique: PASS')
print('Manifest IDs unique: PASS')

# ------------------------------------------------------------
# 2. VERIFY EXACT SPLIT JOIN
# ------------------------------------------------------------

joined = features.merge(
    manifest,
    on='model_window_id',
    how='outer',
    validate='one_to_one',
    indicator=True
)

print('\n2. JOIN VERIFICATION')
print('Join classifications:')
print(
    joined['_merge'].value_counts(
        dropna=False
    ).to_dict()
)

if not joined['_merge'].eq('both').all():
    raise RuntimeError(
        'STOP: Feature and manifest populations '
        'are not identical. Inspect before proceeding.'
    )

joined = joined.drop(columns=['_merge'])

print('\nSplit distribution:')
print(
    joined['split'].astype(str)
    .value_counts(dropna=False)
    .to_dict()
)

valid_splits = {'train', 'validation', 'test'}

actual_splits = set(
    joined['split'].dropna().astype(str)
)

if (
    joined['split'].isna().any()
    or not actual_splits.issubset(valid_splits)
    or 'train' not in actual_splits
    or 'test' not in actual_splits
):
    raise RuntimeError(
        f'STOP: Unexpected split values: {actual_splits}'
    )

train = joined.loc[
    joined['split'].astype(str).eq('train')
].copy()

print('Training windows:', len(train))
print('Non-training windows:',
      len(joined) - len(train))

if len(train) < 100:
    raise RuntimeError(
        'STOP: Unexpectedly small training population.'
    )

print('TRAINING PARTITION: VERIFIED')

# ------------------------------------------------------------
# 3. VERIFY FEATURE TYPES AND FINITE VALUES
# ------------------------------------------------------------

print('\n3. TRAINING FEATURE VALIDITY')

for col in feature_cols:
    if not pd.api.types.is_numeric_dtype(train[col]):
        raise RuntimeError(
            f'STOP: Nonnumeric feature: {col}'
        )

X_train = train[
    feature_cols
].to_numpy(dtype=np.float64)

finite_mask = np.isfinite(X_train)

print('Training matrix shape:', X_train.shape)
print('Nonfinite training feature values:',
      int((~finite_mask).sum()))

if not finite_mask.all():
    raise RuntimeError(
        'STOP: Nonfinite training feature values '
        'require investigation.'
    )

# ------------------------------------------------------------
# 4. DETECT CONSTANT FEATURES
# ------------------------------------------------------------

records = []

for j, col in enumerate(feature_cols):

    values = X_train[:, j]

    first = float(values[0])

    # Exact constant: every stored training value equals
    # the first stored training value.
    is_constant = bool(
        np.all(values == first)
    )

    observed_min = float(np.min(values))
    observed_max = float(np.max(values))
    observed_mean = float(np.mean(values))
    observed_std = float(np.std(values, ddof=0))

    records.append({
        'feature': col,
        'is_constant_exact': is_constant,
        'train_min': observed_min,
        'train_max': observed_max,
        'train_mean': observed_mean,
        'train_std_population': observed_std,
        'train_range':
            observed_max - observed_min
    })

variance = pd.DataFrame(records)

constant_features = variance.loc[
    variance['is_constant_exact'],
    'feature'
].tolist()

print('\n4. CONSTANT-FEATURE RESULTS')
print('Total features:', len(feature_cols))
print('Exactly constant:', len(constant_features))
print('Nonconstant:',
      len(feature_cols) - len(constant_features))

print('\nConstant feature names:')

if constant_features:
    for col in constant_features:
        print('  ', col)
else:
    print('  (none)')

print('\nConstant feature values:')

if constant_features:
    print(
        variance.loc[
            variance['is_constant_exact'],
            [
                'feature',
                'train_min',
                'train_max',
                'train_std_population'
            ]
        ].to_string(index=False)
    )

# ------------------------------------------------------------
# 5. COMPARE SAVED NORMALIZATION METADATA
# ------------------------------------------------------------

print('\n' + '=' * 100)
print('5. SAVED NORMALIZATION METADATA')
print('=' * 100)

required_norm = {
    'feature',
    'training_mean',
    'training_std',
    'constant_in_training'
}

if not required_norm.issubset(
    normalization.columns
):
    raise RuntimeError(
        'STOP: Unexpected normalization metadata schema.'
    )

if (
    normalization['feature'].duplicated().any()
    or set(normalization['feature']) != set(feature_cols)
):
    raise RuntimeError(
        'STOP: Feature normalization schema mismatch.'
    )

raw_flags = (
    normalization['constant_in_training']
    .astype(str)
    .str.strip()
    .str.lower()
)

allowed_flags = {'true', 'false', '1', '0'}

if not set(raw_flags).issubset(allowed_flags):
    raise RuntimeError(
        'STOP: Unrecognized constant feature flags: '
        + str(sorted(set(raw_flags)))
    )

normalization = normalization.copy()

normalization['constant_flag_bool'] = (
    raw_flags.isin({'true', '1'})
)

check = variance.merge(
    normalization[
        [
            'feature',
            'training_mean',
            'training_std',
            'constant_flag_bool'
        ]
    ],
    on='feature',
    validate='one_to_one'
)

check['constant_flag_matches'] = (
    check['is_constant_exact']
    == check['constant_flag_bool']
)

print('Saved metadata constant count:',
      int(check['constant_flag_bool'].sum()))

print('Measured constant count:',
      int(check['is_constant_exact'].sum()))

print('Constant-flag disagreements:',
      int((~check['constant_flag_matches']).sum()))

if not check['constant_flag_matches'].all():

    print('\nDISAGREEMENTS:')
    print(
        check.loc[
            ~check['constant_flag_matches'],
            [
                'feature',
                'is_constant_exact',
                'constant_flag_bool',
                'train_range',
                'training_std'
            ]
        ].to_string(index=False)
    )

    print(
        '\nSTATUS: REVIEW REQUIRED — no source '
        'values will be changed.'
    )

else:
    print('SAVED CONSTANT-FEATURE FLAGS: PASS')

# Diagnostic comparison only.
# No requirement that saved training_std equals
# population std: the original convention may differ.

check['mean_difference'] = abs(
    check['train_mean']
    - check['training_mean']
)

check['std_difference'] = abs(
    check['train_std_population']
    - check['training_std']
)

print('\nMaximum training-mean discrepancy:',
      float(check['mean_difference'].max()))

print('Maximum population-std discrepancy:',
      float(check['std_difference'].max()))

# ------------------------------------------------------------
# 6. INVESTIGATE n_points
# ------------------------------------------------------------

print('\n' + '=' * 100)
print('6. n_points FEATURE AUDIT')
print('=' * 100)

n_points_cols = [
    c for c in feature_cols
    if c.endswith('__n_points')
]

print('n_points features:', n_points_cols)

if len(n_points_cols) != 3:
    raise RuntimeError(
        'STOP: Expected three BLUED n_points features.'
    )

for col in n_points_cols:

    values = train[col]

    print('\nFeature:', col)
    print('  Unique training values:',
          int(values.nunique(dropna=False)))
    print('  Minimum:', float(values.min()))
    print('  Maximum:', float(values.max()))
    print('  Standard deviation:',
          float(values.std(ddof=0)))

print('\nManifest measurement coverage:')
print('Observed-points minimum:',
      train['observed_points'].min())
print('Observed-points maximum:',
      train['observed_points'].max())
print('Coverage-ratio minimum:',
      train['coverage_ratio'].min())
print('Coverage-ratio maximum:',
      train['coverage_ratio'].max())

# ------------------------------------------------------------
# 7. SAVE RESULTS TO NOTEBOOK MEMORY ONLY
# ------------------------------------------------------------

D7_BLUED_TRAIN = train
D7_BLUED_VARIANCE = check
D7_BLUED_CONSTANT_FEATURES = constant_features
D7_BLUED_FEATURE_COLUMNS = feature_cols

print('\n' + '=' * 100)
print('CELL 4 COMPLETE — TRAINING VARIANCE AUDIT')
print('=' * 100)
print('Feature-to-manifest join verified.')
print('Training-only variance analysis completed.')
print('Saved constant-feature metadata checked.')
print('No feature selection or model changes performed.')
print('No files written.')
print('C3-4 remains OPEN.')
print('=' * 100)


In [ ]:

# ============================================================
# D7 / C3-4 — CELL 5
# BLUED CONSTANT FEATURES AND FROZEN MODEL INPUT AUDIT
#
# READ ONLY
# No retraining
# No predictions
# No source files modified
# ============================================================

import numpy as np
import pandas as pd
import joblib
import sklearn
from pathlib import Path

print("=" * 90)
print("D7 / C3-4 — BLUED FINAL INPUT VERIFICATION")
print("=" * 90)

# Reuse the verified Cell 4 join.
if "joined" not in globals():
    raise RuntimeError(
        "STOP: Cell 4 results are unavailable."
    )

if "D7_BLUED_FEATURE_COLUMNS" not in globals():
    raise RuntimeError(
        "STOP: Cell 4 feature list is unavailable."
    )

feature_cols = D7_BLUED_FEATURE_COLUMNS

n_points_cols = [
    "mean_power__n_points",
    "max_power__n_points",
    "std_power__n_points"
]

# ------------------------------------------------------------
# 1. VERIFY n_points ACROSS ALL THREE SPLITS
# ------------------------------------------------------------

print("\n1. n_points ACROSS TRAIN / VALIDATION / TEST")

for feature in n_points_cols:

    print("\nFeature:", feature)

    summary = joined.groupby(
        "split"
    )[feature].agg(
        ["count", "nunique", "min", "max"]
    )

    print(summary.to_string())

# ------------------------------------------------------------
# 2. VERIFY STORED NORMALIZATION CONVENTION
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("2. SAVED NORMALIZATION VALUES")
print("=" * 90)

norm = pd.read_csv(
    "/content/drive/MyDrive/Master/Thesis/Code/"
    "experiments/EXP_BLUED_FIVE_MODEL_LABEL_BLIND/"
    "common/blued_feature_normalization.csv"
)

selected_norm = norm.loc[
    norm["feature"].isin(n_points_cols),
    [
        "feature",
        "training_mean",
        "training_std",
        "constant_in_training"
    ]
]

print(selected_norm.to_string(index=False))

if len(selected_norm) != 3:
    raise RuntimeError(
        "STOP: Incomplete normalization metadata."
    )

# Diagnostic only: use stored training parameters.
# This does not alter the frozen model inputs.
for _, row in selected_norm.iterrows():

    feature = row["feature"]
    mean = float(row["training_mean"])
    std = float(row["training_std"])

    if std <= 0:
        raise RuntimeError(
            f"STOP: Invalid saved scale for {feature}"
        )

    values = joined[feature].to_numpy(
        dtype=np.float64
    )

    normalized = (values - mean) / std

    print(
        f"{feature}: "
        f"max_abs_normalized="
        f"{np.max(np.abs(normalized)):.12g}"
    )

# ------------------------------------------------------------
# 3. INSPECT SAVED CLASSICAL ESTIMATOR DIMENSIONS
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("3. FROZEN MODEL INPUT DIMENSIONS")
print("=" * 90)

print("Current sklearn version:", sklearn.__version__)
print("Frozen feature matrix width:", len(feature_cols))

model_root = Path(
    "/content/drive/MyDrive/Master/Thesis/Code/"
    "experiments/EXP_BLUED_FIVE_MODEL_LABEL_BLIND/"
    "controlled_feature_models"
)

models = {
    "Isolation Forest": "isolation_forest",
    "Local Outlier Factor": "local_outlier_factor",
    "One-Class SVM": "one_class_svm"
}

model_results = {}

def locate_estimator(obj):
    """
    Inspect common saved-model wrappers.
    Do not fit or predict.
    """
    if hasattr(obj, "n_features_in_"):
        return obj

    if isinstance(obj, dict):
        for key in [
            "model",
            "estimator",
            "classifier",
            "detector",
            "pipeline"
        ]:
            if key in obj:
                result = locate_estimator(obj[key])
                if result is not None:
                    return result

    if hasattr(obj, "steps"):
        for _, step in reversed(obj.steps):
            if hasattr(step, "n_features_in_"):
                return step

    return None

for display_name, folder in models.items():

    path = model_root / folder / "model.joblib"

    print("\nModel:", display_name)

    if not path.is_file():
        print("STATUS: MODEL FILE MISSING")
        model_results[display_name] = None
        continue

    # Load only the user's own existing trusted
    # thesis artifacts. joblib files use pickle.
    try:
        saved_object = joblib.load(path)
    except Exception as exc:
        print(
            "STATUS: COULD NOT INSPECT",
            type(exc).__name__,
            str(exc)[:250]
        )
        model_results[display_name] = None
        continue

    estimator = locate_estimator(saved_object)

    print(
        "Saved object type:",
        type(saved_object).__name__
    )

    if estimator is None:
        print("Input dimension: NOT DETERMINED")
        model_results[display_name] = None
        continue

    n_features = int(estimator.n_features_in_)

    model_results[display_name] = n_features

    print("Stored input features:", n_features)

    if n_features == 69:
        print("STATUS: 69-DIMENSION INPUT CONFIRMED")
    elif n_features == 66:
        print("STATUS: 66-DIMENSION INPUT CONFIRMED")
    else:
        print("STATUS: UNEXPECTED INPUT DIMENSION")

    if hasattr(estimator, "feature_names_in_"):

        saved_names = list(
            estimator.feature_names_in_
        )

        print(
            "Saved feature names:",
            len(saved_names)
        )

        print(
            "Names match feature CSV order:",
            saved_names == feature_cols
        )

# ------------------------------------------------------------
# 4. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("4. AUDIT SUMMARY")
print("=" * 90)

print("Expected BLUED features:", len(feature_cols))
print("Constant training features:", len(n_points_cols))

print("\nSaved estimator dimensions:")
for name, dimension in model_results.items():
    print(f"  {name}: {dimension}")

print("\nNo model retrained.")
print("No model predictions generated.")
print("No source files modified.")
print("C3-4 remains OPEN.")
print("=" * 90)


In [ ]:

# ============================================================
# D7 / C3-4 — CELL 6
# RESIDENTIAL TRAINING-ONLY FEATURE CORRELATION AUDIT
#
# READ ONLY
# No model fitting
# No feature removal
# No files written
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

EXP_ROOT = Path(
    '/content/drive/MyDrive/Master/Thesis/'
    'Code/experiments'
)

SOURCES = {
    'UK-DALE': EXP_ROOT / (
        'EXP_SPLIT_BOUNDARY_PURGE_CORRECTION/'
        'UKDALE/ukdale_if_scored_boundary_corrected.csv'
    ),
    'AMPds2': EXP_ROOT / (
        'EXP_SPLIT_BOUNDARY_PURGE_CORRECTION/'
        'AMPDS2/ampds2_if_scored_boundary_corrected.csv'
    )
}

FEATURES = [
    'energy', 'first_last_diff', 'iqr', 'kurtosis',
    'mad', 'max', 'max_abs_diff', 'mean',
    'mean_abs_diff', 'median', 'min', 'n_points',
    'peak_count', 'q25', 'q75', 'range',
    'rms', 'skew', 'slope', 'std',
    'std_diff', 'sum', 'var'
]

EXPECTED_TRAIN = {
    'UK-DALE': 218358,
    'AMPds2': 441483
}

D7_CORRELATION_RESULTS = {}

print('=' * 95)
print('D7 / C3-4 — RESIDENTIAL FEATURE CORRELATION')
print('=' * 95)

for dataset, path in SOURCES.items():

    print('\n' + '=' * 95)
    print('DATASET:', dataset)
    print('=' * 95)

    if not path.is_file():
        raise RuntimeError(
            f'STOP: Missing frozen source: {path}'
        )

    z_features = [
        f'{feature}_z'
        for feature in FEATURES
    ]

    columns = [
        'split', 'series_key', 'n_points',
        *z_features
    ]

    if len(columns) != len(set(columns)):
        raise RuntimeError(
            'STOP: Duplicate requested column names.'
        )

    # Load only the columns necessary for this audit.
    df = pd.read_csv(
        path,
        usecols=columns,
        low_memory=False
    )

    split_names = (
        df['split']
        .astype(str)
        .str.strip()
        .str.lower()
    )

    print('\n1. SOURCE AND SPLIT VALIDATION')
    print('Source rows:', len(df))
    print('Split distribution:')
    print(split_names.value_counts().to_dict())

    allowed = {
        'train', 'training',
        'validation', 'val', 'test'
    }

    if not set(split_names.unique()).issubset(allowed):
        raise RuntimeError(
            'STOP: Unexpected split labels.'
        )

    train = df.loc[
        split_names.isin({'train', 'training'})
    ].copy()

    actual = len(train)
    expected = EXPECTED_TRAIN[dataset]

    print('Training rows:', actual)
    print('Expected rows:', expected)

    if actual != expected:
        raise RuntimeError(
            f'STOP: Training population mismatch '
            f'for {dataset}.'
        )

    if train['series_key'].isna().any():
        raise RuntimeError(
            'STOP: Missing training series identifiers.'
        )

    print(
        'Training series:',
        train['series_key'].nunique()
    )

    print('TRAINING POPULATION: PASS')

    # --------------------------------------------------------
    # 2. FEATURE MATRIX VALIDATION
    # --------------------------------------------------------

    print('\n2. NUMERIC FEATURE VALIDATION')

    X = train[z_features].copy()

    X.columns = FEATURES

    if not all(
        pd.api.types.is_numeric_dtype(X[c])
        for c in FEATURES
    ):
        raise RuntimeError(
            'STOP: Nonnumeric feature detected.'
        )

    if not np.isfinite(
        X.to_numpy(dtype=np.float64)
    ).all():
        raise RuntimeError(
            'STOP: Nonfinite training feature detected.'
        )

    print('Feature matrix:', X.shape)
    print('Nonfinite values: 0')
    print('FEATURE MATRIX: PASS')

    # --------------------------------------------------------
    # 3. CONSTANT FEATURE ANALYSIS
    # --------------------------------------------------------

    print('\n3. FEATURE VARIABILITY')

    minima = X.min(axis=0)
    maxima = X.max(axis=0)

    constant_features = [
        feature
        for feature in FEATURES
        if minima[feature] == maxima[feature]
    ]

    variable_features = [
        feature
        for feature in FEATURES
        if feature not in constant_features
    ]

    print('Total features:', len(FEATURES))
    print('Constant features:', len(constant_features))
    print('Variable features:', len(variable_features))
    print('Constant names:', constant_features)

    if len(variable_features) < 2:
        raise RuntimeError(
            'STOP: Insufficient variable features.'
        )

    # --------------------------------------------------------
    # 4. n_points AUDIT
    # --------------------------------------------------------

    print('\n4. n_points COVERAGE AUDIT')

    n = train['n_points']

    if not pd.api.types.is_numeric_dtype(n):
        raise RuntimeError(
            'STOP: n_points is not numeric.'
        )

    if not np.isfinite(
        n.to_numpy(dtype=np.float64)
    ).all():
        raise RuntimeError(
            'STOP: Nonfinite n_points values.'
        )

    print('Training minimum:', n.min())
    print('Training maximum:', n.max())
    print('Unique values:', n.nunique())

    print('\nMost frequent n_points values:')
    print(
        n.value_counts()
        .head(10)
        .to_string()
    )

    per_series = (
        train.groupby('series_key')['n_points']
        .agg(['count', 'nunique', 'min', 'max'])
    )

    variable_series = per_series.loc[
        per_series['nunique'] > 1
    ]

    print('\nNumber of series:',
          len(per_series))

    print('Series with variable n_points:',
          len(variable_series))

    print('\nExample per-series coverage:')
    print(
        per_series.head(12).to_string()
    )

    if len(variable_series):
        print('\nSeries with variable coverage:')
        print(
            variable_series.head(15).to_string()
        )

    # --------------------------------------------------------
    # 5. PEARSON AND SPEARMAN CORRELATIONS
    # --------------------------------------------------------

    print('\n5. CORRELATION ANALYSIS')

    X_var = X[variable_features]

    pearson = X_var.corr(method='pearson')
    spearman = X_var.corr(method='spearman')

    if (
        pearson.isna().any().any()
        or spearman.isna().any().any()
    ):
        raise RuntimeError(
            'STOP: Undefined correlation detected.'
        )

    pairs = []

    for i, first in enumerate(variable_features):

        for second in variable_features[i + 1:]:

            r = float(
                pearson.loc[first, second]
            )

            rho = float(
                spearman.loc[first, second]
            )

            pairs.append({
                'feature_1': first,
                'feature_2': second,
                'pearson_r': r,
                'spearman_rho': rho,
                'abs_pearson': abs(r),
                'abs_spearman': abs(rho)
            })

    pair_df = pd.DataFrame(pairs)

    pair_df = pair_df.sort_values(
        ['abs_pearson', 'feature_1', 'feature_2'],
        ascending=[False, True, True]
    ).reset_index(drop=True)

    print('Total comparable feature pairs:',
          len(pair_df))

    for threshold in [0.90, 0.95, 0.99]:

        pearson_count = int(
            (pair_df['abs_pearson'] >= threshold).sum()
        )

        spearman_count = int(
            (pair_df['abs_spearman'] >= threshold).sum()
        )

        print(
            f'Absolute correlation >= {threshold:.2f}: '
            f'Pearson={pearson_count}, '
            f'Spearman={spearman_count}'
        )

    print('\nTOP 20 PAIRS — ABSOLUTE PEARSON:')
    print(
        pair_df[
            [
                'feature_1',
                'feature_2',
                'pearson_r',
                'spearman_rho'
            ]
        ].head(20).to_string(
            index=False,
            float_format=lambda v: f'{v:.6f}'
        )
    )

    top_spearman = (
        pair_df.sort_values(
            'abs_spearman',
            ascending=False
        )
    )

    print('\nTOP 10 PAIRS — ABSOLUTE SPEARMAN:')
    print(
        top_spearman[
            [
                'feature_1',
                'feature_2',
                'pearson_r',
                'spearman_rho'
            ]
        ].head(10).to_string(
            index=False,
            float_format=lambda v: f'{v:.6f}'
        )
    )

    # --------------------------------------------------------
    # 6. SPECIFIC CONSUMPTION-LEVEL COMPARISONS
    # --------------------------------------------------------

    print('\n6. CONSUMPTION-LEVEL FEATURE GROUP')

    level_features = [
        feature for feature in [
            'energy', 'sum', 'mean', 'rms',
            'median', 'q25', 'q75'
        ]
        if feature in variable_features
    ]

    print('Available level features:',
          level_features)

    print('\nPearson matrix:')
    print(
        pearson.loc[
            level_features,
            level_features
        ].round(4).to_string()
    )

    # Preserve in notebook memory, without writing files.
    D7_CORRELATION_RESULTS[dataset] = {
        'pearson': pearson,
        'spearman': spearman,
        'pairs': pair_df,
        'constant_features': constant_features,
        'variable_features': variable_features,
        'n_points_by_series': per_series,
        'training_rows': actual,
        'training_series': train['series_key'].nunique()
    }

    print('\nDATASET AUDIT COMPLETE:', dataset)

print('\n' + '=' * 95)
print('CELL 6 COMPLETE — TRAINING-ONLY CORRELATION AUDIT')
print('=' * 95)
print('Both residential training populations verified.')
print('Pearson and Spearman correlations calculated.')
print('Feature redundancy and n_points assessed.')
print('Results retained in notebook memory only.')
print('No model fitting or prediction.')
print('No files written or modified.')
print('C3-4 remains OPEN.')
print('=' * 95)


In [ ]:

# ============================================================
# D7 / C3-4 — CELL 7
# WITHIN-SERIES CORRELATIONS AND FEATURE HEATMAPS
#
# Frozen residential training populations only
# No model fitting or predictions
# Writes only NEW D7 evidence files
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 95)
print("D7 / C3-4 — WITHIN-SERIES FEATURE DEPENDENCE")
print("=" * 95)

if "D7_CORRELATION_RESULTS" not in globals():
    raise RuntimeError(
        "STOP: Cell 6 results are unavailable."
    )

EXP_ROOT = Path(
    "/content/drive/MyDrive/Master/Thesis/Code/experiments"
)

SOURCES = {
    "UK-DALE": (
        EXP_ROOT
        / "EXP_SPLIT_BOUNDARY_PURGE_CORRECTION"
        / "UKDALE"
        / "ukdale_if_scored_boundary_corrected.csv"
    ),
    "AMPds2": (
        EXP_ROOT
        / "EXP_SPLIT_BOUNDARY_PURGE_CORRECTION"
        / "AMPDS2"
        / "ampds2_if_scored_boundary_corrected.csv"
    )
}

EXPECTED = {
    "UK-DALE": {
        "train": 218358,
        "series": 16
    },
    "AMPds2": {
        "train": 441483,
        "series": 21
    }
}

BASE_FEATURES = [
    "energy", "first_last_diff", "iqr",
    "kurtosis", "mad", "max",
    "max_abs_diff", "mean", "mean_abs_diff",
    "median", "min", "n_points",
    "peak_count", "q25", "q75",
    "range", "rms", "skew", "slope",
    "std", "std_diff", "sum", "var"
]

KEY_PAIRS = [
    ("energy", "mean"),
    ("energy", "sum"),
    ("mean", "sum"),
    ("mean", "rms"),
    ("std", "var"),
    ("max", "range"),
    ("max_abs_diff", "std_diff"),
    ("mean_abs_diff", "std_diff")
]

# ------------------------------------------------------------
# 1. VERIFY ALL INPUTS BEFORE WRITING
# ------------------------------------------------------------

for dataset, path in SOURCES.items():

    if not path.is_file():
        raise RuntimeError(
            f"STOP: Missing source for {dataset}: {path}"
        )

    previous = D7_CORRELATION_RESULTS[dataset]

    if previous["training_rows"] != EXPECTED[dataset]["train"]:
        raise RuntimeError(
            f"STOP: Historical train count mismatch: {dataset}"
        )

    if previous["training_series"] != EXPECTED[dataset]["series"]:
        raise RuntimeError(
            f"STOP: Historical series count mismatch: {dataset}"
        )

    if previous["constant_features"] != ["n_points"]:
        raise RuntimeError(
            f"STOP: Unexpected constant features: {dataset}"
        )

print("\nPRE-WRITE INPUT CHECKS: PASS")

# ------------------------------------------------------------
# 2. LOAD TRAINING DATA AND VERIFY CORRELATIONS
# ------------------------------------------------------------

audit = {}

for dataset, path in SOURCES.items():

    print("\n" + "=" * 95)
    print("DATASET:", dataset)
    print("=" * 95)

    z_cols = [
        feature + "_z"
        for feature in BASE_FEATURES
    ]

    df = pd.read_csv(
        path,
        usecols=["split", "series_key", *z_cols],
        low_memory=False
    )

    train = df.loc[
        df["split"].astype(str).eq("train")
    ].copy()

    del df

    if len(train) != EXPECTED[dataset]["train"]:
        raise RuntimeError(
            f"STOP: Train count mismatch for {dataset}"
        )

    if train["series_key"].nunique() != EXPECTED[dataset]["series"]:
        raise RuntimeError(
            f"STOP: Series count mismatch for {dataset}"
        )

    train = train.rename(
        columns={
            f + "_z": f
            for f in BASE_FEATURES
        }
    )

    variable_features = (
        D7_CORRELATION_RESULTS[dataset][
            "variable_features"
        ]
    )

    X = train[variable_features]

    if not np.isfinite(
        X.to_numpy(dtype=np.float64)
    ).all():
        raise RuntimeError(
            f"STOP: Nonfinite feature values: {dataset}"
        )

    # Independent confirmation of Cell 6.
    pearson = X.corr(method="pearson")
    spearman = X.corr(method="spearman")

    previous_p = D7_CORRELATION_RESULTS[
        dataset
    ]["pearson"]

    previous_s = D7_CORRELATION_RESULTS[
        dataset
    ]["spearman"]

    max_pearson_diff = float(
        np.max(np.abs(
            pearson.to_numpy()
            - previous_p.to_numpy()
        ))
    )

    max_spearman_diff = float(
        np.max(np.abs(
            spearman.to_numpy()
            - previous_s.to_numpy()
        ))
    )

    print("\n1. POOLED CORRELATION REPRODUCTION")

    print(
        "Max Pearson matrix discrepancy:",
        f"{max_pearson_diff:.12g}"
    )

    print(
        "Max Spearman matrix discrepancy:",
        f"{max_spearman_diff:.12g}"
    )

    if (
        max_pearson_diff > 1e-10
        or max_spearman_diff > 1e-10
    ):
        raise RuntimeError(
            f"STOP: Correlation reproduction failed: {dataset}"
        )

    print("POOLED CORRELATIONS: PASS")

    # --------------------------------------------------------
    # 3. WITHIN-SERIES CORRELATIONS
    # --------------------------------------------------------

    print("\n2. WITHIN-SERIES ANALYSIS")

    records = []

    for series_key, group in train.groupby(
        "series_key", sort=True
    ):

        # Series-specific correlation matrix.
        local_corr = group[
            variable_features
        ].corr(method="pearson")

        for first, second in KEY_PAIRS:

            r = float(
                local_corr.loc[first, second]
            )

            records.append({
                "dataset": dataset,
                "series_key": str(series_key),
                "training_windows": len(group),
                "feature_1": first,
                "feature_2": second,
                "within_series_pearson_r": r
            })

    within_series = pd.DataFrame(records)

    summaries = []

    for first, second in KEY_PAIRS:

        subset = within_series.loc[
            (within_series["feature_1"] == first)
            & (within_series["feature_2"] == second)
        ]

        valid = subset[
            "within_series_pearson_r"
        ].dropna()

        summaries.append({
            "dataset": dataset,
            "feature_1": first,
            "feature_2": second,
            "pooled_pearson_r": float(
                pearson.loc[first, second]
            ),
            "eligible_series": len(valid),
            "total_series": EXPECTED[dataset]["series"],
            "median_within_series_r": (
                float(valid.median())
                if len(valid) else np.nan
            ),
            "min_within_series_r": (
                float(valid.min())
                if len(valid) else np.nan
            ),
            "max_within_series_r": (
                float(valid.max())
                if len(valid) else np.nan
            ),
            "series_with_abs_r_ge_095": int(
                (valid.abs() >= 0.95).sum()
            )
        })

    summary = pd.DataFrame(summaries)

    print(
        summary[
            [
                "feature_1",
                "feature_2",
                "pooled_pearson_r",
                "eligible_series",
                "median_within_series_r",
                "series_with_abs_r_ge_095"
            ]
        ].to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}"
        )
    )

    print(
        "\nUndefined within-series correlations "
        "are excluded from the median and "
        "eligible-series counts."
    )

    audit[dataset] = {
        "pearson": pearson,
        "spearman": spearman,
        "within_series": within_series,
        "summary": summary,
        "pairs": D7_CORRELATION_RESULTS[
            dataset
        ]["pairs"],
        "training_rows": len(train),
        "series_count": train["series_key"].nunique()
    }

    print("\nDATASET VERIFICATION: PASS")

# ------------------------------------------------------------
# 4. CREATE NEW EVIDENCE DIRECTORY
# ------------------------------------------------------------

timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ")

OUTPUT = (
    EXP_ROOT
    / f"D7_C3_4_FEATURE_DEPENDENCE_{timestamp}"
)

if OUTPUT.exists():
    raise RuntimeError(
        "STOP: Output directory already exists."
    )

OUTPUT.mkdir(parents=True, exist_ok=False)

print("\n" + "=" * 95)
print("3. SAVING D7 EVIDENCE")
print("=" * 95)

print("Output folder:", OUTPUT)

# ------------------------------------------------------------
# 5. GENERATE PEARSON AND SPEARMAN HEATMAPS
# ------------------------------------------------------------

for dataset, result in audit.items():

    filename_prefix = (
        "ukdale"
        if dataset == "UK-DALE"
        else "ampds2"
    )

    result["pearson"].to_csv(
        OUTPUT / f"{filename_prefix}_pearson_matrix.csv"
    )

    result["spearman"].to_csv(
        OUTPUT / f"{filename_prefix}_spearman_matrix.csv"
    )

    result["pairs"].to_csv(
        OUTPUT / f"{filename_prefix}_feature_pairs.csv",
        index=False
    )

    result["within_series"].to_csv(
        OUTPUT / f"{filename_prefix}_within_series_pairs.csv",
        index=False
    )

    result["summary"].to_csv(
        OUTPUT / f"{filename_prefix}_within_series_summary.csv",
        index=False
    )

    for method in ["pearson", "spearman"]:

        matrix = result[method]
        labels = matrix.columns.tolist()

        fig, ax = plt.subplots(
            figsize=(11.5, 10.5)
        )

        plot = ax.imshow(
            matrix.to_numpy(),
            cmap="coolwarm",
            vmin=-1,
            vmax=1,
            interpolation="nearest"
        )

        ax.set_xticks(
            np.arange(len(labels))
        )

        ax.set_yticks(
            np.arange(len(labels))
        )

        ax.set_xticklabels(
            labels,
            rotation=60,
            ha="right",
            fontsize=9
        )

        ax.set_yticklabels(
            labels,
            fontsize=9
        )

        ax.set_title(
            f"{dataset} — Training Feature Correlation\n"
            f"{method.capitalize()} | "
            f"{result['training_rows']:,} windows | "
            f"{result['series_count']} series",
            fontsize=13,
            pad=17
        )

        colorbar = fig.colorbar(
            plot,
            ax=ax,
            shrink=0.80,
            pad=0.03
        )

        colorbar.set_label(
            "Correlation coefficient",
            fontsize=10
        )

        fig.text(
            0.5,
            0.01,
            "Training-only, series-normalized features. "
            "Constant n_points excluded.",
            ha="center",
            fontsize=9
        )

        fig.tight_layout(
            rect=[0, 0.035, 1, 1]
        )

        outpath = (
            OUTPUT
            / f"{filename_prefix}_{method}_heatmap.png"
        )

        fig.savefig(
            outpath,
            dpi=300,
            bbox_inches="tight"
        )

        print(
            "\nSaved:",
            outpath.name
        )

        plt.show()
        plt.close(fig)

# ------------------------------------------------------------
# 6. PRESERVE AN AUDIT MANIFEST
# ------------------------------------------------------------

manifest = {
    "audit": "D7_C3_4_feature_dependence",
    "created_utc": timestamp,
    "data_partition": "train_only",
    "feature_representation": "23_original_features",
    "correlation_features": 22,
    "excluded_constant_feature": "n_points",
    "correlation_methods": [
        "pearson",
        "spearman"
    ],
    "model_refitted": False,
    "historical_data_modified": False,
    "datasets": {
        dataset: {
            "source_csv": str(SOURCES[dataset]),
            "training_windows": result["training_rows"],
            "series_count": result["series_count"]
        }
        for dataset, result in audit.items()
    }
}

with open(
    OUTPUT / "d7_correlation_manifest.json",
    "w",
    encoding="utf-8"
) as handle:
    json.dump(
        manifest,
        handle,
        indent=2
    )

# ------------------------------------------------------------
# 7. FINAL VERIFICATION
# ------------------------------------------------------------

expected_files = 15

generated = sorted(
    p for p in OUTPUT.iterdir()
    if p.is_file()
)

print("\n" + "=" * 95)
print("CELL 7 — FINAL VERIFICATION")
print("=" * 95)

print("Generated files:", len(generated))
print("Expected files:", expected_files)

for path in generated:
    print(
        f"{path.name} | {path.stat().st_size:,} bytes"
    )

if len(generated) != expected_files:
    raise RuntimeError(
        "STOP: Evidence file count mismatch."
    )

if any(p.stat().st_size == 0 for p in generated):
    raise RuntimeError(
        "STOP: Empty output file detected."
    )

D7_OUTPUT_DIR = OUTPUT
D7_WITHIN_SERIES_AUDIT = audit

print("\nCELL 7: PASSED")
print("Within-series correlations calculated.")
print("Four correlation heatmaps saved.")
print("Pooled matrices and supporting CSVs saved.")
print("D7 evidence manifest preserved.")
print("No historical experiments modified.")
print("No models retrained.")
print("C3-4 remains OPEN pending interpretation.")
print("=" * 95)


In [ ]:

# ============================================================
# D7 / C3-4 — CELL 8
# FINAL CORRELATION EVIDENCE VERIFICATION
# READ ONLY — NO FILES MODIFIED
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
from PIL import Image

OUTPUT = Path(
    "/content/drive/MyDrive/Master/Thesis/Code/"
    "experiments/"
    "D7_C3_4_FEATURE_DEPENDENCE_20261107T115204Z"
)

print("=" * 85)
print("D7 / C3-4 — FINAL EVIDENCE VERIFICATION")
print("=" * 85)

if not OUTPUT.is_dir():
    raise RuntimeError("STOP: D7 evidence folder missing.")

expected = {"d7_correlation_manifest.json"}

for prefix in ["ukdale", "ampds2"]:
    expected.update({
        f"{prefix}_pearson_matrix.csv",
        f"{prefix}_spearman_matrix.csv",
        f"{prefix}_feature_pairs.csv",
        f"{prefix}_within_series_pairs.csv",
        f"{prefix}_within_series_summary.csv",
        f"{prefix}_pearson_heatmap.png",
        f"{prefix}_spearman_heatmap.png",
    })

actual = {
    p.name for p in OUTPUT.iterdir()
    if p.is_file()
}

print("\nExpected files:", len(expected))
print("Actual files:", len(actual))
print("Missing:", sorted(expected - actual))
print("Unexpected:", sorted(actual - expected))

if actual != expected:
    raise RuntimeError("STOP: Evidence inventory mismatch.")

with open(
    OUTPUT / "d7_correlation_manifest.json",
    encoding="utf-8"
) as f:
    manifest = json.load(f)

assert manifest["data_partition"] == "train_only"
assert manifest["model_refitted"] is False
assert manifest["historical_data_modified"] is False

for prefix, expected_rows, expected_series in [
    ("ukdale", 218358, 16),
    ("ampds2", 441483, 21),
]:
    print(f"\nDATASET: {prefix.upper()}")

    matrices = {}

    for method in ["pearson", "spearman"]:
        path = OUTPUT / f"{prefix}_{method}_matrix.csv"
        matrix = pd.read_csv(path, index_col=0)

        assert matrix.shape == (22, 22)
        assert list(matrix.index) == list(matrix.columns)
        assert np.isfinite(matrix.to_numpy()).all()
        assert np.allclose(
            matrix.to_numpy(),
            matrix.to_numpy().T,
            atol=1e-10
        )
        assert np.allclose(
            np.diag(matrix), 1.0, atol=1e-10
        )

        matrices[method] = matrix

        image_path = OUTPUT / f"{prefix}_{method}_heatmap.png"

        with Image.open(image_path) as im:
            width, height = im.size
            im.verify()

        assert width > 1000 and height > 1000

        print(
            f"{method.capitalize()}: "
            f"22x22 matrix PASS; "
            f"PNG {width}x{height} PASS"
        )

    pearson = matrices["pearson"]

    for a, b in [
        ("energy", "mean"),
        ("energy", "sum"),
        ("mean", "sum")
    ]:
        assert np.isclose(
            pearson.loc[a, b], 1.0, atol=1e-10
        )

    pairs = pd.read_csv(
        OUTPUT / f"{prefix}_feature_pairs.csv"
    )

    local = pd.read_csv(
        OUTPUT / f"{prefix}_within_series_pairs.csv"
    )

    summary = pd.read_csv(
        OUTPUT / f"{prefix}_within_series_summary.csv"
    )

    assert len(pairs) == 231
    assert len(local) == expected_series * 8
    assert len(summary) == 8

    print("Feature pairs:", len(pairs), "PASS")
    print("Within-series records:", len(local), "PASS")
    print("Within-series summaries:", len(summary), "PASS")
    print("Training windows expected:", expected_rows)
    print("All dataset checks: PASS")

print("\n" + "=" * 85)
print("CELL 8: PASSED — D7 CORRELATION EVIDENCE COMPLETE")
print("15 files verified.")
print("Four heatmaps verified.")
print("Pooled and within-series results verified.")
print("Historical experiments unchanged.")
print("No model retraining performed.")
print("C3-4 remains OPEN pending manuscript integration.")
print("=" * 85)


In [ ]:

# ============================================================
# D7 / C3-4 — CELL 9
# BLUED FROZEN CONFIGURATION AUDIT
#
# READ ONLY
# No model fitting or prediction
# No files written or modified
# ============================================================

from pathlib import Path
import pandas as pd
import joblib
import sklearn

ROOT = Path(
    "/content/drive/MyDrive/Master/Thesis/Code/"
    "experiments/EXP_BLUED_FIVE_MODEL_LABEL_BLIND"
)

COMMON = ROOT / "common"
MODELS = ROOT / "controlled_feature_models"

print("=" * 90)
print("D7 / C3-4 — BLUED FROZEN CONFIGURATION AUDIT")
print("=" * 90)
print("scikit-learn:", sklearn.__version__)

# ------------------------------------------------------------
# 1. VERIFY ORIGINAL AND REDUCED FEATURE NAMES
# ------------------------------------------------------------

feature_path = COMMON / "blued_features_1min_69features.csv"
norm_path = COMMON / "blued_feature_normalization.csv"
manifest_path = COMMON / "blued_1min_stride30s_model_manifest.csv"

for path in [feature_path, norm_path, manifest_path]:
    if not path.is_file():
        raise RuntimeError(f"STOP: Missing {path}")

original = [
    c for c in pd.read_csv(
        feature_path, nrows=0
    ).columns
    if c != "model_window_id"
]

constant = [
    "mean_power__n_points",
    "max_power__n_points",
    "std_power__n_points"
]

reduced = [
    c for c in original
    if c not in constant
]

assert len(original) == 69
assert len(reduced) == 66
assert all(c in original for c in constant)

normalization = pd.read_csv(norm_path)

assert normalization["feature"].tolist() == original

manifest = pd.read_csv(
    manifest_path,
    usecols=["model_window_id", "split"]
)

assert manifest["model_window_id"].is_unique

split_counts = manifest["split"].value_counts().to_dict()

assert split_counts == {
    "train": 12248,
    "validation": 4079,
    "test": 4079
}

print("\n1. FEATURE REPRESENTATION")
print("Original dimensions:", len(original))
print("Reduced dimensions:", len(reduced))
print("Removed features:", constant)
print("Split distribution:", split_counts)
print("SOURCE VALIDATION: PASS")

# ------------------------------------------------------------
# 2. INSPECT SAVED MODEL CONFIGURATIONS
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("2. HISTORICAL MODEL CONFIGURATIONS")
print("=" * 90)

model_folders = {
    "Isolation Forest": "isolation_forest",
    "Local Outlier Factor": "local_outlier_factor",
    "One-Class SVM": "one_class_svm"
}

D7_FROZEN_CONFIGS = {}

for name, folder in model_folders.items():

    model_path = MODELS / folder / "model.joblib"
    scores_path = MODELS / folder / "test_window_scores.csv"

    if not model_path.is_file():
        raise RuntimeError(
            f"STOP: Missing model: {model_path}"
        )

    if not scores_path.is_file():
        raise RuntimeError(
            f"STOP: Missing scores: {scores_path}"
        )

    # Load only the user's own trusted model artifacts.
    model = joblib.load(model_path)

    if not hasattr(model, "get_params"):
        raise RuntimeError(
            f"STOP: Unexpected model object: {name}"
        )

    n_features = getattr(model, "n_features_in_", None)

    if n_features != 69:
        raise RuntimeError(
            f"STOP: Unexpected input dimension for {name}"
        )

    params = model.get_params(deep=False)

    scores = pd.read_csv(scores_path)

    required = {
        "model_window_id",
        "anomaly_score",
        "threshold",
        "flagged",
        "event_window_label"
    }

    if not required.issubset(scores.columns):
        raise RuntimeError(
            f"STOP: Unexpected score schema for {name}"
        )

    if scores["model_window_id"].duplicated().any():
        raise RuntimeError(
            f"STOP: Duplicate test IDs for {name}"
        )

    if len(scores) != 4079:
        raise RuntimeError(
            f"STOP: Test population mismatch for {name}"
        )

    D7_FROZEN_CONFIGS[name] = {
        "folder": folder,
        "model_type": type(model).__name__,
        "parameters": params,
        "test_windows": len(scores)
    }

    print("\nMODEL:", name)
    print("Saved class:", type(model).__name__)
    print("Input features:", n_features)
    print("Test score rows:", len(scores))
    print("Saved parameters:")

    for key, value in sorted(params.items()):
        print(f"  {key}: {value}")

    print("MODEL METADATA: PASS")

# ------------------------------------------------------------
# 3. FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("CELL 9: PASSED — CONFIGURATION AUDIT")
print("Three historical classical models inspected.")
print("Original and reduced feature schemas identified.")
print("Historical score populations verified.")
print("No models retrained or predictions generated.")
print("No files written or modified.")
print("C3-4 remains OPEN.")
print("=" * 90)


In [ ]:

# ============================================================
# D7 / C3-4 — CELL 10
# BLUED HISTORICAL 69-FEATURE SCORE REPRODUCTION
#
# READ ONLY
# No model fitting
# No feature removal from historical inputs
# No files written or modified
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import sklearn

ROOT = Path(
    "/content/drive/MyDrive/Master/Thesis/Code/"
    "experiments/EXP_BLUED_FIVE_MODEL_LABEL_BLIND"
)

COMMON = ROOT / "common"
MODELS = ROOT / "controlled_feature_models"

print("=" * 95)
print("D7 / C3-4 — HISTORICAL SCORE REPRODUCTION")
print("=" * 95)
print("scikit-learn:", sklearn.__version__)

# ------------------------------------------------------------
# 1. LOAD HISTORICAL FEATURE AND NORMALIZATION RECORDS
# ------------------------------------------------------------

features = pd.read_csv(
    COMMON / "blued_features_1min_69features.csv"
)

norm = pd.read_csv(
    COMMON / "blued_feature_normalization.csv"
)

manifest = pd.read_csv(
    COMMON / "blued_1min_stride30s_model_manifest.csv",
    usecols=["model_window_id", "split"]
)

feature_cols = [
    c for c in features.columns
    if c != "model_window_id"
]

assert len(feature_cols) == 69
assert features["model_window_id"].is_unique
assert manifest["model_window_id"].is_unique
assert norm["feature"].is_unique

norm = norm.set_index("feature").loc[feature_cols]

means = norm["training_mean"].to_numpy(
    dtype=np.float64
)

scales = norm["training_std"].to_numpy(
    dtype=np.float64
)

assert np.isfinite(means).all()
assert np.isfinite(scales).all()
assert np.all(scales > 0)

# Join the frozen split information.
data = features.merge(
    manifest,
    on="model_window_id",
    validate="one_to_one",
    indicator=True
)

assert len(data) == 20406
assert data["_merge"].eq("both").all()

print("\n1. INPUT POPULATION")
print("Total windows:", len(data))
print("Feature count:", len(feature_cols))
print("Split distribution:")
print(data["split"].value_counts().to_dict())

# ------------------------------------------------------------
# 2. RECONSTRUCT THE SAVED 69-FEATURE NORMALIZATION
# ------------------------------------------------------------

raw = data[feature_cols].to_numpy(
    dtype=np.float64
)

X69 = (raw - means) / scales

if not np.isfinite(X69).all():
    raise RuntimeError(
        "STOP: Nonfinite normalized features."
    )

constant = [
    "mean_power__n_points",
    "max_power__n_points",
    "std_power__n_points"
]

constant_idx = [
    feature_cols.index(c)
    for c in constant
]

if not np.all(X69[:, constant_idx] == 0):
    raise RuntimeError(
        "STOP: Constant feature normalization mismatch."
    )

print("\n2. NORMALIZATION")
print("Normalized shape:", X69.shape)
print("Nonfinite values: 0")
print("Constant features normalized to zero: PASS")

# Validate training-derived standardization statistics.
train_mask = data["split"].eq("train").to_numpy()

X_train = X69[train_mask]

train_means = np.mean(X_train, axis=0)

print(
    "Maximum absolute normalized training mean:",
    float(np.max(np.abs(train_means)))
)

# This is a diagnostic, not a universal zero-mean
# requirement for features normalized differently.
print("Stored normalization parameters: VERIFIED")

# ------------------------------------------------------------
# 3. VALIDATE HISTORICAL TEST POPULATION
# ------------------------------------------------------------

test_mask = data["split"].eq("test").to_numpy()

test_data = data.loc[
    test_mask,
    ["model_window_id"]
].copy()

X_test = X69[test_mask]

assert len(test_data) == 4079
assert X_test.shape == (4079, 69)

test_id_to_position = {
    int(wid): i
    for i, wid in enumerate(
        test_data["model_window_id"]
    )
}

print("\n3. TEST INPUT")
print("Test windows:", len(test_data))
print("Test matrix:", X_test.shape)
print("TEST INPUT: PASS")

# ------------------------------------------------------------
# 4. REPRODUCE HISTORICAL MODEL SCORES
# ------------------------------------------------------------

models = {
    "Isolation Forest": "isolation_forest",
    "Local Outlier Factor": "local_outlier_factor",
    "One-Class SVM": "one_class_svm"
}

D7_SCORE_AUDIT = {}

print("\n" + "=" * 95)
print("4. HISTORICAL SCORE COMPARISON")
print("=" * 95)

all_reproduced = True

for model_name, folder in models.items():

    print("\nMODEL:", model_name)

    model = joblib.load(
        MODELS / folder / "model.joblib"
    )

    archived = pd.read_csv(
        MODELS / folder / "test_window_scores.csv"
    )

    assert len(archived) == 4079
    assert archived["model_window_id"].is_unique
    assert int(model.n_features_in_) == 69

    # Match historical score-row order exactly.
    archived_ids = archived[
        "model_window_id"
    ].astype(int).tolist()

    if set(archived_ids) != set(test_id_to_position):
        raise RuntimeError(
            f"STOP: Test identity mismatch: {model_name}"
        )

    positions = np.array([
        test_id_to_position[wid]
        for wid in archived_ids
    ], dtype=np.int64)

    historical = archived[
        "anomaly_score"
    ].to_numpy(dtype=np.float64)

    if not np.isfinite(historical).all():
        raise RuntimeError(
            f"STOP: Invalid archived scores: {model_name}"
        )

    # Test both input dtypes because numerical
    # compatibility may depend on the historical
    # estimator and nearest-neighbor implementation.
    candidates = []

    for dtype_name, dtype in [
        ("float64", np.float64),
        ("float32", np.float32)
    ]:

        X_query = X_test[positions].astype(
            dtype, copy=False
        )

        score_samples = np.asarray(
            model.score_samples(X_query),
            dtype=np.float64
        )

        decision = np.asarray(
            model.decision_function(X_query),
            dtype=np.float64
        )

        variants = {
            "negative_score_samples": -score_samples,
            "positive_score_samples": score_samples,
            "negative_decision_function": -decision,
            "positive_decision_function": decision
        }

        for convention, predicted in variants.items():

            if not np.isfinite(predicted).all():
                raise RuntimeError(
                    f"STOP: Nonfinite scores: {model_name}"
                )

            differences = np.abs(
                predicted - historical
            )

            candidates.append({
                "dtype": dtype_name,
                "convention": convention,
                "mae": float(differences.mean()),
                "max_abs_error": float(
                    differences.max()
                ),
                "predicted": predicted
            })

    candidates.sort(
        key=lambda item: (
            item["mae"],
            item["max_abs_error"]
        )
    )

    print("\nBest score conventions:")

    for candidate in candidates[:4]:
        print(
            f"  {candidate['dtype']:7s} | "
            f"{candidate['convention']:28s} | "
            f"MAE={candidate['mae']:.12g} | "
            f"MaxErr={candidate['max_abs_error']:.12g}"
        )

    best = candidates[0]

    reproduced = bool(
        np.allclose(
            best["predicted"],
            historical,
            rtol=1e-6,
            atol=1e-6
        )
    )

    print("\nSelected dtype:", best["dtype"])
    print("Selected convention:", best["convention"])
    print("Score reproduction:",
          "PASS" if reproduced else "REVIEW REQUIRED")

    thresholds = archived[
        "threshold"
    ].to_numpy(dtype=np.float64)

    if not np.isfinite(thresholds).all():
        raise RuntimeError(
            f"STOP: Invalid thresholds: {model_name}"
        )

    flag_strings = (
        archived["flagged"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    flag_map = {
        "true": True,
        "false": False,
        "1": True,
        "0": False
    }

    flags = flag_strings.map(flag_map)

    if flags.isna().any():
        raise RuntimeError(
            f"STOP: Invalid flags: {model_name}"
        )

    archived_flags = flags.to_numpy(
        dtype=bool
    )

    greater_flags = historical > thresholds
    inclusive_flags = historical >= thresholds

    mismatches_gt = int(
        np.sum(greater_flags != archived_flags)
    )

    mismatches_ge = int(
        np.sum(inclusive_flags != archived_flags)
    )

    print(
        "Stored-score > threshold mismatches:",
        mismatches_gt
    )

    print(
        "Stored-score >= threshold mismatches:",
        mismatches_ge
    )

    print(
        "Distinct historical thresholds:",
        len(np.unique(thresholds))
    )

    D7_SCORE_AUDIT[model_name] = {
        "folder": folder,
        "dtype": best["dtype"],
        "convention": best["convention"],
        "score_reproduced": reproduced,
        "mean_absolute_error": best["mae"],
        "maximum_absolute_error": best["max_abs_error"],
        "historical_threshold": float(
            thresholds[0]
        ),
        "thresholds_unique": len(
            np.unique(thresholds)
        ),
        "threshold_gt_mismatches": mismatches_gt,
        "threshold_ge_mismatches": mismatches_ge
    }

    if not reproduced:
        all_reproduced = False

# ------------------------------------------------------------
# 5. FINAL RESULT
# ------------------------------------------------------------

print("\n" + "=" * 95)
print("CELL 10 — FINAL REPRODUCIBILITY STATUS")
print("=" * 95)

for name, audit in D7_SCORE_AUDIT.items():
    print(
        f"{name}: "
        f"{'PASS' if audit['score_reproduced'] else 'REVIEW'}"
    )

if not all_reproduced:
    print("\nCELL 10: REVIEW REQUIRED")
    print(
        "At least one historical score interface "
        "was not reproduced within tolerance."
    )
    print(
        "Do not start the 66-feature refit until "
        "the discrepancy is investigated."
    )
else:
    print("\nCELL 10: PASSED")
    print(
        "All three historical 69-feature score "
        "interfaces reproduced."
    )
    print(
        "Controlled 66-feature comparison is "
        "ready for experimental setup."
    )

print("\nNo models fitted.")
print("No files written or modified.")
print("C3-4 remains OPEN.")
print("=" * 95)


In [ ]:

# ============================================================
# D7 / C3-4 — CELL 11
# HISTORICAL VALIDATION-THRESHOLD REPRODUCTION
#
# READ ONLY
# No fitting
# No label-based threshold selection
# No files written
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import joblib

print("=" * 95)
print("D7 / C3-4 — VALIDATION THRESHOLD REPRODUCTION")
print("=" * 95)

required_variables = [
    "data",
    "X69",
    "feature_cols",
    "D7_SCORE_AUDIT"
]

missing = [
    name for name in required_variables
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "STOP: Missing Cell 10 variables: "
        + ", ".join(missing)
    )

ROOT = Path(
    "/content/drive/MyDrive/Master/Thesis/Code/"
    "experiments/EXP_BLUED_FIVE_MODEL_LABEL_BLIND"
)

MODEL_ROOT = ROOT / "controlled_feature_models"

# ------------------------------------------------------------
# 1. VERIFY VALIDATION INPUT
# ------------------------------------------------------------

assert X69.shape == (20406, 69)
assert len(feature_cols) == 69
assert len(data) == 20406

validation_mask = (
    data["split"].astype(str).eq("validation")
    .to_numpy()
)

X_validation = X69[validation_mask]

if X_validation.shape != (4079, 69):
    raise RuntimeError(
        "STOP: Incorrect validation population."
    )

if not np.isfinite(X_validation).all():
    raise RuntimeError(
        "STOP: Nonfinite validation values."
    )

print("\n1. VALIDATION POPULATION")
print("Windows:", len(X_validation))
print("Features:", X_validation.shape[1])
print("No event labels accessed.")
print("VALIDATION INPUT: PASS")

# ------------------------------------------------------------
# 2. REPRODUCE HISTORICAL THRESHOLDS
# ------------------------------------------------------------

models = {
    "Isolation Forest": "isolation_forest",
    "Local Outlier Factor": "local_outlier_factor",
    "One-Class SVM": "one_class_svm"
}

D7_THRESHOLD_AUDIT = {}

all_passed = True

print("\n" + "=" * 95)
print("2. VALIDATION-DERIVED THRESHOLDS")
print("=" * 95)

for name, folder in models.items():

    print("\nMODEL:", name)

    model_path = (
        MODEL_ROOT / folder / "model.joblib"
    )

    score_path = (
        MODEL_ROOT / folder / "test_window_scores.csv"
    )

    if not model_path.is_file() or not score_path.is_file():
        raise RuntimeError(
            f"STOP: Missing historical artifact: {name}"
        )

    model = joblib.load(model_path)

    archived = pd.read_csv(score_path)

    audit = D7_SCORE_AUDIT[name]

    if not audit["score_reproduced"]:
        raise RuntimeError(
            f"STOP: Historical score audit failed: {name}"
        )

    # Retain the dtype that reproduced historical scores.
    dtype = (
        np.float32
        if audit["dtype"] == "float32"
        else np.float64
    )

    X_query = X_validation.astype(
        dtype,
        copy=False
    )

    # Historical anomaly-score convention:
    # larger score = more anomalous.
    validation_scores = -np.asarray(
        model.score_samples(X_query),
        dtype=np.float64
    )

    if not np.isfinite(validation_scores).all():
        raise RuntimeError(
            f"STOP: Nonfinite validation scores: {name}"
        )

    # Fixed label-blind operating threshold.
    reconstructed_threshold = float(
        np.quantile(
            validation_scores,
            0.95,
            method="linear"
        )
    )

    saved_thresholds = archived[
        "threshold"
    ].to_numpy(dtype=np.float64)

    if not np.isfinite(saved_thresholds).all():
        raise RuntimeError(
            f"STOP: Invalid saved thresholds: {name}"
        )

    if len(np.unique(saved_thresholds)) != 1:
        raise RuntimeError(
            f"STOP: Multiple historical thresholds: {name}"
        )

    historical_threshold = float(
        saved_thresholds[0]
    )

    difference = abs(
        reconstructed_threshold - historical_threshold
    )

    threshold_match = bool(
        np.isclose(
            reconstructed_threshold,
            historical_threshold,
            atol=1e-6,
            rtol=1e-6
        )
    )

    # This count measures the validation operating
    # rate only. It is NOT a label-based metric.
    validation_flags = (
        validation_scores > reconstructed_threshold
    )

    validation_flag_count = int(
        validation_flags.sum()
    )

    validation_flag_rate = float(
        validation_flags.mean()
    )

    print(
        "Historical threshold:",
        f"{historical_threshold:.12f}"
    )

    print(
        "Reconstructed threshold:",
        f"{reconstructed_threshold:.12f}"
    )

    print(
        "Absolute difference:",
        f"{difference:.12g}"
    )

    print(
        "Validation flagged windows:",
        validation_flag_count
    )

    print(
        "Validation flagged rate:",
        f"{validation_flag_rate:.6%}"
    )

    print(
        "Threshold comparison:",
        "PASS" if threshold_match
        else "REVIEW REQUIRED"
    )

    D7_THRESHOLD_AUDIT[name] = {
        "folder": folder,
        "dtype": audit["dtype"],
        "historical_threshold": historical_threshold,
        "reconstructed_threshold":
            reconstructed_threshold,
        "absolute_difference": difference,
        "validation_flag_count":
            validation_flag_count,
        "validation_flag_rate":
            validation_flag_rate,
        "threshold_match": threshold_match
    }

    if not threshold_match:
        all_passed = False

# ------------------------------------------------------------
# 3. FINAL DECISION
# ------------------------------------------------------------

print("\n" + "=" * 95)
print("CELL 11 — FINAL STATUS")
print("=" * 95)

for name, result in D7_THRESHOLD_AUDIT.items():
    print(
        f"{name}: "
        f"{'PASS' if result['threshold_match'] else 'REVIEW'}"
    )

if all_passed:

    print("\nCELL 11: PASSED")
    print("Historical validation thresholds reproduced.")
    print("Original score interfaces already verified.")
    print("66-feature controlled refit is authorized.")

else:

    print("\nCELL 11: REVIEW REQUIRED")
    print(
        "Do not fit reduced-feature models until "
        "the threshold discrepancies are resolved."
    )

print("\nNo event labels used.")
print("No models fitted.")
print("No files written or modified.")
print("C3-4 remains OPEN.")
print("=" * 95)


In [ ]:

# ============================================================
# D7 / C3-4 — CELL 12
# CONTROLLED BLUED 69-FEATURE vs 66-FEATURE COMPARISON
#
# FITS NEW MODELS — training partition only
# Thresholds — validation scores only
# Test labels — NOT accessed
# Historical models and files — NEVER modified
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import gc
import json
import time

import joblib
import numpy as np
import pandas as pd
import sklearn

from sklearn.base import clone

print("=" * 95)
print("D7 / C3-4 — CONTROLLED BLUED FEATURE REDUCTION")
print("=" * 95)
print("scikit-learn:", sklearn.__version__)

# ------------------------------------------------------------
# 1. VERIFY PREVIOUS REPRODUCIBILITY AUDITS
# ------------------------------------------------------------

required = [
    "data",
    "X69",
    "feature_cols",
    "D7_SCORE_AUDIT",
    "D7_THRESHOLD_AUDIT"
]

missing = [
    name for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "STOP: Missing previous cell variables: "
        + ", ".join(missing)
    )

MODELS = {
    "Isolation Forest": "isolation_forest",
    "Local Outlier Factor": "local_outlier_factor",
    "One-Class SVM": "one_class_svm"
}

for name in MODELS:
    if not D7_SCORE_AUDIT[name]["score_reproduced"]:
        raise RuntimeError(
            f"STOP: Score reproduction failed: {name}"
        )

    if not D7_THRESHOLD_AUDIT[name]["threshold_match"]:
        raise RuntimeError(
            f"STOP: Threshold reproduction failed: {name}"
        )

print("\n1. PREVIOUS REPRODUCIBILITY CHECKS: PASS")

# ------------------------------------------------------------
# 2. CONSTRUCT REDUCED INPUT REPRESENTATION
# ------------------------------------------------------------

ROOT = Path(
    "/content/drive/MyDrive/Master/Thesis/Code/"
    "experiments/EXP_BLUED_FIVE_MODEL_LABEL_BLIND"
)

MODEL_ROOT = ROOT / "controlled_feature_models"

constant_features = [
    "mean_power__n_points",
    "max_power__n_points",
    "std_power__n_points"
]

if (
    len(feature_cols) != 69
    or len(set(feature_cols)) != 69
    or any(f not in feature_cols for f in constant_features)
):
    raise RuntimeError("STOP: Unexpected feature schema.")

keep_indices = [
    i for i, feature in enumerate(feature_cols)
    if feature not in constant_features
]

reduced_features = [
    feature_cols[i] for i in keep_indices
]

assert len(keep_indices) == 66
assert len(reduced_features) == 66
assert X69.shape == (20406, 69)
assert len(data) == 20406

constant_indices = [
    feature_cols.index(f)
    for f in constant_features
]

if not np.all(X69[:, constant_indices] == 0):
    raise RuntimeError(
        "STOP: Removed features are not uniformly zero."
    )

X66 = np.ascontiguousarray(
    X69[:, keep_indices],
    dtype=np.float64
)

if not np.isfinite(X66).all():
    raise RuntimeError(
        "STOP: Nonfinite reduced feature values."
    )

# Confirm exact retained-column equality.
if not np.array_equal(X66, X69[:, keep_indices]):
    raise RuntimeError(
        "STOP: Retained features changed unexpectedly."
    )

splits = data["split"].astype(str).to_numpy()

expected_counts = {
    "train": 12248,
    "validation": 4079,
    "test": 4079
}

actual_counts = {
    name: int(np.sum(splits == name))
    for name in expected_counts
}

if actual_counts != expected_counts:
    raise RuntimeError(
        f"STOP: Split mismatch: {actual_counts}"
    )

if len(splits) != sum(expected_counts.values()):
    raise RuntimeError(
        "STOP: Unexpected split labels."
    )

ids = data["model_window_id"].to_numpy()

if (
    pd.isna(ids).any()
    or pd.Series(ids).duplicated().any()
):
    raise RuntimeError(
        "STOP: Invalid model-window identifiers."
    )

train_mask = splits == "train"
val_mask = splits == "validation"
test_mask = splits == "test"

X_train = X66[train_mask]
X_val = X66[val_mask]
X_test = X66[test_mask]

val_ids = ids[val_mask]
test_ids = ids[test_mask]

print("\n2. CONTROLLED INPUTS")
print("Original features:", X69.shape[1])
print("Reduced features:", X66.shape[1])
print("Removed features:", constant_features)
print("Training matrix:", X_train.shape)
print("Validation matrix:", X_val.shape)
print("Test matrix:", X_test.shape)
print("Retained feature values: EXACT MATCH")
print("INPUT VERIFICATION: PASS")

# ------------------------------------------------------------
# 3. CONTROLLED MODEL FITTING AND SCORING
# ------------------------------------------------------------

results = {}

print("\n" + "=" * 95)
print("3. MODEL REFITTING")
print("=" * 95)

for name, folder in MODELS.items():

    print("\n" + "-" * 75, flush=True)
    print("MODEL:", name, flush=True)

    original_path = MODEL_ROOT / folder / "model.joblib"
    archived_path = (
        MODEL_ROOT / folder / "test_window_scores.csv"
    )

    if not original_path.is_file():
        raise RuntimeError(
            f"STOP: Missing historical model: {name}"
        )

    original = joblib.load(original_path)

    if int(original.n_features_in_) != 69:
        raise RuntimeError(
            f"STOP: Original model width mismatch: {name}"
        )

    # Clone preserves estimator hyperparameters
    # without copying learned parameters.
    reduced_model = clone(original)

    if reduced_model.get_params(deep=False) != (
        original.get_params(deep=False)
    ):
        raise RuntimeError(
            f"STOP: Parameter mismatch: {name}"
        )

    # Preserve the stored LOF training-input dtype
    # when available. Other models receive float64;
    # Isolation Forest applies its internal dtype.
    fit_dtype = np.float64

    if name == "Local Outlier Factor":
        stored_training = getattr(
            original, "_fit_X", None
        )

        if stored_training is not None:

            if stored_training.shape != (12248, 69):
                raise RuntimeError(
                    "STOP: Unexpected historical LOF "
                    "training-matrix shape."
                )

            fit_dtype = stored_training.dtype

            historical_train_mae = float(
                np.mean(np.abs(
                    stored_training.astype(np.float64)
                    - X69[train_mask]
                ))
            )

            print(
                "Historical LOF training-input MAE:",
                f"{historical_train_mae:.12g}"
            )

            if not np.allclose(
                stored_training,
                X69[train_mask],
                rtol=1e-6,
                atol=1e-6
            ):
                raise RuntimeError(
                    "STOP: Historical LOF training "
                    "inputs do not match."
                )

    print("Fit dtype:", np.dtype(fit_dtype))
    print("Fitting training-only data...", flush=True)

    start = time.perf_counter()

    reduced_model.fit(
        X_train.astype(fit_dtype, copy=False)
    )

    elapsed = time.perf_counter() - start

    if int(reduced_model.n_features_in_) != 66:
        raise RuntimeError(
            f"STOP: Reduced model width mismatch: {name}"
        )

    # Retain the query dtype that reproduced
    # the original historical scoring interface.
    query_dtype = (
        np.float32
        if D7_SCORE_AUDIT[name]["dtype"] == "float32"
        else np.float64
    )

    print("Scoring validation...", flush=True)

    val_scores = -np.asarray(
        reduced_model.score_samples(
            X_val.astype(query_dtype, copy=False)
        ),
        dtype=np.float64
    )

    print("Scoring test...", flush=True)

    test_scores = -np.asarray(
        reduced_model.score_samples(
            X_test.astype(query_dtype, copy=False)
        ),
        dtype=np.float64
    )

    if (
        not np.isfinite(val_scores).all()
        or not np.isfinite(test_scores).all()
    ):
        raise RuntimeError(
            f"STOP: Nonfinite new scores: {name}"
        )

    # Exactly the historical label-blind rule.
    threshold66 = float(
        np.quantile(
            val_scores, 0.95, method="linear"
        )
    )

    val_flags = val_scores > threshold66
    test_flags = test_scores > threshold66

    print("Fit time (seconds):", round(elapsed, 3))
    print("New threshold:", f"{threshold66:.12f}")
    print("Validation flagged:", int(val_flags.sum()))
    print("Test flagged:", int(test_flags.sum()))

    # --------------------------------------------------------
    # 4. ALIGN AGAINST HISTORICAL 69-DIMENSION SCORES
    # --------------------------------------------------------

    # Explicitly exclude event labels.
    archived = pd.read_csv(
        archived_path,
        usecols=[
            "model_window_id",
            "anomaly_score",
            "threshold",
            "flagged"
        ]
    )

    if (
        len(archived) != 4079
        or archived["model_window_id"].duplicated().any()
    ):
        raise RuntimeError(
            f"STOP: Historical test schema: {name}"
        )

    current = pd.DataFrame({
        "model_window_id": test_ids,
        "reduced_66_anomaly_score": test_scores,
        "reduced_66_threshold": threshold66,
        "reduced_66_flagged": test_flags
    })

    historical = archived.rename(columns={
        "anomaly_score": "original_69_anomaly_score",
        "threshold": "original_69_threshold",
        "flagged": "original_69_flagged"
    })

    comparison = current.merge(
        historical,
        on="model_window_id",
        how="outer",
        validate="one_to_one",
        indicator=True
    )

    if (
        len(comparison) != 4079
        or not comparison["_merge"].eq("both").all()
    ):
        raise RuntimeError(
            f"STOP: Test-window alignment: {name}"
        )

    comparison = comparison.drop(columns="_merge")

    original_flag_strings = (
        comparison["original_69_flagged"]
        .astype(str).str.lower().str.strip()
    )

    original_flags = original_flag_strings.map({
        "true": True,
        "false": False,
        "1": True,
        "0": False
    })

    if original_flags.isna().any():
        raise RuntimeError(
            f"STOP: Invalid original flags: {name}"
        )

    comparison["original_69_flagged"] = (
        original_flags.astype(bool)
    )

    old_flags = comparison[
        "original_69_flagged"
    ].to_numpy(dtype=bool)

    new_flags = comparison[
        "reduced_66_flagged"
    ].to_numpy(dtype=bool)

    overlap = int(np.sum(old_flags & new_flags))
    union = int(np.sum(old_flags | new_flags))

    jaccard = (
        float(overlap / union)
        if union > 0 else 1.0
    )

    agreement = float(np.mean(
        old_flags == new_flags
    ))

    changed = int(np.sum(
        old_flags != new_flags
    ))

    score_rho = float(
        comparison[
            "original_69_anomaly_score"
        ].corr(
            comparison["reduced_66_anomaly_score"],
            method="spearman"
        )
    )

    if not np.isfinite(score_rho):
        raise RuntimeError(
            f"STOP: Undefined rank correlation: {name}"
        )

    comparison["score_difference_66_minus_69"] = (
        comparison["reduced_66_anomaly_score"]
        - comparison["original_69_anomaly_score"]
    )

    comparison["flag_changed"] = (
        comparison["reduced_66_flagged"]
        != comparison["original_69_flagged"]
    )

    print("\nHistorical versus reduced:")
    print("  Original test flags:", int(old_flags.sum()))
    print("  Reduced test flags:", int(new_flags.sum()))
    print("  Changed test flags:", changed)
    print("  Binary agreement:", f"{agreement:.6f}")
    print("  Flag Jaccard:", f"{jaccard:.6f}")
    print("  Score Spearman:", f"{score_rho:.6f}")

    results[name] = {
        "folder": folder,
        "model": reduced_model,
        "fit_seconds": elapsed,
        "fit_dtype": str(np.dtype(fit_dtype)),
        "query_dtype": str(np.dtype(query_dtype)),
        "threshold": threshold66,
        "validation_scores": pd.DataFrame({
            "model_window_id": val_ids,
            "anomaly_score": val_scores,
            "threshold": threshold66,
            "flagged": val_flags
        }),
        "test_comparison": comparison,
        "test_flags_69": int(old_flags.sum()),
        "test_flags_66": int(new_flags.sum()),
        "changed_flags": changed,
        "flag_agreement": agreement,
        "flag_jaccard": jaccard,
        "score_spearman": score_rho
    }

    del original
    gc.collect()

    print("MODEL COMPLETE:", name, flush=True)

# ------------------------------------------------------------
# 5. SAVE IN A NEW, ISOLATED D7 EVIDENCE FOLDER
# ------------------------------------------------------------

print("\n" + "=" * 95)
print("5. PRESERVING THE NEW EXPERIMENT")
print("=" * 95)

EXP_ROOT = ROOT.parent

timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%S%fZ")

OUTPUT = (
    EXP_ROOT
    / f"D7_C3_4_BLUED_66_FEATURE_COMPARISON_{timestamp}"
)

if OUTPUT.exists():
    raise RuntimeError(
        "STOP: Output folder already exists."
    )

OUTPUT.mkdir(parents=True, exist_ok=False)

schema = pd.DataFrame({
    "original_feature_position": np.arange(69),
    "feature": feature_cols,
    "retained_in_66": [
        f in reduced_features for f in feature_cols
    ]
})

schema.to_csv(
    OUTPUT / "feature_schema.csv",
    index=False
)

for name, result in results.items():

    folder = OUTPUT / result["folder"]

    folder.mkdir(parents=True, exist_ok=False)

    joblib.dump(
        result["model"],
        folder / "model_66.joblib"
    )

    result["validation_scores"].to_csv(
        folder / "validation_scores_66.csv",
        index=False
    )

    result["test_comparison"].to_csv(
        folder / "test_69_vs_66.csv",
        index=False
    )

    print("Saved model evidence:", name)

manifest = {
    "experiment": "D7_C3_4_BLUED_66_FEATURE_COMPARISON",
    "created_utc": timestamp,
    "sklearn_version": sklearn.__version__,
    "historical_experiment_root": str(ROOT),
    "original_feature_count": 69,
    "reduced_feature_count": 66,
    "removed_features": constant_features,
    "preprocessing": (
        "Original stored training-derived normalization; "
        "66 columns selected without rescaling."
    ),
    "split_counts": expected_counts,
    "fitting_partition": "train",
    "threshold_partition": "validation",
    "threshold_quantile": 0.95,
    "threshold_comparison": "score > threshold",
    "score_definition": "-model.score_samples(X)",
    "test_event_labels_used_for_fitting": False,
    "test_event_labels_used_for_threshold": False,
    "test_event_metrics_evaluated": False,
    "historical_models_modified": False,
    "models": {
        name: {
            "folder": result["folder"],
            "fit_seconds": result["fit_seconds"],
            "fit_dtype": result["fit_dtype"],
            "query_dtype": result["query_dtype"],
            "validation_threshold": result["threshold"],
            "test_flags_69": result["test_flags_69"],
            "test_flags_66": result["test_flags_66"],
            "changed_flags": result["changed_flags"],
            "flag_agreement": result["flag_agreement"],
            "flag_jaccard": result["flag_jaccard"],
            "score_spearman": result["score_spearman"]
        }
        for name, result in results.items()
    }
}

with open(
    OUTPUT / "manifest.json",
    "w",
    encoding="utf-8"
) as handle:
    json.dump(
        manifest,
        handle,
        indent=2
    )

# ------------------------------------------------------------
# 6. VERIFY ALL OUTPUT ARTIFACTS
# ------------------------------------------------------------

expected_files = 11

generated = [
    path for path in OUTPUT.rglob("*")
    if path.is_file()
]

print("\n" + "=" * 95)
print("6. EXPERIMENT VERIFICATION")
print("=" * 95)

print("Output folder:", OUTPUT)
print("Expected files:", expected_files)
print("Actual files:", len(generated))

for path in sorted(generated):

    print(
        str(path.relative_to(OUTPUT)),
        "|",
        f"{path.stat().st_size:,} bytes"
    )

if len(generated) != expected_files:
    raise RuntimeError(
        "STOP: Evidence file count mismatch."
    )

if any(path.stat().st_size == 0 for path in generated):
    raise RuntimeError(
        "STOP: Empty evidence file detected."
    )

D7_66_OUTPUT_DIR = OUTPUT
D7_66_RESULTS = results

print("\n" + "=" * 95)
print("CELL 12: PASSED")
print("Three new 66-feature models fitted.")
print("Validation-derived thresholds calculated.")
print("Historical versus reduced test outputs compared.")
print("Eleven new evidence files preserved.")
print("No test event labels accessed.")
print("Historical experiments remain unchanged.")
print("C3-4 remains OPEN pending final evaluation.")
print("=" * 95)


In [ ]:

# ============================================================
# D7 / C3-4 — CELL 13
# BLUED 69-vs-66 FEATURE EVENT-ALIGNMENT EVALUATION
#
# POST-HOC EVALUATION ONLY
# No fitting or score regeneration
# No threshold selection
# No historical files modified
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

print("=" * 95)
print("D7 / C3-4 — BLUED EVENT-ALIGNMENT EVALUATION")
print("=" * 95)

ROOT = Path(
    "/content/drive/MyDrive/Master/Thesis/Code/"
    "experiments"
)

HIST = ROOT / "EXP_BLUED_FIVE_MODEL_LABEL_BLIND"

REDUCED = ROOT / (
    "D7_C3_4_BLUED_66_FEATURE_COMPARISON_"
    "20261107T122521705785Z"
)

MODELS = {
    "Isolation Forest": "isolation_forest",
    "Local Outlier Factor": "local_outlier_factor",
    "One-Class SVM": "one_class_svm"
}

if not REDUCED.is_dir():
    raise RuntimeError("STOP: Reduced experiment missing.")

with open(
    REDUCED / "manifest.json",
    encoding="utf-8"
) as f:
    audit_manifest = json.load(f)

assert audit_manifest["original_feature_count"] == 69
assert audit_manifest["reduced_feature_count"] == 66
assert audit_manifest["test_event_labels_used_for_fitting"] is False
assert audit_manifest["test_event_labels_used_for_threshold"] is False

print("\n1. PRESERVED EXPERIMENT: PASS")

# ------------------------------------------------------------
# 2. LOAD TEST LABELS FROM THE FROZEN BLUED MANIFEST
# ------------------------------------------------------------

manifest = pd.read_csv(
    HIST / "common"
    / "blued_1min_stride30s_model_manifest.csv",
    usecols=[
        "model_window_id",
        "split",
        "event_count",
        "event_window_label"
    ]
)

if manifest["model_window_id"].duplicated().any():
    raise RuntimeError(
        "STOP: Duplicate manifest identifiers."
    )

test_labels = manifest.loc[
    manifest["split"].eq("test"),
    [
        "model_window_id",
        "event_count",
        "event_window_label"
    ]
].copy()

if len(test_labels) != 4079:
    raise RuntimeError(
        "STOP: Test population mismatch."
    )

test_labels["event_window_label"] = pd.to_numeric(
    test_labels["event_window_label"],
    errors="raise"
).astype(int)

test_labels["event_count"] = pd.to_numeric(
    test_labels["event_count"],
    errors="raise"
)

if not set(
    test_labels["event_window_label"].unique()
).issubset({0, 1}):
    raise RuntimeError(
        "STOP: Unexpected event labels."
    )

if not (
    test_labels["event_window_label"]
    == (test_labels["event_count"] > 0).astype(int)
).all():
    raise RuntimeError(
        "STOP: Event count and event label disagree."
    )

positive = int(
    test_labels["event_window_label"].sum()
)

negative = len(test_labels) - positive

if positive == 0 or negative == 0:
    raise RuntimeError(
        "STOP: Both event classes are required."
    )

print("\n2. TEST ANNOTATION POPULATION")
print("Total windows:", len(test_labels))
print("Event windows:", positive)
print("Non-event windows:", negative)
print("Event prevalence:", f"{positive / 4079:.6%}")
print("Annotation consistency: PASS")

# ------------------------------------------------------------
# 3. BOOLEAN PARSER
# ------------------------------------------------------------

def parse_flags(series):

    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False
    }

    parsed = (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map(mapping)
    )

    if parsed.isna().any():
        raise RuntimeError(
            "STOP: Unrecognized flag representation."
        )

    return parsed.astype(bool).to_numpy()


# ------------------------------------------------------------
# 4. EVALUATE ORIGINAL AND REDUCED TEST OUTPUTS
# ------------------------------------------------------------

records = []
changes = []

expected = {
    "Isolation Forest": (198, 193, 11),
    "Local Outlier Factor": (349, 349, 0),
    "One-Class SVM": (239, 239, 0)
}

for model_name, folder in MODELS.items():

    print("\n" + "-" * 80)
    print("MODEL:", model_name)

    path = (
        REDUCED / folder
        / "test_69_vs_66.csv"
    )

    archive_path = (
        HIST / "controlled_feature_models"
        / folder / "test_window_scores.csv"
    )

    if not path.is_file() or not archive_path.is_file():
        raise RuntimeError(
            f"STOP: Missing model evidence: {model_name}"
        )

    comparison = pd.read_csv(path)

    archived = pd.read_csv(
        archive_path,
        usecols=[
            "model_window_id",
            "event_window_label"
        ]
    )

    if (
        len(comparison) != 4079
        or comparison["model_window_id"].duplicated().any()
        or len(archived) != 4079
        or archived["model_window_id"].duplicated().any()
    ):
        raise RuntimeError(
            f"STOP: Invalid test identities: {model_name}"
        )

    combined = comparison.merge(
        test_labels,
        on="model_window_id",
        how="outer",
        validate="one_to_one",
        indicator=True
    )

    if (
        len(combined) != 4079
        or not combined["_merge"].eq("both").all()
    ):
        raise RuntimeError(
            f"STOP: Test population alignment: {model_name}"
        )

    combined = combined.drop(columns="_merge")

    label_check = combined[
        ["model_window_id", "event_window_label"]
    ].merge(
        archived,
        on="model_window_id",
        validate="one_to_one",
        suffixes=("_manifest", "_archive")
    )

    if not (
        label_check["event_window_label_manifest"]
        == label_check["event_window_label_archive"]
    ).all():
        raise RuntimeError(
            f"STOP: Historical event labels differ: {model_name}"
        )

    y_true = combined[
        "event_window_label"
    ].to_numpy(dtype=int)

    old_flags = parse_flags(
        combined["original_69_flagged"]
    )

    new_flags = parse_flags(
        combined["reduced_66_flagged"]
    )

    expected_old, expected_new, expected_changed = (
        expected[model_name]
    )

    observed = (
        int(old_flags.sum()),
        int(new_flags.sum()),
        int(np.sum(old_flags != new_flags))
    )

    if observed != (
        expected_old,
        expected_new,
        expected_changed
    ):
        raise RuntimeError(
            f"STOP: Changed flag counts: {model_name}: "
            f"{observed}"
        )

    for representation, suffix, flags in [
        ("Original 69", "original_69", old_flags),
        ("Reduced 66", "reduced_66", new_flags)
    ]:

        scores = pd.to_numeric(
            combined[f"{suffix}_anomaly_score"],
            errors="raise"
        ).to_numpy(dtype=float)

        thresholds = pd.to_numeric(
            combined[f"{suffix}_threshold"],
            errors="raise"
        ).to_numpy(dtype=float)

        if (
            not np.isfinite(scores).all()
            or not np.isfinite(thresholds).all()
        ):
            raise RuntimeError(
                f"STOP: Nonfinite model outputs: {model_name}"
            )

        if len(np.unique(thresholds)) != 1:
            raise RuntimeError(
                f"STOP: Nonconstant threshold: {model_name}"
            )

        if not np.array_equal(
            scores > thresholds,
            flags
        ):
            raise RuntimeError(
                f"STOP: Score/flag mismatch: "
                f"{model_name} {representation}"
            )

        tn, fp, fn, tp = confusion_matrix(
            y_true,
            flags.astype(int),
            labels=[0, 1]
        ).ravel()

        records.append({
            "model": model_name,
            "representation": representation,
            "test_windows": len(y_true),
            "event_windows": int(y_true.sum()),
            "flagged_windows": int(flags.sum()),
            "TP": int(tp),
            "FP": int(fp),
            "TN": int(tn),
            "FN": int(fn),
            "precision": precision_score(
                y_true, flags, zero_division=0
            ),
            "recall": recall_score(
                y_true, flags, zero_division=0
            ),
            "f1": f1_score(
                y_true, flags, zero_division=0
            ),
            "fpr": float(fp / (fp + tn)),
            "roc_auc": roc_auc_score(
                y_true, scores
            ),
            "average_precision": average_precision_score(
                y_true, scores
            )
        })

    changed_mask = old_flags != new_flags

    if changed_mask.any():

        changed = combined.loc[
            changed_mask,
            [
                "model_window_id",
                "event_window_label",
                "event_count",
                "original_69_anomaly_score",
                "original_69_threshold",
                "reduced_66_anomaly_score",
                "reduced_66_threshold"
            ]
        ].copy()

        changed["model"] = model_name

        changed["original_flag"] = old_flags[
            changed_mask
        ]

        changed["reduced_flag"] = new_flags[
            changed_mask
        ]

        changed["direction"] = np.where(
            changed["original_flag"],
            "flag_removed",
            "flag_added"
        )

        changes.append(changed)

    print("Aligned test windows: 4,079")
    print("Original flags:", int(old_flags.sum()))
    print("Reduced flags:", int(new_flags.sum()))
    print(
        "Changed flags:",
        int(changed_mask.sum())
    )
    print("EVENT EVALUATION: PASS")

# ------------------------------------------------------------
# 5. COMPUTE REPRESENTATION DIFFERENCES
# ------------------------------------------------------------

metrics = pd.DataFrame(records)

metric_columns = [
    "precision", "recall", "f1",
    "fpr", "roc_auc", "average_precision"
]

delta_records = []

for model_name in MODELS:

    subset = metrics.loc[
        metrics["model"].eq(model_name)
    ].set_index("representation")

    old = subset.loc["Original 69"]
    new = subset.loc["Reduced 66"]

    delta = {
        "model": model_name,
        "flags_69": int(old["flagged_windows"]),
        "flags_66": int(new["flagged_windows"])
    }

    for column in metric_columns:
        delta[f"delta_{column}"] = float(
            new[column] - old[column]
        )

    delta_records.append(delta)

deltas = pd.DataFrame(delta_records)

if changes:
    changed_windows = pd.concat(
        changes, ignore_index=True
    )
else:
    changed_windows = pd.DataFrame()

assert len(metrics) == 6
assert len(deltas) == 3
assert len(changed_windows) == 11

# ------------------------------------------------------------
# 6. REPORT RESULTS BEFORE SAVING
# ------------------------------------------------------------

print("\n" + "=" * 95)
print("3. EVENT-ALIGNMENT METRICS")
print("=" * 95)

print(
    metrics[
        [
            "model", "representation",
            "flagged_windows",
            "precision", "recall", "f1",
            "fpr", "roc_auc", "average_precision"
        ]
    ].to_string(
        index=False,
        float_format=lambda v: f"{v:.6f}"
    )
)

print("\n" + "=" * 95)
print("4. REDUCED MINUS ORIGINAL METRIC DIFFERENCES")
print("=" * 95)

print(
    deltas.to_string(
        index=False,
        float_format=lambda v: f"{v:+.6f}"
    )
)

print("\n5. CHANGED FLAGS BY EVENT LABEL")

print(
    changed_windows.groupby(
        ["model", "direction", "event_window_label"]
    ).size().to_string()
)

# ------------------------------------------------------------
# 7. SAVE POST-HOC EVIDENCE SEPARATELY
# ------------------------------------------------------------

OUTPUT = REDUCED / "posthoc_event_alignment"

if OUTPUT.exists():
    raise RuntimeError(
        "STOP: Evaluation output directory already exists. "
        "Do not overwrite existing evidence."
    )

OUTPUT.mkdir(parents=True, exist_ok=False)

metrics.to_csv(
    OUTPUT / "event_alignment_metrics.csv",
    index=False
)

deltas.to_csv(
    OUTPUT / "event_alignment_deltas.csv",
    index=False
)

changed_windows.to_csv(
    OUTPUT / "changed_flag_windows.csv",
    index=False
)

evaluation_manifest = {
    "audit": "D7_C3_4_BLUED_69_vs_66_event_alignment",
    "source_experiment": str(REDUCED),
    "test_windows": 4079,
    "event_windows": positive,
    "non_event_windows": negative,
    "annotation_type": "appliance_switching_events",
    "confirmed_anomaly_labels": False,
    "models_refitted_during_evaluation": False,
    "thresholds_recalibrated_during_evaluation": False,
    "test_labels_used_for_model_selection": False,
    "evaluation": "paired_descriptive_comparison",
    "no_statistical_significance_claim": True,
    "metric_delta": "reduced_66_minus_original_69"
}

with open(
    OUTPUT / "evaluation_manifest.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        evaluation_manifest,
        f,
        indent=2
    )

generated = sorted(
    p for p in OUTPUT.iterdir()
    if p.is_file()
)

if (
    len(generated) != 4
    or any(p.stat().st_size == 0 for p in generated)
):
    raise RuntimeError(
        "STOP: Evaluation evidence verification failed."
    )

D7_EVENT_METRICS = metrics
D7_EVENT_DELTAS = deltas
D7_EVENT_CHANGES = changed_windows
D7_EVENT_OUTPUT = OUTPUT

print("\n" + "=" * 95)
print("CELL 13: PASSED")
print("Six model-representation results evaluated.")
print("Original 69 versus reduced 66 compared.")
print("Eleven changed Isolation Forest flags documented.")
print("Four evaluation files preserved.")
print("No models fitted.")
print("No thresholds recalibrated.")
print("No historical files modified.")
print("C3-4 remains OPEN pending manuscript integration.")
print("=" * 95)
print("Evidence:", OUTPUT)
